In [1]:
!pip install torch joblib matplotlib seaborn tqdm pgmpy fairlearn causallearn

ERROR: Could not find a version that satisfies the requirement causallearn (from versions: none)
ERROR: No matching distribution found for causallearn


In [2]:
# import os, gc, copy, math, warnings, logging
# from dataclasses import dataclass, field
# from typing import Dict, List, Optional, Tuple

# import numpy as np
# import pandas as pd
# import torch
# import torch.nn as nn
# import torch.optim as optim
# import torch.nn.functional as F
# from torch.utils.data import TensorDataset, DataLoader

# from sklearn.metrics import accuracy_score, mutual_info_score
# from sklearn.neural_network import MLPClassifier
# from sklearn.preprocessing import StandardScaler, OneHotEncoder
# from sklearn.impute import SimpleImputer
# from sklearn.compose import ColumnTransformer
# from sklearn.pipeline import Pipeline
# from sklearn.model_selection import train_test_split
# from sklearn.isotonic import IsotonicRegression

# import joblib
# from tqdm import tqdm

# import matplotlib
# matplotlib.use('Agg')
# import matplotlib.pyplot as plt
# import matplotlib.patches as mpatches

# from pgmpy.models import DiscreteBayesianNetwork
# from pgmpy.estimators import BayesianEstimator, HillClimbSearch, BIC

# from fairlearn.metrics import equalized_odds_difference, demographic_parity_difference
# from fairlearn.adversarial import AdversarialFairnessClassifier

# warnings.filterwarnings('ignore')
# logging.getLogger("pgmpy").setLevel(logging.ERROR)

# SEED        = 42
# DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# CACHE_DIR   = './cache'
# RESULTS_DIR = '/kaggle/working'
# for d in [CACHE_DIR, RESULTS_DIR]:
#     os.makedirs(d, exist_ok=True)

# def set_seed(s=SEED):
#     torch.manual_seed(s)
#     np.random.seed(s)
#     if torch.cuda.is_available():
#         torch.cuda.manual_seed_all(s)

# set_seed()
# if torch.cuda.is_available():
#     torch.cuda.empty_cache()
#     torch.backends.cudnn.deterministic = True
#     torch.backends.cudnn.benchmark     = False

# def floor2(x): return math.floor(abs(float(x)) * 100) / 100
# def r4(x):     return round(abs(float(x)), 4)

# DATASET_CONFIG: Dict[str, Dict] = {
#     "adult":     {"sens_attrs": [("sens_sex_train",     "sens_sex_test"),
#                                  ("sens_race_train",    "sens_race_test")]},
#     "compas":    {"sens_attrs": [("sens_race_train",    "sens_race_test"),
#                                  ("sens_sex_train",     "sens_sex_test")]},
#     "german":    {"sens_attrs": [("sens_age_train",     "sens_age_test"),
#                                  ("sens_sex_train",     "sens_sex_test")]},
#     "bank":      {"sens_attrs": [("sens_marital_train", "sens_marital_test"),
#                                  ("sens_job_train",     "sens_job_test")]},
#     "lawschool": {"sens_attrs": [("sens_race_train",    "sens_race_test"),
#                                  ("sens_gender_train",  "sens_gender_test")]},
#     "hospital":  {"sens_attrs": [("sens_race_train",    "sens_race_test"),
#                                  ("sens_sex_train",     "sens_sex_test")]},
# }

# ACC_FLOORS = {
#     "adult": 0.75, "compas": 0.62, "german": 0.62,
#     "bank": 0.77, "lawschool": 0.88, "hospital": 0.54,
# }

# EPOCH_CFG = {
#     "german":    dict(total=180, warmup=25, patience=25),
#     "adult":     dict(total=150, warmup=20, patience=20),
#     "compas":    dict(total=150, warmup=20, patience=20),
#     "bank":      dict(total=120, warmup=15, patience=18),
#     "lawschool": dict(total=80,  warmup=15, patience=15),
#     "hospital":  dict(total=120, warmup=15, patience=18),
# }

# # Per-dataset α caps: German/COMPAS need a much harder push
# ALPHA_CAPS = {
#     "german":    4.0,
#     "compas":    3.5,
#     "adult":     2.5,
#     "bank":      2.0,
#     "lawschool": 1.5,
#     "hospital":  2.5,
# }

# STAGE_RECORDS:       Dict[str, Dict] = {}
# MEDIATOR_SCORES_ALL: Dict[str, Dict] = {}
# TRAINING_CURVES:     Dict[str, Dict] = {}
# ABLATION_RECORDS:    Dict[str, Dict] = {}

# # ─────────────────────────────────────────────
# #  Shared helpers
# # ─────────────────────────────────────────────
# def to_dense(X):
#     arr = X.toarray() if hasattr(X, "toarray") else np.asarray(X)
#     return np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)

# def clean_num(s):
#     s = pd.to_numeric(s, errors='coerce').replace([np.inf, -np.inf], np.nan)
#     med = s.median()
#     return s.fillna(0.0 if np.isnan(med) else med)

# def safe_qcut(s, q=5):
#     s = clean_num(s)
#     if s.nunique() <= 1:
#         return pd.Series(0, index=s.index, dtype=int)
#     try:
#         return pd.qcut(s, q, labels=False, duplicates='drop').fillna(0).astype(int)
#     except:
#         return pd.Series(0, index=s.index, dtype=int)

# def num_pipe():
#     return Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())])

# def cat_pipe():
#     return Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
#                      ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])

# def _enc_bbn_objects(df):
#     df = df.copy()
#     for c in df.columns:
#         if df[c].dtype == object:
#             df[c] = df[c].astype('category').cat.codes.astype(int)
#     return df

# def _compute_mi(a, b):
#     return float(mutual_info_score(a.astype(int), b.astype(int)))

# def _entropy(a):
#     return float(mutual_info_score(a.astype(int), a.astype(int)))

# # ─────────────────────────────────────────────
# #  Label-leak / proxy filter — inlined per loader
# #  (no separate function called at top-level)
# # ─────────────────────────────────────────────
# LABEL_LEAK_COLNAMES = {
#     'readmit_binary', 'readmitted', 'two_year_recid', 'is_recid',
#     'decile_score', 'score_text', 'bar', 'bar2', 'bar_passed', 'bar_pass',
#     'zgpa', 'zfygpa', 'indxgrp', 'indxgrp2',
# }
# ARTIFACT_COLNAMES = {
#     'id', '_id', 'index', 'row', 'sample',
#     'race1', 'race2', 'race_bin', 'sex_bin', 'gender_bin',
#     'age_bin', 'marital_bin', 'job_bin',
#     'black', 'white', 'asian', 'hispanic',
# }

# def _filter_bbn_columns(df: pd.DataFrame, sens_col: str, y_col: str,
#                          extra_drop: tuple = ()) -> pd.DataFrame:
#     """
#     Per-dataset BBN column filter (inlined into each loader, not a global wrapper).
#     Drops: known label-leak names, artifact/ID cols, constants,
#            cols with MI(col,y)/H(y) > 0.60, cols with MI(col,s)/H(col) > 0.85.
#     """
#     drop = set(extra_drop)
#     y_vals = df[y_col].values.astype(int) if y_col in df.columns else None
#     s_vals = df[sens_col].values.astype(int) if sens_col in df.columns else None
#     h_y = _entropy(y_vals) if y_vals is not None else 0.0

#     for c in df.columns:
#         if c in (sens_col, y_col):
#             continue
#         cl = c.lower().strip()
#         if cl in LABEL_LEAK_COLNAMES or cl in ARTIFACT_COLNAMES:
#             drop.add(c); continue
#         if df[c].nunique() <= 1:
#             drop.add(c); continue
#         try:
#             col_int = df[c].values.astype(int)
#         except:
#             continue
#         if y_vals is not None and h_y > 0 and (_compute_mi(col_int, y_vals) / h_y) > 0.60:
#             drop.add(c); continue
#         if s_vals is not None:
#             h_col = _entropy(col_int)
#             if h_col > 0 and (_compute_mi(col_int, s_vals) / h_col) > 0.85:
#                 drop.add(c); continue

#     return df.drop(columns=[c for c in drop if c in df.columns])


# # ─────────────────────────────────────────────
# #  Dataset loaders
# #  Each loader returns a 'nn_feature_names' key:
# #  the list of expanded NN column names (after OHE).
# #  This is needed by the mediator split to correctly
# #  identify which NN dimensions correspond to each
# #  raw BBN feature. (FIX for bug 2)
# # ─────────────────────────────────────────────
# def _get_nn_feature_names(pre, num_c, cat_c):
#     """Extract expanded feature names from a fitted ColumnTransformer."""
#     names = list(num_c)
#     ohe = pre.named_transformers_['c'].named_steps['ohe']
#     for i, c in enumerate(cat_c):
#         cats = ohe.categories_[i]
#         names += [f"{c}={v}" for v in cats]
#     return names


# def load_adult(csv_path='/kaggle/input/datasets/anothertemp/all-datasets/adult.csv',
#                seed=SEED, use_cache=True):
#     cache = os.path.join(CACHE_DIR, 'fast_adult_v11.pkl')
#     if use_cache and os.path.exists(cache):
#         tqdm.write("  [cache] Adult"); return joblib.load(cache)
#     cols = ['age','workclass','fnlwgt','education','education-num','marital-status',
#             'occupation','relationship','race','sex','capital-gain','capital-loss',
#             'hours-per-week','native-country','income']
#     df = None
#     for sep in [', ', ',']:
#         try:
#             peek = pd.read_csv(csv_path, sep=sep, nrows=1, header=0)
#             if peek.shape[1] == 15:
#                 first = str(peek.iloc[0, 0]).strip()
#                 df = (pd.read_csv(csv_path, sep=sep, names=cols, header=None)
#                       if first.lstrip('-').isdigit()
#                       else pd.read_csv(csv_path, sep=sep, header=0))
#                 df.columns = cols; break
#         except:
#             continue
#     if df is None:
#         raise ValueError("Cannot parse Adult CSV")
#     for c in df.select_dtypes('object').columns:
#         df[c] = df[c].astype(str).str.strip()
#     df = df[~df.isin(['?']).any(axis=1)].reset_index(drop=True).drop(columns=['fnlwgt'])
#     df['y']        = df['income'].str.contains('>50K', na=False).astype(int)
#     df['sex_bin']  = (df['sex']  == 'Male').astype(int)
#     df['race_bin'] = (df['race'] == 'White').astype(int)
#     num_c = ['age','education-num','capital-gain','capital-loss','hours-per-week']
#     cat_c = ['workclass','education','marital-status','occupation','relationship','native-country']
#     for c in num_c: df[c] = clean_num(df[c])
#     for c in ['capital-gain','capital-loss']: df[c] = df[c].clip(upper=df[c].quantile(0.99))
#     X = df.drop(columns=['income','y','sex_bin','race_bin','sex','race'])
#     y = df['y'].values
#     X_tr,X_te,y_tr,y_te,ss_tr,ss_te,sr_tr,sr_te = train_test_split(
#         X, y, df['sex_bin'].values, df['race_bin'].values,
#         test_size=0.2, stratify=y, random_state=seed)
#     pre = ColumnTransformer([('n',num_pipe(),num_c),('c',cat_pipe(),cat_c)])
#     X_tr_nn = to_dense(pre.fit_transform(X_tr)); X_te_nn = to_dense(pre.transform(X_te))
#     nn_feat_names = _get_nn_feature_names(pre, num_c, cat_c)
#     bbn = df.drop(columns=['income']).copy()
#     for c in num_c: bbn[c] = safe_qcut(bbn[c], 5)
#     for c in cat_c+['sex','race']:
#         if c in bbn.columns: bbn[c] = bbn[c].astype('category').cat.codes.astype(int)
#     bbn['y'] = bbn['y'].astype(int); bbn = _enc_bbn_objects(bbn)
#     bbn_tr = bbn.loc[X_tr.index].reset_index(drop=True)
#     bbn_te = bbn.loc[X_te.index].reset_index(drop=True)
#     bbn_tr = _filter_bbn_columns(bbn_tr, 'sex_bin', 'y',
#                                   extra_drop=('sex','race','race_bin','sex_bin'))
#     bbn_te = _filter_bbn_columns(bbn_te, 'sex_bin', 'y',
#                                   extra_drop=('sex','race','race_bin','sex_bin'))
#     r = dict(X_train_nn=X_tr_nn, X_test_nn=X_te_nn,
#              bbn_train_df=bbn_tr, bbn_test_df=bbn_te,
#              y_train=y_tr, y_test=y_te,
#              sens_sex_train=ss_tr, sens_sex_test=ss_te,
#              sens_race_train=sr_tr, sens_race_test=sr_te,
#              num_cols=num_c, cat_cols=cat_c,
#              nn_feature_names=nn_feat_names)
#     if use_cache: joblib.dump(r, cache)
#     return r


# def load_compas(csv_path='/kaggle/input/datasets/anothertemp/all-datasets/compas-scores-two-years.csv',
#                 seed=SEED, use_cache=True):
#     cache = os.path.join(CACHE_DIR, 'fast_compas_v11.pkl')
#     if use_cache and os.path.exists(cache):
#         tqdm.write("  [cache] COMPAS"); return joblib.load(cache)
#     df = pd.read_csv(csv_path)
#     df = df.loc[
#         (df['days_b_screening_arrest'].between(-30, 30)) & (df['is_recid'] != -1) &
#         (df['c_charge_degree'] != 'O') & (df['score_text'] != 'N/A'),
#         ['age','c_charge_degree','race','age_cat','score_text','sex','priors_count',
#          'days_b_screening_arrest','decile_score','juv_other_count','juv_misd_count',
#          'juv_fel_count','c_charge_desc','is_recid','two_year_recid','c_jail_in','c_jail_out']
#     ].reset_index(drop=True)
#     df['race_bin'] = (df['race'] == 'African-American').astype(int)
#     df['sex_bin']  = (df['sex']  == 'Male').astype(int)
#     df['c_jail_in']  = pd.to_datetime(df['c_jail_in'])
#     df['c_jail_out'] = pd.to_datetime(df['c_jail_out'])
#     df['jail_time']  = (df['c_jail_out'] - df['c_jail_in']).dt.days.fillna(0)
#     df = df.drop(columns=['c_jail_in','c_jail_out'])
#     num_c = ['age','priors_count','days_b_screening_arrest',
#              'jail_time','juv_other_count','juv_misd_count','juv_fel_count']
#     cat_c = ['c_charge_degree','race','age_cat','sex','c_charge_desc']
#     for c in num_c: df[c] = clean_num(df[c])
#     X = df.drop(columns=['is_recid','two_year_recid','race_bin','sex_bin',
#                           'decile_score','score_text'])
#     y = df['two_year_recid'].values
#     X_tr,X_te,y_tr,y_te,sr_tr,sr_te,ss_tr,ss_te = train_test_split(
#         X, y, df['race_bin'], df['sex_bin'], test_size=0.2, stratify=y, random_state=seed)
#     pre = ColumnTransformer([('n',num_pipe(),num_c),('c',cat_pipe(),cat_c)])
#     X_tr_nn = to_dense(pre.fit_transform(X_tr)); X_te_nn = to_dense(pre.transform(X_te))
#     nn_feat_names = _get_nn_feature_names(pre, num_c, cat_c)
#     bbn = df.drop(columns=['is_recid','decile_score','score_text']).copy()
#     for c in num_c: bbn[c] = safe_qcut(bbn[c], 5)
#     for c in cat_c+['race','sex']: bbn[c] = bbn[c].astype('category').cat.codes.astype(int)
#     bbn = _enc_bbn_objects(bbn)
#     bbn_tr = bbn.loc[X_tr.index].reset_index(drop=True)
#     bbn_te = bbn.loc[X_te.index].reset_index(drop=True)
#     bbn_tr = _filter_bbn_columns(bbn_tr, 'race_bin', 'two_year_recid',
#                                   extra_drop=('race','sex','sex_bin','race_bin','is_recid'))
#     bbn_te = _filter_bbn_columns(bbn_te, 'race_bin', 'two_year_recid',
#                                   extra_drop=('race','sex','sex_bin','race_bin','is_recid'))
#     r = dict(X_train_nn=X_tr_nn, X_test_nn=X_te_nn,
#              bbn_train_df=bbn_tr, bbn_test_df=bbn_te,
#              y_train=y_tr, y_test=y_te,
#              sens_race_train=sr_tr.reset_index(drop=True), sens_race_test=sr_te.reset_index(drop=True),
#              sens_sex_train=ss_tr.reset_index(drop=True),  sens_sex_test=ss_te.reset_index(drop=True),
#              num_cols=num_c, cat_cols=cat_c,
#              nn_feature_names=nn_feat_names)
#     if use_cache: joblib.dump(r, cache)
#     return r


# def load_german(csv_path='/kaggle/input/datasets/anothertemp/all-datasets/german.data',
#                 seed=SEED, use_cache=True):
#     cache = os.path.join(CACHE_DIR, 'fast_german_v11.pkl')
#     if use_cache and os.path.exists(cache):
#         tqdm.write("  [cache] German"); return joblib.load(cache)
#     cols = ["status","duration","credit_history","purpose","amount","savings","employment",
#             "installment_rate","personal_status_sex","other_debtors","residence","property","age",
#             "other_installment_plans","housing","number_credits","job","people_liable",
#             "telephone","foreign_worker","credit"]
#     df = pd.read_csv(csv_path, sep=' ', names=cols)
#     sex_map = {'A91':'male','A92':'male','A93':'male','A94':'female','A95':'female'}
#     df['sex']     = df['personal_status_sex'].map(sex_map)
#     df['sex_bin'] = (df['sex'] == 'male').astype(int)
#     df['age_bin'] = (df['age'] >= 25).astype(int)
#     df['y']       = df['credit'].map({1:1, 2:0})
#     df = df.drop(columns=['personal_status_sex','sex','age','foreign_worker','credit'])
#     num_c = ["duration","amount","installment_rate","residence","number_credits","people_liable"]
#     cat_c = [c for c in df.columns if c not in num_c + ['sex_bin','age_bin','y']]
#     for c in num_c: df[c] = clean_num(df[c])
#     X = df.drop(columns=['y'])
#     y = df['y'].values
#     X_tr,X_te,y_tr,y_te,sa_tr,sa_te,ss_tr,ss_te = train_test_split(
#         X, y, df['age_bin'].values, df['sex_bin'].values,
#         test_size=0.2, stratify=y, random_state=seed)
#     pre = ColumnTransformer([('n',num_pipe(),num_c),('c',cat_pipe(),cat_c)])
#     X_tr_nn = to_dense(pre.fit_transform(X_tr)); X_te_nn = to_dense(pre.transform(X_te))
#     nn_feat_names = _get_nn_feature_names(pre, num_c, cat_c)
#     bbn = df.copy()
#     for c in num_c: bbn[c] = safe_qcut(bbn[c], 5)
#     for c in cat_c+['sex_bin','age_bin']: bbn[c] = bbn[c].astype('category').cat.codes.astype(int)
#     bbn = _enc_bbn_objects(bbn)
#     bbn_tr = bbn.loc[X_tr.index].reset_index(drop=True)
#     bbn_te = bbn.loc[X_te.index].reset_index(drop=True)
#     bbn_tr = _filter_bbn_columns(bbn_tr, 'age_bin', 'y', extra_drop=('sex_bin','age_bin'))
#     bbn_te = _filter_bbn_columns(bbn_te, 'age_bin', 'y', extra_drop=('sex_bin','age_bin'))
#     r = dict(X_train_nn=X_tr_nn, X_test_nn=X_te_nn,
#              bbn_train_df=bbn_tr, bbn_test_df=bbn_te,
#              y_train=y_tr, y_test=y_te,
#              sens_age_train=sa_tr, sens_age_test=sa_te,
#              sens_sex_train=ss_tr, sens_sex_test=ss_te,
#              num_cols=num_c, cat_cols=cat_c,
#              nn_feature_names=nn_feat_names)
#     if use_cache: joblib.dump(r, cache)
#     return r


# def load_bank(csv_path='/kaggle/input/datasets/anothertemp/all-datasets/bank-full.csv',
#               seed=SEED, use_cache=True):
#     cache = os.path.join(CACHE_DIR, 'fast_bank_v11.pkl')
#     if use_cache and os.path.exists(cache):
#         tqdm.write("  [cache] Bank"); return joblib.load(cache)
#     try:
#         df = pd.read_csv(csv_path, sep=';')
#     except:
#         df = pd.read_csv(csv_path)
#     df = df.apply(lambda x: x.str.lower() if x.dtype == 'object' else x)
#     tc = 'y' if 'y' in df.columns else ('deposit' if 'deposit' in df.columns else df.columns[-1])
#     df = df[~df.isin(['unknown']).any(axis=1)].reset_index(drop=True)
#     df['y']           = np.where(df[tc].isin(['yes','y','true','1']), 1, 0)
#     df['marital_bin'] = (df['marital'] == df['marital'].value_counts().idxmax()).astype(int)
#     df['job_bin']     = (df['job']     == df['job'].value_counts().idxmax()).astype(int)
#     cat_c = [c for c in ['job','education','default','housing','loan','contact','month','poutcome'] if c in df.columns]
#     num_c = [c for c in ['age','balance','day','duration','campaign','pdays','previous'] if c in df.columns]
#     for c in num_c: df[c] = clean_num(df[c])
#     for c in ['balance','duration','pdays','previous']:
#         if c in df.columns: df[c] = df[c].clip(upper=df[c].quantile(0.99))
#     X = df.drop(columns=['y','marital_bin','job_bin'])
#     y = df['y'].values
#     X_tr,X_te,y_tr,y_te,sm_tr,sm_te,sj_tr,sj_te = train_test_split(
#         X, y, df['marital_bin'], df['job_bin'], test_size=0.2, stratify=y, random_state=seed)
#     pre = ColumnTransformer([('n',num_pipe(),num_c),('c',cat_pipe(),cat_c)])
#     X_tr_nn = to_dense(pre.fit_transform(X_tr)); X_te_nn = to_dense(pre.transform(X_te))
#     nn_feat_names = _get_nn_feature_names(pre, num_c, cat_c)
#     bbn = df.copy()
#     for c in num_c: bbn[c] = safe_qcut(bbn[c], 5)
#     for c in cat_c+['marital','job']: bbn[c] = bbn[c].astype('category').cat.codes.astype(int)
#     bbn = _enc_bbn_objects(bbn)
#     bbn_tr = bbn.loc[X_tr.index].reset_index(drop=True)
#     bbn_te = bbn.loc[X_te.index].reset_index(drop=True)
#     bbn_tr = _filter_bbn_columns(bbn_tr, 'marital_bin', 'y',
#                                   extra_drop=('marital','job','marital_bin','job_bin'))
#     bbn_te = _filter_bbn_columns(bbn_te, 'marital_bin', 'y',
#                                   extra_drop=('marital','job','marital_bin','job_bin'))
#     r = dict(X_train_nn=X_tr_nn, X_test_nn=X_te_nn,
#              bbn_train_df=bbn_tr, bbn_test_df=bbn_te,
#              y_train=y_tr, y_test=y_te,
#              sens_marital_train=sm_tr.reset_index(drop=True), sens_marital_test=sm_te.reset_index(drop=True),
#              sens_job_train=sj_tr.reset_index(drop=True),     sens_job_test=sj_te.reset_index(drop=True),
#              num_cols=num_c, cat_cols=cat_c,
#              nn_feature_names=nn_feat_names)
#     if use_cache: joblib.dump(r, cache)
#     return r


# def load_lawschool(csv_path='/kaggle/input/datasets/anothertemp/all-datasets/bar_pass_prediction.csv',
#                    use_cache=True):
#     cache = os.path.join(CACHE_DIR, 'fast_lawschool_v11.pkl')
#     if use_cache and os.path.exists(cache):
#         tqdm.write("  [cache] LawSchool"); return joblib.load(cache)
#     df = pd.read_csv(csv_path)
#     df.columns = [c.strip().lower() for c in df.columns]
#     df = df.dropna(subset=['pass_bar','race','sex'])
#     for c in df.select_dtypes(include=[np.number]).columns: df[c] = clean_num(df[c])
#     mc = df['race'].value_counts().idxmax()
#     df['race_bin']   = (df['race'] == mc).astype(int)
#     gm = {v:i for i,v in enumerate(sorted(df['sex'].unique()))}
#     df['gender_bin'] = df['sex'].map(gm)
#     num_c = [c for c in ['lsat','ugpa','fam_inc','age']
#              if c in df.columns and df[c].nunique() > 1]
#     cat_c = [c for c in ['tier','cluster','fulltime','parttime'] if c in df.columns]
#     X = df[num_c + cat_c + ['race','sex']]
#     y = df['pass_bar'].values
#     X_tr,X_te,y_tr,y_te,sr_tr,sr_te,sg_tr,sg_te = train_test_split(
#         X, y, df['race_bin'], df['gender_bin'], test_size=0.2, stratify=y, random_state=SEED)
#     pre = ColumnTransformer([('n',num_pipe(),num_c),('c',cat_pipe(),cat_c+['race','sex'])])
#     X_tr_nn = to_dense(pre.fit_transform(X_tr)); X_te_nn = to_dense(pre.transform(X_te))
#     nn_feat_names = _get_nn_feature_names(pre, num_c, cat_c+['race','sex'])
#     bbn = df.copy()
#     for c in num_c: bbn[c] = safe_qcut(df[c], 5)
#     for c in cat_c+['race','sex']:
#         if c in bbn.columns: bbn[c] = bbn[c].astype('category').cat.codes.astype(int)
#     if 'pass_bar' in bbn.columns and bbn['pass_bar'].dtype == object:
#         bbn['pass_bar'] = bbn['pass_bar'].astype('category').cat.codes.astype(int)
#     bbn = _enc_bbn_objects(bbn)
#     bbn_tr = bbn.loc[X_tr.index].reset_index(drop=True)
#     bbn_te = bbn.loc[X_te.index].reset_index(drop=True)
#     bbn_tr = _filter_bbn_columns(bbn_tr, 'race_bin', 'pass_bar',
#                                   extra_drop=('race','sex','race_bin','gender_bin'))
#     bbn_te = _filter_bbn_columns(bbn_te, 'race_bin', 'pass_bar',
#                                   extra_drop=('race','sex','race_bin','gender_bin'))
#     r = dict(X_train_nn=X_tr_nn, X_test_nn=X_te_nn,
#              bbn_train_df=bbn_tr, bbn_test_df=bbn_te,
#              y_train=y_tr, y_test=y_te,
#              sens_race_train=sr_tr.reset_index(drop=True),   sens_race_test=sr_te.reset_index(drop=True),
#              sens_gender_train=sg_tr.reset_index(drop=True), sens_gender_test=sg_te.reset_index(drop=True),
#              num_cols=num_c, cat_cols=cat_c,
#              nn_feature_names=nn_feat_names)
#     if use_cache: joblib.dump(r, cache)
#     return r


# def load_hospital(csv_path='/kaggle/input/datasets/anothertemp/all-datasets/diabetes_hospital_fairlearn.csv',
#                   seed=SEED, use_cache=True):
#     cache = os.path.join(CACHE_DIR, 'fast_hospital_v11.pkl')
#     if use_cache and os.path.exists(cache):
#         tqdm.write("  [cache] Hospital"); return joblib.load(cache)
#     df = pd.read_csv(csv_path)
#     df = df.drop(columns=[c for c in ['max_glu_serum','A1Cresult','readmitted.1'] if c in df.columns])
#     df = df[~df.isin(['Missing']).any(axis=1)].dropna(subset=['race','gender']).reset_index(drop=True)
#     age_map = {"'0-10'":5,"'10-20'":15,"'20-30'":25,"'30-40'":35,"'40-50'":45,
#                "'50-60'":55,"'60-70'":65,"'70-80'":75,"'80-90'":85,"'90-100'":95,
#                "'30 years or younger'":20,"'30-60 years'":45,"'Over 60 years'":70}
#     df['age']        = df['age'].replace(age_map).astype(float)
#     df['y']          = df['readmit_binary'].astype(int)
#     mc               = df['race'].value_counts().idxmax()
#     df['race_bin']   = (df['race'] == mc).astype(int)
#     df['gender_bin'] = df['gender'].map({'Male':0,'Female':1}).fillna(0).astype(int)
#     cat_c = ['discharge_disposition_id','admission_source_id','medical_specialty',
#              'primary_diagnosis','insulin','change','diabetesMed']
#     num_c = ['age','time_in_hospital','num_lab_procedures','num_procedures','num_medications',
#              'number_diagnoses','had_emergency','had_inpatient_days','had_outpatient_days','medicare','medicaid']
#     for c in num_c: df[c] = clean_num(df[c])
#     X = df.drop(columns=['readmit_binary','y','race_bin','gender_bin'])
#     y = df['y'].values
#     X_tr,X_te,y_tr,y_te,sr_tr,sr_te,sg_tr,sg_te = train_test_split(
#         X, y, df['race_bin'], df['gender_bin'], test_size=0.2, stratify=y, random_state=seed)
#     pre = ColumnTransformer([('n',num_pipe(),num_c),('c',cat_pipe(),cat_c)])
#     X_tr_nn = to_dense(pre.fit_transform(X_tr)); X_te_nn = to_dense(pre.transform(X_te))
#     nn_feat_names = _get_nn_feature_names(pre, num_c, cat_c)
#     bbn = df.copy()
#     for c in num_c: bbn[c] = safe_qcut(bbn[c], 5)
#     for c in cat_c+['race','gender']: bbn[c] = bbn[c].astype('category').cat.codes.astype(int)
#     bbn = _enc_bbn_objects(bbn)
#     bbn_tr = bbn.loc[X_tr.index].reset_index(drop=True)
#     bbn_te = bbn.loc[X_te.index].reset_index(drop=True)
#     bbn_tr = _filter_bbn_columns(bbn_tr, 'race_bin', 'y',
#                                   extra_drop=('race','gender','race_bin','gender_bin','readmit_binary'))
#     bbn_te = _filter_bbn_columns(bbn_te, 'race_bin', 'y',
#                                   extra_drop=('race','gender','race_bin','gender_bin','readmit_binary'))
#     r = dict(X_train_nn=X_tr_nn, X_test_nn=X_te_nn,
#              bbn_train_df=bbn_tr, bbn_test_df=bbn_te,
#              y_train=y_tr, y_test=y_te,
#              sens_race_train=sr_tr.reset_index(drop=True), sens_race_test=sr_te.reset_index(drop=True),
#              sens_sex_train=sg_tr.reset_index(drop=True),  sens_sex_test=sg_te.reset_index(drop=True),
#              num_cols=num_c, cat_cols=cat_c,
#              nn_feature_names=nn_feat_names)
#     if use_cache: joblib.dump(r, cache)
#     return r


# # ─────────────────────────────────────────────
# #  BBN structure learning
# #  FIX: run HillClimb on the full bbn_df (not a subsample).
# #  Subsample only for the CPT fit (which is expensive for large n).
# #  The separate DiscreteBayesianNetwork.fit() call is removed —
# #  it was redundant and caused the "1%" appearance in the log.
# # ─────────────────────────────────────────────
# def _learn_graph_structure(bbn_df: pd.DataFrame, sens_col: str,
#                              label_col: str, max_nodes: int = 18) -> list:
#     cols = [c for c in bbn_df.columns if c != label_col]
#     if sens_col not in cols:
#         cols = [sens_col] + cols
#     cols = cols[:max_nodes]
#     sub = bbn_df[cols + ([label_col] if label_col in bbn_df.columns else [])].copy()
#     try:
#         hc  = HillClimbSearch(sub)
#         est = hc.estimate(scoring_method=BIC(sub), max_iter=500, max_indegree=4)
#         edges = list(est.edges())
#         causal = [(u, v) for u, v in edges if u == sens_col or v == label_col]
#         return causal if causal else edges
#     except:
#         return [(sens_col, c) for c in cols if c != sens_col]


# # ─────────────────────────────────────────────
# #  NIE — unified (single function, replaces both
# #  _nie_score and _gate3_nie_mediation from v9/v10)
# # ─────────────────────────────────────────────
# def _compute_nie(bbn_df: pd.DataFrame, sens_col: str,
#                  med_col: str, label_col: str) -> Tuple[float, float, bool]:
#     """
#     Natural Indirect Effect: NIE(S→M→Y) = MI(S,M) × E[MI(M,Y|S=s)].
#     Returns (nie_absolute, nie_ratio, is_valid_mediator).
#     is_valid when nie_ratio > 0.01 AND nie > 1e-6.
#     """
#     s = bbn_df[sens_col].values.astype(int)
#     m = bbn_df[med_col].values.astype(int)
#     y = bbn_df[label_col].values.astype(int)

#     mi_sm = _compute_mi(s, m)
#     mi_sy = _compute_mi(s, y)

#     cond_mi_parts = []
#     weights       = []
#     for sv in np.unique(s):
#         mask = s == sv
#         if mask.sum() > 15:
#             cond_mi_parts.append(_compute_mi(m[mask], y[mask]))
#             weights.append(mask.sum())

#     if not cond_mi_parts:
#         return 0.0, 0.0, False

#     w_total  = sum(weights)
#     avg_cond = sum(c * w for c, w in zip(cond_mi_parts, weights)) / w_total
#     nie      = float(mi_sm * avg_cond)
#     nie_ratio = nie / max(mi_sy, 1e-9)
#     is_valid  = nie_ratio > 0.01 and nie > 1e-6

#     return nie, nie_ratio, is_valid


# @dataclass
# class MediatorBiasScores:
#     sens_col:    str
#     label_col:   str
#     scores:      Dict[str, float] = field(default_factory=dict)
#     nie_scores:  Dict[str, float] = field(default_factory=dict)
#     nie_ratio:   Dict[str, float] = field(default_factory=dict)
#     sens_mi:     float = 0.0
#     verification: Dict[str, dict] = field(default_factory=dict)

#     def fit(self, bbn_df: pd.DataFrame):
#         if self.sens_col not in bbn_df.columns or self.label_col not in bbn_df.columns:
#             return self
#         s = bbn_df[self.sens_col].values.astype(int)
#         y = bbn_df[self.label_col].values.astype(int)
#         self.sens_mi = _compute_mi(s, y)
#         h_s = _entropy(s); h_y = _entropy(y)

#         candidates = [c for c in bbn_df.columns if c not in (self.sens_col, self.label_col)]
#         raw_scores = {}

#         for feat in candidates:
#             f = bbn_df[feat].values.astype(int)
#             mi_sf = _compute_mi(s, f)
#             mi_fy = _compute_mi(f, y)
#             h_f   = _entropy(f)

#             mi_sf_n = mi_sf / max(h_s, h_f, 1e-9)
#             mi_fy_n = mi_fy / max(h_f, h_y, 1e-9)

#             nie, nie_r, valid = _compute_nie(bbn_df, self.sens_col, feat, self.label_col)

#             self.verification[feat] = dict(
#                 nie=nie, nie_ratio=nie_r, mi_sf_n=mi_sf_n, mi_fy_n=mi_fy_n, valid=valid)
#             self.nie_ratio[feat] = nie_r

#             if not valid:
#                 raw_scores[feat]      = 0.0
#                 self.nie_scores[feat] = 0.0
#                 continue

#             raw_scores[feat]      = float(0.35*mi_sf_n + 0.35*mi_fy_n + 0.30*nie_r)
#             self.nie_scores[feat] = nie

#         valid_sc = {k: v for k, v in raw_scores.items() if v > 0}
#         if valid_sc:
#             mx = max(valid_sc.values()) + 1e-9
#             self.scores = {k: (v/mx if v > 0 else 0.0) for k, v in raw_scores.items()}
#         else:
#             self.scores = raw_scores

#         valid_nie = {k: v for k, v in self.nie_scores.items() if v > 0}
#         if valid_nie:
#             mx_nie = max(valid_nie.values()) + 1e-9
#             self.nie_scores = {k: (v/mx_nie if v > 0 else 0.0) for k, v in self.nie_scores.items()}
#         return self

#     def top_mediators(self, k: int = 6, threshold: float = 0.03) -> List[Tuple[str, float]]:
#         verified = {f for f, vr in self.verification.items() if vr.get('valid', False)}
#         ranked   = sorted(verified, key=lambda f: self.scores.get(f, 0.0), reverse=True)
#         return [(f, self.scores[f]) for f in ranked if self.scores.get(f, 0.0) >= threshold][:k]


# TOP_K_MEDIATORS = 6


# @dataclass
# class BBNPathAnalyzer:
#     sens_col:    str
#     label_col:   str
#     bias_scores: MediatorBiasScores = field(default=None)
#     _pathway_weights: Dict[str, float] = field(default_factory=dict, repr=False)
#     _nie_weights:     Dict[str, float] = field(default_factory=dict, repr=False)
#     _learned_edges:   list = field(default_factory=list, repr=False)
#     BBN_SUBSAMPLE = 800

#     def fit(self, bbn_df: pd.DataFrame):
#         self.bias_scores = MediatorBiasScores(sens_col=self.sens_col, label_col=self.label_col)
#         self.bias_scores.fit(bbn_df)
#         top  = self.bias_scores.top_mediators(k=TOP_K_MEDIATORS, threshold=0.03)
#         self._pathway_weights = {f: s for f, s in top}
#         self._nie_weights     = {f: self.bias_scores.nie_scores.get(f, 0.0) for f, _ in top}
#         top_feats = [f for f, _ in top]

#         # FIX: run HillClimb on the full bbn_df (limited to max_nodes cols)
#         # Only subsample for the CPT fit, not for structure learning
#         bbn_cols = ([self.sens_col] + top_feats +
#                     ([self.label_col] if self.label_col in bbn_df.columns else []))
#         bbn_cols = [c for c in bbn_cols if c in bbn_df.columns]

#         if len(bbn_cols) >= 3:
#             self._learned_edges = _learn_graph_structure(
#                 bbn_df[bbn_cols], self.sens_col, self.label_col, max_nodes=len(bbn_cols))
#         else:
#             self._learned_edges = [(self.sens_col, f) for f in top_feats if f in bbn_df.columns]

#         # No separate DiscreteBayesianNetwork.fit() — it was redundant with HillClimb
#         return self

#     def get_pathway_weights(self) -> Dict[str, float]:    return self._pathway_weights
#     def get_nie_weights(self)     -> Dict[str, float]:    return self._nie_weights
#     def get_sens_mi(self)         -> float:               return self.bias_scores.sens_mi if self.bias_scores else 0.0
#     def get_feature_bias_scores(self) -> Dict[str, float]: return self.bias_scores.scores if self.bias_scores else {}
#     def get_verification(self)    -> Dict[str, dict]:     return self.bias_scores.verification if self.bias_scores else {}
#     def get_learned_edges(self)   -> list:                return self._learned_edges


# # ─────────────────────────────────────────────
# #  Mediator split — FIX for bug 2
# #  Map BBN mediator raw feature names → NN dimension indices
# #  using the actual expanded feature name list from the preprocessor.
# # ─────────────────────────────────────────────
# def _split_x_by_mediators(X_nn: np.ndarray, mediator_names: list,
#                             nn_feature_names: list) -> Tuple[np.ndarray, np.ndarray]:
#     """
#     FIX (bug 2): Previously mapped BBN column indices → NN dims, which is wrong
#     because NN dims come from a ColumnTransformer with OHE.
#     Now: find all NN dim indices whose expanded name starts with any mediator base name.
#     """
#     if not mediator_names or not nn_feature_names:
#         # no-mediator fallback: first quarter → med path, rest → task path
#         half = max(X_nn.shape[1] // 4, 1)
#         return X_nn[:, :half], X_nn[:, half:]

#     med_idx = []
#     for i, fname in enumerate(nn_feature_names):
#         # numeric feature: exact match; categorical OHE: starts with "colname="
#         base = fname.split('=')[0]
#         if base in mediator_names or fname in mediator_names:
#             med_idx.append(i)

#     if not med_idx:
#         half = max(X_nn.shape[1] // 4, 1)
#         return X_nn[:, :half], X_nn[:, half:]

#     task_idx = [i for i in range(X_nn.shape[1]) if i not in set(med_idx)]
#     if not task_idx:
#         task_idx = list(range(X_nn.shape[1]))
#     return X_nn[:, med_idx], X_nn[:, task_idx]


# def _split_x_ablation_no_bbn(X_nn: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
#     """
#     FIX (ablation bug 5): In no_bbn mode, force all features into the task path.
#     This gives the ablation a fair comparison (no accidental mediator routing).
#     A tiny dummy 'med' slice of 1 zero column is returned so the encoder dim doesn't break.
#     """
#     dummy = np.zeros((X_nn.shape[0], 1), dtype=X_nn.dtype)
#     return dummy, X_nn


# # ─────────────────────────────────────────────
# #  Neural architecture (unchanged)
# # ─────────────────────────────────────────────
# class GradRevFn(torch.autograd.Function):
#     @staticmethod
#     def forward(ctx, x, alpha):
#         ctx.alpha = alpha
#         return x.view_as(x)
#     @staticmethod
#     def backward(ctx, grad_output):
#         return -ctx.alpha * grad_output, None


# class SplitInputEncoder(nn.Module):
#     def __init__(self, in_dim_med: int, in_dim_task: int,
#                  hidden: int = 256, z_med_dim: int = 64, z_task_dim: int = 128,
#                  dropout: float = 0.3):
#         super().__init__()
#         self.z_med_dim  = z_med_dim
#         self.z_task_dim = z_task_dim
#         self.med_encoder = nn.Sequential(
#             nn.Linear(in_dim_med, max(hidden // 2, 16)), nn.LayerNorm(max(hidden // 2, 16)), nn.GELU(), nn.Dropout(dropout),
#             nn.Linear(max(hidden // 2, 16), max(hidden // 2, 16)), nn.LayerNorm(max(hidden // 2, 16)), nn.GELU(), nn.Dropout(dropout),
#             nn.Linear(max(hidden // 2, 16), z_med_dim), nn.LayerNorm(z_med_dim),
#         )
#         self.task_encoder = nn.Sequential(
#             nn.Linear(in_dim_task, hidden), nn.LayerNorm(hidden), nn.GELU(), nn.Dropout(dropout),
#             nn.Linear(hidden, hidden), nn.LayerNorm(hidden), nn.GELU(), nn.Dropout(dropout),
#             nn.Linear(hidden, z_task_dim), nn.LayerNorm(z_task_dim),
#         )

#     def forward(self, x_med, x_task):
#         return self.task_encoder(x_task), self.med_encoder(x_med)


# class TaskHead(nn.Module):
#     def __init__(self, z_dim: int = 128):
#         super().__init__()
#         self.net = nn.Sequential(
#             nn.Linear(z_dim, z_dim // 2), nn.GELU(),
#             nn.Linear(z_dim // 2, 1),
#         )
#     def forward(self, z): return self.net(z).squeeze(-1)


# class TaskAdversary(nn.Module):
#     def __init__(self, z_task_dim: int = 128, hidden: int = 128, nie_scalar: float = 1.0):
#         super().__init__()
#         self.nie_scalar = nie_scalar
#         self.net = nn.Sequential(
#             nn.Linear(z_task_dim, hidden), nn.LayerNorm(hidden), nn.GELU(), nn.Dropout(0.2),
#             nn.Linear(hidden, hidden // 2), nn.GELU(),
#             nn.Linear(hidden // 2, 2),
#         )
#     def forward(self, z_task, alpha: float = 1.0):
#         return self.net(GradRevFn.apply(z_task, alpha * self.nie_scalar))


# class LeakageProbe(nn.Module):
#     def __init__(self, z_task_dim: int = 128):
#         super().__init__()
#         self.net = nn.Linear(z_task_dim, 2)
#     def forward(self, z): return self.net(z)


# class MediatorReconstructor(nn.Module):
#     def __init__(self, z_med_dim: int = 64, n_mediators: int = 5, hidden: int = 128):
#         super().__init__()
#         self.net = nn.Sequential(
#             nn.Linear(z_med_dim, hidden), nn.GELU(),
#             nn.Linear(hidden, hidden), nn.GELU(),
#             nn.Linear(hidden, n_mediators),
#         )
#     def forward(self, z): return self.net(z)


# # ─────────────────────────────────────────────
# #  Loss functions
# # ─────────────────────────────────────────────
# def fair_loss_eo(logit, y, s, margin=0.02):
#     p = torch.sigmoid(logit)
#     tprs, fprs = [], []
#     for g in (0, 1):
#         pm = (s == g) & (y == 1); nm = (s == g) & (y == 0)
#         tprs.append(p[pm].mean() if pm.sum() > 1 else torch.tensor(0.5, device=logit.device))
#         fprs.append(p[nm].mean() if nm.sum() > 1 else torch.tensor(0.5, device=logit.device))
#     return F.relu(torch.max(torch.abs(tprs[0]-tprs[1]), torch.abs(fprs[0]-fprs[1])) - margin)


# def fair_loss_dp(logit, s, margin=0.02):
#     p = torch.sigmoid(logit)
#     p0 = p[s==0].mean() if (s==0).sum() > 1 else torch.tensor(0.5, device=logit.device)
#     p1 = p[s==1].mean() if (s==1).sum() > 1 else torch.tensor(0.5, device=logit.device)
#     return F.relu(torch.abs(p0-p1) - margin)


# def fair_loss_eo_bounded(logit, y, s, margin=0.08):
#     """
#     FIX (bug 4): EO bounding weight raised to 0.08 (was 0.25 absolute but weight 0.4 too weak).
#     This is called with weight 1.0 from the DP training branch.
#     """
#     return fair_loss_eo(logit, y, s, margin=margin)


# def weighted_orthogonality_loss(z_med, z_task, med_weights):
#     zm = z_med  - z_med.mean(0, keepdim=True)
#     zt = z_task - z_task.mean(0, keepdim=True)
#     cross = torch.mm(F.normalize(zm, dim=1).T, F.normalize(zt, dim=1)) / zm.shape[0]
#     csq   = cross ** 2
#     if med_weights is not None and med_weights.shape[0] == csq.shape[0]:
#         return (csq * med_weights.unsqueeze(1).expand_as(csq)).mean()
#     return csq.mean()


# def counterfactual_consistency_loss(z_med, s):
#     """
#     Penalise correlation between z_med and the sensitive attribute.
#     (Simplified from v10 — the original was computing but not using
#     the counterfactual logits, making the loss a no-op in effect.)
#     """
#     z_med_c = z_med - z_med.mean(0, keepdim=True)
#     s_c     = s.float() - s.float().mean()
#     correlation = (z_med_c * s_c.unsqueeze(1)).abs().mean()
#     return correlation


# # ─────────────────────────────────────────────
# #  ROC / post-processing
# # ─────────────────────────────────────────────
# def _roc_points(proba, y):
#     thresholds = np.unique(proba)
#     pos = y.sum(); neg = len(y) - pos
#     tprs, fprs = [], []
#     for t in thresholds:
#         pred = (proba >= t).astype(int)
#         tp = ((pred==1)&(y==1)).sum(); fp = ((pred==1)&(y==0)).sum()
#         tprs.append(tp/pos if pos>0 else 0.0)
#         fprs.append(fp/neg if neg>0 else 0.0)
#     fprs_a = np.array(fprs); tprs_a = np.array(tprs)
#     idx    = np.argsort(fprs_a)
#     return fprs_a[idx], tprs_a[idx]


# def _convex_hull_roc(fprs, tprs, n=100):
#     pts = np.vstack([[0.,0.], np.stack([fprs,tprs],1), [1.,1.]])
#     pts = pts[np.argsort(pts[:,0])]
#     hf  = [pts[0,0]]; ht = [pts[0,1]]
#     for i in range(1, len(pts)):
#         while len(hf) >= 2:
#             dx1=hf[-1]-hf[-2]; dy1=ht[-1]-ht[-2]
#             dx2=pts[i,0]-hf[-2]; dy2=pts[i,1]-ht[-2]
#             if dx1*dy2 <= dy1*dx2: hf.pop(); ht.pop()
#             else: break
#         hf.append(pts[i,0]); ht.append(pts[i,1])
#     xp = np.linspace(0,1,n)
#     return xp, np.interp(xp, np.array(hf), np.array(ht))


# def calibrate_proba(proba_tr, y_tr, proba_te):
#     try:
#         iso = IsotonicRegression(out_of_bounds='clip')
#         iso.fit(proba_tr, y_tr)
#         return iso.predict(proba_te).astype(np.float64)
#     except:
#         return proba_te


# def hardt_postprocess(proba, y, s, acc_floor, target='eo',
#                        proba_cal_src=None, n_roc_pts=100):
#     """
#     FIX (bug 3): Accept Hardt result if metric improves meaningfully OR
#     if current metric is very high (>0.10) even at some accuracy cost.
#     acc tolerance widened to 0.035 for high-violation datasets.
#     """
#     s = s.astype(int)
#     if proba_cal_src is not None:
#         proba = calibrate_proba(proba_cal_src[0], proba_cal_src[1], proba)
#     roc = {}
#     for g in (0,1):
#         m = s==g
#         if m.sum() < 10: roc[g] = (np.array([0.,1.]), np.array([0.,1.]))
#         else:
#             rf, rt = _roc_points(proba[m], y[m])
#             roc[g] = _convex_hull_roc(rf, rt, n=n_roc_pts)

#     pos0=(y[s==0]==1).sum(); neg0=(y[s==0]==0).sum()
#     pos1=(y[s==1]==1).sum(); neg1=(y[s==1]==0).sum()
#     n0=(s==0).sum(); n1=(s==1).sum(); n=len(y)
#     floor_guard = acc_floor * 0.96  # slightly softer

#     def pred_from_roc(g, fp_v, tp_v):
#         m  = s==g; pg=proba[m]; yg=y[m]
#         pos_m=yg==1; neg_m=yg==0
#         pred_g = np.zeros(m.sum(), dtype=np.float32)
#         if pos_m.sum()>0:
#             n_tp = int(round(tp_v*pos_m.sum()))
#             pred_g[np.where(pos_m)[0][np.argsort(-pg[pos_m])[:n_tp]]] = 1.0
#         if neg_m.sum()>0:
#             n_fp = int(round(fp_v*neg_m.sum()))
#             pred_g[np.where(neg_m)[0][np.argsort(-pg[neg_m])[:n_fp]]] = 1.0
#         return pred_g

#     best_metric=1.0; best_acc=0.0; best_pred=None
#     fpr0,tpr0=roc[0]; fpr1,tpr1=roc[1]
#     for i in range(len(fpr0)):
#         for j in range(len(fpr1)):
#             f0,t0=fpr0[i],tpr0[i]; f1,t1=fpr1[j],tpr1[j]
#             if target=='eo': metric=max(abs(t0-t1),abs(f0-f1))
#             else:
#                 pr0=t0*(pos0/n0)+f0*(neg0/n0) if n0>0 else 0.
#                 pr1=t1*(pos1/n1)+f1*(neg1/n1) if n1>0 else 0.
#                 metric=abs(pr0-pr1)
#             acc=((t0*pos0+(1-f0)*neg0)/n0*n0+(t1*pos1+(1-f1)*neg1)/n1*n1)/n if n>0 else 0.
#             if acc < floor_guard: continue
#             if metric<best_metric or (abs(metric-best_metric)<1e-6 and acc>best_acc):
#                 best_metric=metric; best_acc=acc; best_pred=(f0,t0,f1,t1)

#     if best_pred is None:
#         tqdm.write(f"    [Hardt] no feasible point, using 0.5 threshold")
#         pred=(proba>=0.5).astype(int)
#         try:
#             m=r4(equalized_odds_difference(y,pred,sensitive_features=s) if target=='eo'
#                  else demographic_parity_difference(y,pred,sensitive_features=s))
#         except: m=1.0
#         return pred,m,float((pred==y).mean())

#     f0,t0,f1,t1=best_pred
#     pred=np.zeros(n,dtype=int)
#     pred[s==0]=pred_from_roc(0,f0,t0).astype(int)
#     pred[s==1]=pred_from_roc(1,f1,t1).astype(int)
#     try:
#         fm=r4(equalized_odds_difference(y,pred,sensitive_features=s) if target=='eo'
#               else demographic_parity_difference(y,pred,sensitive_features=s))
#     except: fm=r4(best_metric)
#     return pred,fm,float((pred==y).mean())


# @torch.no_grad()
# def get_proba(enc, task_hd, X_med, X_task):
#     enc.eval(); task_hd.eval()
#     dev = next(enc.parameters()).device
#     if not isinstance(X_med, torch.Tensor):
#         X_med  = torch.tensor(X_med,  dtype=torch.float32)
#         X_task = torch.tensor(X_task, dtype=torch.float32)
#     z_task, _ = enc(X_med.to(dev), X_task.to(dev))
#     return torch.sigmoid(task_hd(z_task)).cpu().numpy()


# # ─────────────────────────────────────────────
# #  Core training loop
# # ─────────────────────────────────────────────
# def _train_single(dataset_name, data, bbn_analyzer, primary_sens_train,
#                   primary_sens_test, acc_floor, target, ablation_mode='full'):
#     cfg      = EPOCH_CFG.get(dataset_name, dict(total=150, warmup=20, patience=20))
#     total_ep = cfg['total']
#     warmup_ep = cfg['warmup']
#     patience  = cfg['patience']
#     batch     = 2048

#     use_bbn  = ablation_mode not in ('no_bbn',)
#     use_adv  = ablation_mode not in ('no_adv',)
#     use_post = ablation_mode not in ('no_post',)

#     y_np = data['y_test']
#     s_np = primary_sens_test
#     nn_feat_names = data.get('nn_feature_names', [])
#     alpha_cap = ALPHA_CAPS.get(dataset_name, 2.5)

#     if use_bbn:
#         sens_mi   = bbn_analyzer.get_sens_mi()
#         pw        = bbn_analyzer.get_pathway_weights()
#         nie_w     = bbn_analyzer.get_nie_weights()
#         nie_scalar = float(np.clip(1.0 + np.mean(list(nie_w.values())) * 5.0, 1.0, 4.0)) if nie_w else 1.0
#         global_alpha = float(np.clip(0.5 + sens_mi * 8.0, 0.5, alpha_cap))
#         top_med_names = list(pw.keys())[:TOP_K_MEDIATORS]
#         bbn_cols  = list(data['bbn_train_df'].columns)
#         med_col_idx = [bbn_cols.index(m) for m in top_med_names if m in bbn_cols]
#         n_med_feats = len(med_col_idx)
#         bbn_tr_arr  = data['bbn_train_df'].values.astype(np.float32)
#         med_w_np    = np.array([pw.get(f, 0.0) for f in top_med_names], dtype=np.float32)
#         if med_w_np.sum() > 0: med_w_np /= med_w_np.sum()

#         # FIX (bug 2): use corrected split
#         X_tr_med, X_tr_task = _split_x_by_mediators(
#             data['X_train_nn'], top_med_names, nn_feat_names)
#         X_te_med, X_te_task = _split_x_by_mediators(
#             data['X_test_nn'], top_med_names, nn_feat_names)
#     else:
#         sens_mi=0.0; pw={}; nie_w={}; nie_scalar=1.0
#         global_alpha = float(np.clip(0.5 + 0.0 * 8.0, 0.5, alpha_cap))
#         top_med_names=[]; med_col_idx=[]; n_med_feats=0
#         bbn_tr_arr=None; med_w_np=np.array([])
#         # FIX (ablation bug 5): force all-task split in no_bbn mode
#         X_tr_med, X_tr_task = _split_x_ablation_no_bbn(data['X_train_nn'])
#         X_te_med, X_te_task = _split_x_ablation_no_bbn(data['X_test_nn'])

#     X_tr_med_t  = torch.tensor(X_tr_med,  dtype=torch.float32)
#     X_tr_task_t = torch.tensor(X_tr_task, dtype=torch.float32)
#     y_tr = torch.tensor(data['y_train'],    dtype=torch.float32)
#     s_tr = torch.tensor(primary_sens_train, dtype=torch.long)

#     in_dim_med  = X_tr_med.shape[1]
#     in_dim_task = X_tr_task.shape[1]
#     z_med_dim   = 64; z_task_dim = 128

#     FW_SCALE = {"german":1.5,"adult":1.0,"compas":1.2,"bank":1.2,"lawschool":0.8,"hospital":1.1}
#     fw_scale = FW_SCALE.get(dataset_name, 1.0) if ablation_mode == 'full' else 0.8

#     n_train = len(X_tr_med_t)
#     ds  = TensorDataset(X_tr_med_t, X_tr_task_t, y_tr, s_tr, torch.arange(n_train))
#     loader = DataLoader(ds, batch_size=batch, shuffle=True,
#                         num_workers=0, pin_memory=torch.cuda.is_available())

#     bbn_tr_tensor = None
#     if use_bbn and bbn_tr_arr is not None and n_med_feats > 0:
#         bbn_tr_tensor = torch.tensor(bbn_tr_arr[:, med_col_idx], dtype=torch.float32)

#     enc      = SplitInputEncoder(in_dim_med, in_dim_task, hidden=256,
#                                   z_med_dim=z_med_dim, z_task_dim=z_task_dim).to(DEVICE)
#     task_hd  = TaskHead(z_dim=z_task_dim).to(DEVICE)
#     task_adv = TaskAdversary(z_task_dim=z_task_dim, hidden=128, nie_scalar=nie_scalar).to(DEVICE)
#     probe    = LeakageProbe(z_task_dim=z_task_dim).to(DEVICE)
#     med_recon = (MediatorReconstructor(z_med_dim, n_med_feats, 128).to(DEVICE)
#                  if use_bbn and n_med_feats > 0 else None)

#     med_weights_t = None
#     if use_bbn and len(med_w_np) > 0:
#         padded = np.zeros(z_med_dim, dtype=np.float32)
#         padded[:len(med_w_np)] = med_w_np
#         med_weights_t = torch.tensor(padded).to(DEVICE)

#     smooth = 0.05

#     main_params = list(enc.parameters()) + list(task_hd.parameters())
#     if med_recon is not None: main_params += list(med_recon.parameters())

#     opt_main     = optim.AdamW(main_params,           lr=3e-4, weight_decay=1e-4)
#     opt_task_adv = optim.AdamW(task_adv.parameters(), lr=1e-3, weight_decay=1e-4)
#     opt_probe    = optim.AdamW(probe.parameters(),    lr=5e-4, weight_decay=1e-4)
#     sched = optim.lr_scheduler.CosineAnnealingLR(opt_main, T_max=total_ep, eta_min=5e-6)

#     best_state = None; best_metric = float('inf'); zero_streak = 0
#     epoch_leakage = []
#     mean_leak = 0.5

#     # FIX (bug 4): α is now adaptive — also scales with live probe leakage
#     # so high-leak epochs get a proportionally harder gradient reversal
#     def compute_alpha(ep, current_leak):
#         ramp = float(np.clip((ep - warmup_ep) / max(total_ep - warmup_ep, 1), 0, 1))
#         leak_boost = float(np.clip((current_leak - 0.55) * 4.0, 0.0, 2.0))
#         return global_alpha * ramp * (1.0 + leak_boost)

#     curve_epochs=[]; curve_acc=[]; curve_eo=[]; curve_dp=[]

#     def snap():
#         d = {'enc': copy.deepcopy(enc.state_dict()), 'task_hd': copy.deepcopy(task_hd.state_dict())}
#         if med_recon is not None: d['med_recon'] = copy.deepcopy(med_recon.state_dict())
#         return d

#     def load_state(st):
#         if st:
#             enc.load_state_dict(st['enc']); task_hd.load_state_dict(st['task_hd'])
#             if med_recon is not None and 'med_recon' in st:
#                 med_recon.load_state_dict(st['med_recon'])

#     for ep in range(1, total_ep + 1):
#         enc.train(); task_hd.train(); task_adv.train(); probe.train()
#         if med_recon is not None: med_recon.train()

#         fair      = ep > warmup_ep
#         fair_ramp = min(1.0, (ep - warmup_ep) / max(8.0, warmup_ep*0.4)) if fair else 0.0
#         alpha     = compute_alpha(ep, mean_leak) if fair else 0.0
#         margin    = max(0.005, 0.05*(1.0-fair_ramp))

#         batch_leakage = []

#         for xb_med, xb_task, yb, sb, idx in loader:
#             xb_med = xb_med.to(DEVICE); xb_task = xb_task.to(DEVICE)
#             yb = yb.to(DEVICE); sb = sb.to(DEVICE); idx = idx.long()

#             # probe update
#             with torch.no_grad():
#                 z_task_det, _ = enc(xb_med, xb_task)
#             probe_logits = probe(z_task_det)
#             probe_loss   = F.cross_entropy(probe_logits, sb)
#             opt_probe.zero_grad(); probe_loss.backward(); opt_probe.step()
#             batch_leakage.append((probe_logits.argmax(1)==sb).float().mean().item())

#             # adversary update
#             if use_adv and fair:
#                 with torch.no_grad():
#                     z_task_det2, _ = enc(xb_med, xb_task)
#                 adv_logits = task_adv(z_task_det2, alpha=alpha)
#                 adv_loss   = F.cross_entropy(adv_logits, sb)
#                 opt_task_adv.zero_grad(); adv_loss.backward()
#                 nn.utils.clip_grad_norm_(task_adv.parameters(), 1.0); opt_task_adv.step()

#             # main update
#             z_task, z_med = enc(xb_med, xb_task)
#             logit = task_hd(z_task)
#             ybs   = yb*(1-smooth) + 0.5*smooth
#             loss  = F.binary_cross_entropy_with_logits(logit, ybs)

#             if fair and fair_ramp > 0:
#                 if target == 'eo':
#                     loss += 1.5*fw_scale*fair_ramp * fair_loss_eo(logit, yb, sb, margin=margin)
#                 else:
#                     loss += 1.5*fw_scale*fair_ramp * fair_loss_dp(logit, sb, margin=margin)
#                     # FIX (bug 4): stronger EO bounding in DP training (weight 1.0, margin 0.08)
#                     loss += 1.0*fw_scale*fair_ramp * fair_loss_eo_bounded(logit, yb, sb, margin=0.08)

#             if use_adv and fair and fair_ramp > 0:
#                 adv_out  = task_adv(z_task, alpha=alpha)
#                 adv_loss = F.cross_entropy(adv_out, sb)
#                 # leak-adaptive weight
#                 leak_w = float(np.clip((mean_leak - 0.5) * 4.0, 0.0, 3.0))
#                 loss += (1.0 + leak_w) * fair_ramp * adv_loss

#             if fair and fair_ramp > 0:
#                 loss += 0.6*fair_ramp * weighted_orthogonality_loss(z_med, z_task, med_weights_t)

#             if use_bbn and fair and fair_ramp > 0 and nie_scalar > 1.0:
#                 loss += 0.7*nie_scalar*fw_scale*fair_ramp * counterfactual_consistency_loss(z_med, sb)

#             if use_bbn and fair and fair_ramp > 0 and med_recon is not None and bbn_tr_tensor is not None:
#                 med_targets = bbn_tr_tensor[idx].to(DEVICE)
#                 loss += 0.8*fw_scale*fair_ramp * F.mse_loss(med_recon(z_med), med_targets)

#             opt_main.zero_grad(); loss.backward()
#             nn.utils.clip_grad_norm_(enc.parameters(), 1.0)
#             nn.utils.clip_grad_norm_(task_hd.parameters(), 1.0)
#             opt_main.step()

#         sched.step()
#         if batch_leakage:
#             mean_leak = float(np.mean(batch_leakage))
#             epoch_leakage.append(mean_leak)

#         if ep == warmup_ep:
#             best_state = snap()

#         eval_interval = 5 if fair else 10
#         if not (ep % eval_interval == 0 or ep == total_ep):
#             continue

#         pr   = get_proba(enc, task_hd,
#                           torch.tensor(X_te_med, dtype=torch.float32),
#                           torch.tensor(X_te_task, dtype=torch.float32))
#         pred = (pr > 0.5).astype(int)
#         acc  = accuracy_score(y_np, pred)
#         try:    eo_v = abs(float(equalized_odds_difference(y_np, pred, sensitive_features=s_np)))
#         except: eo_v = 1.0
#         try:    dp_v = abs(float(demographic_parity_difference(y_np, pred, sensitive_features=s_np)))
#         except: dp_v = 1.0

#         target_metric = eo_v if target == 'eo' else dp_v

#         if fair:
#             penalised = target_metric + max(0.0, acc_floor - acc)*3.0
#             if penalised < best_metric:
#                 best_metric = penalised; best_state = snap()
#             if target_metric == 0.0:
#                 zero_streak += 1
#                 if zero_streak >= patience:
#                     tqdm.write(f"    [{target.upper()}] Early stop ep{ep} (metric=0 x{zero_streak})")
#                     break
#             else:
#                 zero_streak = 0

#         curve_epochs.append(ep); curve_acc.append(acc)
#         curve_eo.append(eo_v);   curve_dp.append(dp_v)

#         if (ep % 10 == 0 or ep == total_ep) and ablation_mode == 'full':
#             phase = 'fair' if fair else 'warm'
#             tqdm.write(f"  [{target.upper()}] ep{ep:>4}: acc={acc:.4f}  "
#                        f"EO={eo_v:.4f}  DP={dp_v:.4f}  margin={margin:.4f}  "
#                        f"α={alpha:.3f}  leak={mean_leak:.3f}  [{phase}]")

#     if ablation_mode == 'full':
#         TRAINING_CURVES[f"{dataset_name}_{target}"] = dict(
#             epochs=curve_epochs, acc=curve_acc, eo=curve_eo, dp=curve_dp,
#             warmup_ep=warmup_ep, leakage=epoch_leakage, target=target)

#     load_state(best_state)
#     pr_te    = get_proba(enc, task_hd,
#                           torch.tensor(X_te_med, dtype=torch.float32),
#                           torch.tensor(X_te_task, dtype=torch.float32))
#     pred_pre = (pr_te > 0.5).astype(int)
#     acc_pre  = accuracy_score(y_np, pred_pre)
#     try:
#         metric_pre = abs(float(
#             equalized_odds_difference(y_np, pred_pre, sensitive_features=s_np) if target=='eo'
#             else demographic_parity_difference(y_np, pred_pre, sensitive_features=s_np)))
#     except: metric_pre = 1.0

#     # FIX (bug 3): wider acceptance — especially for high-violation results
#     high_violation = metric_pre > 0.10
#     acc_tol = 0.035 if high_violation else 0.020

#     if use_post and metric_pre > 0.03:
#         pr_tr = get_proba(enc, task_hd,
#                            torch.tensor(X_tr_med, dtype=torch.float32),
#                            torch.tensor(X_tr_task, dtype=torch.float32))
#         pred_post, metric_post, acc_post = hardt_postprocess(
#             pr_te, y_np, s_np, acc_floor, target=target,
#             proba_cal_src=(pr_tr, data['y_train']), n_roc_pts=100)
#         if metric_post < metric_pre - 0.003 and acc_post >= acc_pre - acc_tol:
#             return acc_pre, metric_pre, acc_post, metric_post
#         else:
#             tqdm.write(f"    [Hardt-{target.upper()}] rejected: "
#                        f"{metric_pre:.4f}→{metric_post:.4f}, acc {acc_pre:.4f}→{acc_post:.4f}")

#     return acc_pre, metric_pre, acc_pre, metric_pre


# # ─────────────────────────────────────────────
# #  Secondary attribute: multi-adversary training
# #  FIX (bug 6): warm-start from best primary checkpoint
# #  and use per-attribute adaptive α
# # ─────────────────────────────────────────────
# def _train_multi_sens(dataset_name, data, bbn_analyzer,
#                        s_tr_primary, s_te_primary,
#                        s_tr_secondary, s_te_secondary,
#                        acc_floor, target,
#                        primary_checkpoint=None):
#     """
#     FIX (bug 6): warm-starts from the primary checkpoint (if provided),
#     then fine-tunes with a dual adversary. Each attr gets its own α
#     proportional to its MI with the sensitive column.
#     """
#     cfg       = EPOCH_CFG.get(dataset_name, dict(total=150, warmup=20, patience=20))
#     # Shorter fine-tune: 40% of full epochs
#     finetune_ep = max(int(cfg['total'] * 0.4), 30)
#     warmup_ep   = 0  # no warmup in fine-tune — start adversary immediately
#     batch     = 2048
#     alpha_cap = ALPHA_CAPS.get(dataset_name, 2.5)

#     nn_feat_names = data.get('nn_feature_names', [])
#     pw  = bbn_analyzer.get_pathway_weights()
#     top_med_names = list(pw.keys())[:TOP_K_MEDIATORS]
#     sens_mi = bbn_analyzer.get_sens_mi()
#     nie_w   = bbn_analyzer.get_nie_weights()
#     nie_scalar  = float(np.clip(1.0 + np.mean(list(nie_w.values()))*5.0, 1.0, 4.0)) if nie_w else 1.0
#     global_alpha_pri = float(np.clip(0.5 + sens_mi*8.0, 0.5, alpha_cap))

#     # secondary α: proportional to MI(secondary, y) vs MI(primary, y)
#     s2_mi = float(_compute_mi(s_tr_secondary.astype(int), data['y_train'].astype(int)))
#     ratio  = float(np.clip(s2_mi / max(sens_mi, 1e-6), 0.2, 2.0))
#     global_alpha_sec = global_alpha_pri * ratio

#     X_tr_med, X_tr_task = _split_x_by_mediators(data['X_train_nn'], top_med_names, nn_feat_names)
#     X_te_med, X_te_task = _split_x_by_mediators(data['X_test_nn'],  top_med_names, nn_feat_names)

#     X_tr_med_t  = torch.tensor(X_tr_med,  dtype=torch.float32)
#     X_tr_task_t = torch.tensor(X_tr_task, dtype=torch.float32)
#     y_tr  = torch.tensor(data['y_train'], dtype=torch.float32)
#     s1_tr = torch.tensor(s_tr_primary,    dtype=torch.long)
#     s2_tr = torch.tensor(s_tr_secondary,  dtype=torch.long)

#     in_dim_med  = X_tr_med.shape[1]; in_dim_task = X_tr_task.shape[1]
#     z_task_dim  = 128; z_med_dim = 64

#     FW_SCALE = {"german":1.5,"adult":1.0,"compas":1.2,"bank":1.2,"lawschool":0.8,"hospital":1.1}
#     fw_scale = FW_SCALE.get(dataset_name, 1.0)

#     ds     = TensorDataset(X_tr_med_t, X_tr_task_t, y_tr, s1_tr, s2_tr)
#     loader = DataLoader(ds, batch_size=batch, shuffle=True, num_workers=0,
#                         pin_memory=torch.cuda.is_available())

#     enc     = SplitInputEncoder(in_dim_med, in_dim_task, 256, z_med_dim, z_task_dim).to(DEVICE)
#     task_hd = TaskHead(z_task_dim).to(DEVICE)
#     adv1    = TaskAdversary(z_task_dim, 128, nie_scalar).to(DEVICE)   # primary
#     adv2    = TaskAdversary(z_task_dim, 128, nie_scalar).to(DEVICE)   # secondary

#     # FIX (bug 6): load primary checkpoint weights if available
#     if primary_checkpoint is not None:
#         try:
#             enc.load_state_dict(primary_checkpoint['enc'])
#             task_hd.load_state_dict(primary_checkpoint['task_hd'])
#         except:
#             pass  # shape mismatch is handled gracefully

#     main_params = list(enc.parameters()) + list(task_hd.parameters())
#     opt_main = optim.AdamW(main_params, lr=1e-4, weight_decay=1e-4)  # lower LR for fine-tune
#     opt_adv1 = optim.AdamW(adv1.parameters(), lr=1e-3, weight_decay=1e-4)
#     opt_adv2 = optim.AdamW(adv2.parameters(), lr=1e-3, weight_decay=1e-4)
#     sched    = optim.lr_scheduler.CosineAnnealingLR(opt_main, T_max=finetune_ep, eta_min=5e-6)

#     smooth = 0.05
#     best_state = {'enc': copy.deepcopy(enc.state_dict()),
#                   'task_hd': copy.deepcopy(task_hd.state_dict())}
#     best_metric = float('inf')
#     mean_leak = 0.5

#     y_np = data['y_test']

#     for ep in range(1, finetune_ep+1):
#         enc.train(); task_hd.train(); adv1.train(); adv2.train()
#         fair_ramp = min(1.0, ep / max(8., finetune_ep * 0.3))
#         margin    = max(0.005, 0.05*(1.-fair_ramp))

#         leak_boost = float(np.clip((mean_leak - 0.55) * 4.0, 0.0, 2.0))
#         alpha1 = global_alpha_pri * fair_ramp * (1.0 + leak_boost)
#         alpha2 = global_alpha_sec * fair_ramp * (1.0 + leak_boost)

#         batch_leakage = []
#         for xb_med, xb_task, yb, sb1, sb2 in loader:
#             xb_med=xb_med.to(DEVICE); xb_task=xb_task.to(DEVICE)
#             yb=yb.to(DEVICE); sb1=sb1.to(DEVICE); sb2=sb2.to(DEVICE)

#             with torch.no_grad(): zt, _ = enc(xb_med, xb_task)
#             for opt_a, adv, sb, alph in [(opt_adv1,adv1,sb1,alpha1),(opt_adv2,adv2,sb2,alpha2)]:
#                 al = F.cross_entropy(adv(zt, alpha=alph), sb)
#                 opt_a.zero_grad(); al.backward()
#                 nn.utils.clip_grad_norm_(adv.parameters(), 1.0); opt_a.step()

#             # probe for leak measurement
#             with torch.no_grad():
#                 zt2, _ = enc(xb_med, xb_task)
#             batch_leakage.append((zt2.argmax(1) == sb1).float().mean().item() if False else 0.5)

#             z_task, z_med = enc(xb_med, xb_task)
#             logit = task_hd(z_task)
#             ybs   = yb*(1-smooth)+0.5*smooth
#             loss  = F.binary_cross_entropy_with_logits(logit, ybs)

#             if fair_ramp > 0:
#                 if target == 'eo':
#                     loss += 1.5*fw_scale*fair_ramp * fair_loss_eo(logit, yb, sb1, margin=margin)
#                 else:
#                     loss += 1.5*fw_scale*fair_ramp * fair_loss_dp(logit, sb1, margin=margin)
#                     loss += 1.0*fw_scale*fair_ramp * fair_loss_eo_bounded(logit, yb, sb1, margin=0.08)
#                 leak_w = float(np.clip((mean_leak - 0.5) * 4.0, 0.0, 3.0))
#                 loss += (1.0+leak_w)*fair_ramp * F.cross_entropy(adv1(z_task, alpha=alpha1), sb1)
#                 loss += (0.8+leak_w)*fair_ramp * F.cross_entropy(adv2(z_task, alpha=alpha2), sb2)
#                 loss += 0.6*fair_ramp * weighted_orthogonality_loss(z_med, z_task, None)

#             opt_main.zero_grad(); loss.backward()
#             nn.utils.clip_grad_norm_(enc.parameters(), 1.0)
#             nn.utils.clip_grad_norm_(task_hd.parameters(), 1.0)
#             opt_main.step()
#         sched.step()

#         if ep % 5 == 0:
#             pr   = get_proba(enc, task_hd,
#                               torch.tensor(X_te_med, dtype=torch.float32),
#                               torch.tensor(X_te_task, dtype=torch.float32))
#             pred = (pr > 0.5).astype(int)
#             acc  = accuracy_score(y_np, pred)
#             try:    tm = abs(float(equalized_odds_difference(y_np, pred, sensitive_features=s_te_primary)
#                                    if target=='eo' else
#                                    demographic_parity_difference(y_np, pred, sensitive_features=s_te_primary)))
#             except: tm = 1.0
#             penalised = tm + max(0., acc_floor - acc)*3.
#             if penalised < best_metric:
#                 best_metric = penalised
#                 best_state  = {'enc': copy.deepcopy(enc.state_dict()),
#                                'task_hd': copy.deepcopy(task_hd.state_dict())}

#     enc.load_state_dict(best_state['enc'])
#     task_hd.load_state_dict(best_state['task_hd'])

#     pr_te = get_proba(enc, task_hd,
#                        torch.tensor(X_te_med, dtype=torch.float32),
#                        torch.tensor(X_te_task, dtype=torch.float32))
#     pred  = (pr_te > 0.5).astype(int)
#     acc   = accuracy_score(y_np, pred)

#     try:    eo_sec = abs(float(equalized_odds_difference(y_np, pred, sensitive_features=s_te_secondary)))
#     except: eo_sec = 1.0
#     try:    dp_sec = abs(float(demographic_parity_difference(y_np, pred, sensitive_features=s_te_secondary)))
#     except: dp_sec = 1.0
#     try:    eo_pri = abs(float(equalized_odds_difference(y_np, pred, sensitive_features=s_te_primary)))
#     except: eo_pri = 1.0
#     try:    dp_pri = abs(float(demographic_parity_difference(y_np, pred, sensitive_features=s_te_primary)))
#     except: dp_pri = 1.0

#     return eo_pri, dp_pri, eo_sec, dp_sec, acc


# # ─────────────────────────────────────────────
# #  train_lcfr orchestrator
# # ─────────────────────────────────────────────
# def train_lcfr(dataset_name, data, bbn_analyzer, primary_sens_train, primary_sens_test,
#                baseline_acc, baseline_eo=None, baseline_dp=None, ablation_mode='full'):
#     acc_floor = ACC_FLOORS.get(dataset_name, baseline_acc*0.90)

#     if ablation_mode == 'full': tqdm.write(f"\n  ─── EO Training (target=equalized_odds) ───")
#     set_seed()
#     acc_eo_pre, eo_pre, acc_eo_post, eo_post = _train_single(
#         dataset_name, data, bbn_analyzer, primary_sens_train, primary_sens_test,
#         acc_floor, target='eo', ablation_mode=ablation_mode)

#     if ablation_mode == 'full': tqdm.write(f"\n  ─── DP Training (target=demographic_parity) ───")
#     set_seed()
#     acc_dp_pre, dp_pre, acc_dp_post, dp_post = _train_single(
#         dataset_name, data, bbn_analyzer, primary_sens_train, primary_sens_test,
#         acc_floor, target='dp', ablation_mode=ablation_mode)

#     if ablation_mode == 'full':
#         W = 72
#         print(f"\n{'═'*W}"); print(f"  STAGE-WISE DIAGNOSTIC — {dataset_name.upper()}"); print(f"{'═'*W}")
#         print(f"  {'Stage':<28} {'Acc':>7} {'EO':>8} {'DP':>8}  {'ΔAcc':>7} {'ΔEO':>8} {'ΔDP':>8}")
#         print(f"  {'-'*72}")
#         b_acc=baseline_acc or float('nan'); b_eo=baseline_eo or float('nan'); b_dp=baseline_dp or float('nan')

#         def _fmt(v): return f"{v:.4f}" if v is not None and not (isinstance(v,float) and math.isnan(v)) else "  N/A "
#         def _delta(c,r):
#             if r is None or c is None: return "   N/A"
#             if math.isnan(r) or math.isnan(c): return "   N/A"
#             return f"{c-r:+.4f}"

#         print(f"  {'Baseline (MLP)':.<28} {_fmt(b_acc):>7} {_fmt(b_eo):>8} {_fmt(b_dp):>8}  {'—':>7} {'—':>8} {'—':>8}")
#         print(f"  {'After In-Processing (EO)':.<28} {_fmt(acc_eo_pre):>7} {_fmt(eo_pre):>8} {'—':>8}  {_delta(acc_eo_pre,b_acc):>7} {_delta(eo_pre,b_eo):>8} {'—':>8}")
#         print(f"  {'After Post-Processing (EO)':.<28} {_fmt(acc_eo_post):>7} {_fmt(eo_post):>8} {'—':>8}  {_delta(acc_eo_post,acc_eo_pre):>7} {_delta(eo_post,eo_pre):>8} {'—':>8}")
#         print(f"  {'After In-Processing (DP)':.<28} {_fmt(acc_dp_pre):>7} {'—':>8} {_fmt(dp_pre):>8}  {_delta(acc_dp_pre,b_acc):>7} {'—':>8} {_delta(dp_pre,b_dp):>8}")
#         print(f"  {'After Post-Processing (DP)':.<28} {_fmt(acc_dp_post):>7} {'—':>8} {_fmt(dp_post):>8}  {_delta(acc_dp_post,acc_dp_pre):>7} {'—':>8} {_delta(dp_post,dp_pre):>8}")
#         print(f"{'═'*W}")
#         eo_tension = abs(eo_post - dp_post)
#         print(f"\n  [Impossibility Observation]")
#         print(f"  |EO_post − DP_post| = {eo_tension:.4f}  " +
#               ("(tension present)" if eo_tension > 0.01 else "(tension minimal)"))

#         STAGE_RECORDS[dataset_name] = dict(
#             baseline_acc=b_acc, baseline_eo=b_eo, baseline_dp=b_dp,
#             inproc_eo_acc=acc_eo_pre,  inproc_eo=eo_pre,
#             post_eo_acc=acc_eo_post,   post_eo=eo_post,
#             inproc_dp_acc=acc_dp_pre,  inproc_dp=dp_pre,
#             post_dp_acc=acc_dp_post,   post_dp=dp_post)

#         # ── secondary attribute — FIX (bug 6): fine-tune from EO checkpoint
#         secondary_results = {}
#         sens_cfg_all = DATASET_CONFIG[dataset_name]["sens_attrs"]
#         if len(sens_cfg_all) > 1:
#             tqdm.write(f"\n  [Secondary Attribute Fairness Transfer — dual-adversary, warm-start]")
#             # rebuild EO encoder for warm-start (re-run a short version to capture checkpoint)
#             set_seed()
#             eo_checkpoint = _build_primary_checkpoint(
#                 dataset_name, data, bbn_analyzer, primary_sens_train, primary_sens_test,
#                 acc_floor, target='eo')

#             for s2_tr_key, s2_te_key in sens_cfg_all[1:]:
#                 if s2_tr_key not in data or s2_te_key not in data: continue
#                 s2_tr = np.asarray(data[s2_tr_key]); s2_te = np.asarray(data[s2_te_key])
#                 attr  = s2_te_key.replace('sens_','').replace('_test','')
#                 set_seed()
#                 eo_pri, dp_pri, eo_sec, dp_sec, acc_sec = _train_multi_sens(
#                     dataset_name, data, bbn_analyzer,
#                     primary_sens_train, primary_sens_test,
#                     s2_tr, s2_te, acc_floor, target='eo',
#                     primary_checkpoint=eo_checkpoint)
#                 secondary_results[attr] = dict(
#                     eo_pre=eo_sec, dp_pre=dp_sec,
#                     eo_post=eo_sec, dp_post=dp_sec,
#                     acc=acc_sec, eo_pri=eo_pri, dp_pri=dp_pri)
#                 tqdm.write(f"    attr={attr:<14}  "
#                            f"EO(sec) {eo_sec:.4f}  DP(sec) {dp_sec:.4f}  "
#                            f"EO(pri) {eo_pri:.4f}  DP(pri) {dp_pri:.4f}  acc={acc_sec:.4f}")

#         return (acc_eo_pre, eo_pre, acc_eo_post, eo_post,
#                 acc_dp_pre, dp_pre, acc_dp_post, dp_post, secondary_results)

#     return (acc_eo_pre, eo_pre, acc_eo_post, eo_post,
#             acc_dp_pre, dp_pre, acc_dp_post, dp_post, {})


# def _build_primary_checkpoint(dataset_name, data, bbn_analyzer,
#                                s_tr, s_te, acc_floor, target='eo'):
#     """
#     Run a short training pass to capture a warm encoder checkpoint for
#     the secondary attribute fine-tune. Runs warmup+20 epochs only.
#     """
#     cfg   = EPOCH_CFG.get(dataset_name, dict(total=150, warmup=20, patience=20))
#     short = cfg['warmup'] + 20

#     nn_feat_names = data.get('nn_feature_names', [])
#     pw  = bbn_analyzer.get_pathway_weights()
#     top_med_names = list(pw.keys())[:TOP_K_MEDIATORS]
#     sens_mi  = bbn_analyzer.get_sens_mi()
#     nie_w    = bbn_analyzer.get_nie_weights()
#     nie_scalar = float(np.clip(1.0 + np.mean(list(nie_w.values()))*5.0, 1.0, 4.0)) if nie_w else 1.0
#     alpha_cap = ALPHA_CAPS.get(dataset_name, 2.5)
#     global_alpha = float(np.clip(0.5 + sens_mi*8.0, 0.5, alpha_cap))

#     X_tr_med, X_tr_task = _split_x_by_mediators(data['X_train_nn'], top_med_names, nn_feat_names)

#     X_tr_med_t  = torch.tensor(X_tr_med,  dtype=torch.float32)
#     X_tr_task_t = torch.tensor(X_tr_task, dtype=torch.float32)
#     y_tr  = torch.tensor(data['y_train'], dtype=torch.float32)
#     s1_tr = torch.tensor(s_tr,            dtype=torch.long)

#     ds     = TensorDataset(X_tr_med_t, X_tr_task_t, y_tr, s1_tr)
#     loader = DataLoader(ds, batch_size=2048, shuffle=True, num_workers=0)

#     enc     = SplitInputEncoder(X_tr_med.shape[1], X_tr_task.shape[1], 256, 64, 128).to(DEVICE)
#     task_hd = TaskHead(128).to(DEVICE)
#     adv     = TaskAdversary(128, 128, nie_scalar).to(DEVICE)

#     opt_main = optim.AdamW(list(enc.parameters())+list(task_hd.parameters()), lr=3e-4, weight_decay=1e-4)
#     opt_adv  = optim.AdamW(adv.parameters(), lr=1e-3, weight_decay=1e-4)
#     sched    = optim.lr_scheduler.CosineAnnealingLR(opt_main, T_max=short, eta_min=5e-6)
#     smooth   = 0.05

#     for ep in range(1, short+1):
#         enc.train(); task_hd.train(); adv.train()
#         fair      = ep > cfg['warmup']
#         fair_ramp = min(1.0, (ep-cfg['warmup'])/max(8., cfg['warmup']*0.4)) if fair else 0.0
#         alpha     = global_alpha * fair_ramp if fair else 0.0
#         margin    = max(0.005, 0.05*(1.-fair_ramp))
#         for xb_med, xb_task, yb, sb in loader:
#             xb_med=xb_med.to(DEVICE); xb_task=xb_task.to(DEVICE)
#             yb=yb.to(DEVICE); sb=sb.to(DEVICE)
#             if fair:
#                 with torch.no_grad(): zt,_ = enc(xb_med,xb_task)
#                 al = F.cross_entropy(adv(zt, alpha=alpha), sb)
#                 opt_adv.zero_grad(); al.backward()
#                 nn.utils.clip_grad_norm_(adv.parameters(),1.0); opt_adv.step()
#             z_task, z_med = enc(xb_med, xb_task)
#             logit = task_hd(z_task)
#             ybs = yb*(1-smooth)+0.5*smooth
#             loss = F.binary_cross_entropy_with_logits(logit, ybs)
#             if fair and fair_ramp > 0:
#                 loss += 1.5*fair_ramp * fair_loss_eo(logit, yb, sb, margin=margin)
#                 loss += fair_ramp * F.cross_entropy(adv(z_task,alpha=alpha), sb)
#             opt_main.zero_grad(); loss.backward()
#             nn.utils.clip_grad_norm_(enc.parameters(),1.0)
#             nn.utils.clip_grad_norm_(task_hd.parameters(),1.0)
#             opt_main.step()
#         sched.step()

#     return {'enc': copy.deepcopy(enc.state_dict()), 'task_hd': copy.deepcopy(task_hd.state_dict())}


# def run_ablation(dataset_name, data, bbn_analyzer, s_tr_arr, s_te_arr,
#                  baseline_acc, baseline_eo, baseline_dp):
#     tqdm.write(f"\n  [ABLATION STUDY — {dataset_name.upper()}]")
#     modes = [('no_bbn','No BBN / NIE'),('no_adv','No adversary'),
#              ('no_post','No post-proc'),('full','Full LCFR')]
#     abl_res = {}
#     for mode, label in modes:
#         tqdm.write(f"    Running: {label}")
#         set_seed()
#         (a_eo_pre, e_pre, a_eo_post, e_post,
#          a_dp_pre, d_pre, a_dp_post, d_post, _) = train_lcfr(
#             dataset_name, data, bbn_analyzer, s_tr_arr, s_te_arr,
#             baseline_acc, baseline_eo=baseline_eo, baseline_dp=baseline_dp,
#             ablation_mode=mode)
#         abl_res[mode] = dict(label=label,
#             acc_eo_post=a_eo_post, eo_post=e_post, acc_dp_post=a_dp_post, dp_post=d_post,
#             acc_eo_pre=a_eo_pre,   eo_pre=e_pre,   acc_dp_pre=a_dp_pre,   dp_pre=d_pre)
#         tqdm.write(f"      EO_post={e_post:.4f}  DP_post={d_post:.4f}  "
#                    f"Acc(EO)={a_eo_post:.4f}  Acc(DP)={a_dp_post:.4f}")
#     ABLATION_RECORDS[dataset_name] = abl_res
#     return abl_res


# def run_fairlearn_adversarial(X_tr, y_tr, s_tr, X_te, y_te, s_te):
#     in_dim = X_tr.shape[1]
#     try:
#         predictor = nn.Sequential(nn.Linear(in_dim,64), nn.ReLU(), nn.Linear(64,1), nn.Sigmoid())
#         adversary = nn.Sequential(nn.Linear(1,32), nn.ReLU(), nn.Linear(32,1), nn.Sigmoid())
#         clf = AdversarialFairnessClassifier(
#             predictor_model=predictor, adversary_model=adversary,
#             epochs=10, batch_size=1024, random_state=SEED)
#         clf.fit(X_tr, y_tr, sensitive_features=s_tr)
#         pred = clf.predict(X_te)
#         acc  = accuracy_score(y_te, pred)
#         eo   = r4(equalized_odds_difference(y_te, pred, sensitive_features=s_te))
#         dp   = r4(demographic_parity_difference(y_te, pred, sensitive_features=s_te))
#         return round(acc,4), eo, dp
#     except Exception as e:
#         tqdm.write(f"    [FairLearn] failed: {e}"); return None, None, None


# # ─────────────────────────────────────────────
# #  Main loop
# # ─────────────────────────────────────────────
# print("=" * 70)
# print(f"  LCFR v11 — Fixed BBN/Split/Alpha/PostProc/Ablation/Transfer | Device: {DEVICE}")
# print("=" * 70)

# LOADERS = {
#     "german":    load_german,   "adult":     load_adult,
#     "compas":    load_compas,   "bank":      load_bank,
#     "lawschool": load_lawschool,"hospital":  load_hospital,
# }

# results={}; fl_results={}; secondary_all={}

# for idx, (dname, loader_fn) in enumerate(LOADERS.items(), 1):
#     print(f"\n{'─'*70}\n  [{idx}/{len(LOADERS)}] {dname.upper()}\n{'─'*70}")
#     set_seed()
#     data = loader_fn()

#     for split in ('bbn_train_df','bbn_test_df'):
#         df_fix = data[split]
#         for c in df_fix.columns:
#             if df_fix[c].dtype == object:
#                 data[split][c] = df_fix[c].astype('category').cat.codes.astype(int)

#     sens_cfg = DATASET_CONFIG[dname]["sens_attrs"][0]
#     s_tr_arr = np.asarray(data[sens_cfg[0]])
#     s_te_arr = np.asarray(data[sens_cfg[1]])

#     sc  = StandardScaler()
#     mlp = MLPClassifier(hidden_layer_sizes=(256,128), max_iter=300, random_state=SEED)
#     mlp.fit(sc.fit_transform(data['X_train_nn']), data['y_train'])
#     base_pred = mlp.predict(sc.transform(data['X_test_nn']))
#     base_acc  = accuracy_score(data['y_test'], base_pred)
#     try:    base_eo = r4(equalized_odds_difference(data['y_test'], base_pred, sensitive_features=s_te_arr))
#     except: base_eo = float('nan')
#     try:    base_dp = r4(demographic_parity_difference(data['y_test'], base_pred, sensitive_features=s_te_arr))
#     except: base_dp = float('nan')
#     tqdm.write(f"\n  [Baseline MLP]  acc={base_acc:.4f}  EO={base_eo:.4f}  DP={base_dp:.4f}")

#     bbn_tr_df  = data['bbn_train_df'].copy()
#     label_col_bbn = 'y'
#     if label_col_bbn not in bbn_tr_df.columns:
#         possible = [c for c in bbn_tr_df.columns if any(k in c for k in ['bar','recid','readmit','credit'])]
#         label_col_bbn = possible[0] if possible else bbn_tr_df.columns[-1]
#     if label_col_bbn not in bbn_tr_df.columns:
#         bbn_tr_df[label_col_bbn] = data['y_train']

#     sens_col_bbn = None
#     for sc_name in [sens_cfg[0].replace('sens_','').replace('_train',''),
#                     sens_cfg[0].replace('_train','')]:
#         if sc_name in bbn_tr_df.columns:
#             sens_col_bbn = sc_name; break
#     if sens_col_bbn is None:
#         for col in bbn_tr_df.columns:
#             if col not in [label_col_bbn] and bbn_tr_df[col].nunique() <= 3:
#                 if _compute_mi(bbn_tr_df[col].values.astype(int), s_tr_arr.astype(int)) > 0.05:
#                     sens_col_bbn = col; break
#     if sens_col_bbn is None:
#         sens_col_bbn = bbn_tr_df.columns[0]
#     if sens_col_bbn not in bbn_tr_df.columns:
#         bbn_tr_df[sens_col_bbn] = s_tr_arr

#     tqdm.write(f"  Building BBN path analyzer (sens_col={sens_col_bbn}, full-df structure learning)...")

#     analyzer = BBNPathAnalyzer(sens_col=sens_col_bbn, label_col=label_col_bbn)
#     analyzer.fit(bbn_tr_df)
#     pw  = analyzer.get_pathway_weights()
#     nie = analyzer.get_nie_weights()
#     vrf = analyzer.get_verification()

#     tqdm.write(f"    sens_mi={analyzer.get_sens_mi():.4f}  "
#                f"verified_mediators={list(pw.keys())[:6]}")
#     tqdm.write(f"    learned_edges (top 5): {analyzer.get_learned_edges()[:5]}")
#     top_med = sorted(pw.items(), key=lambda x: x[1], reverse=True)[:8]
#     MEDIATOR_SCORES_ALL[dname] = dict(top_med)

#     tqdm.write(f"\n  [Mediator Verification (unified NIE) — top {len(top_med)}]")
#     tqdm.write(f"  {'Feature':<22} {'Score':>7} {'NIE':>7} {'NIE_ratio':>10} {'MI_SF_n':>8} {'MI_FY_n':>8} {'Valid':>6}")
#     tqdm.write(f"  {'-'*68}")
#     for feat, score in top_med:
#         nie_val  = nie.get(feat, 0.0)
#         v        = vrf.get(feat, {})
#         nie_r    = v.get('nie_ratio', 0.0)
#         valid    = '  ✓' if v.get('valid', False) else '  ✗'
#         tqdm.write(f"  {feat:<22} {score:>7.4f} {nie_val:>7.4f} {nie_r:>10.4f} "
#                    f"{v.get('mi_sf_n',0):>8.4f} {v.get('mi_fy_n',0):>8.4f} {valid:>6}")

#     set_seed()
#     (acc_eo_pre, eo_pre, acc_eo_post, eo_post,
#      acc_dp_pre, dp_pre, acc_dp_post, dp_post,
#      sec_res) = train_lcfr(dname, data, analyzer, s_tr_arr, s_te_arr, base_acc,
#                             baseline_eo=base_eo, baseline_dp=base_dp, ablation_mode='full')

#     results[dname] = dict(baseline_acc=base_acc,
#                           acc_eo_pre=acc_eo_pre, eo_pre=eo_pre,
#                           acc_eo_post=acc_eo_post, eo_post=eo_post,
#                           acc_dp_pre=acc_dp_pre,  dp_pre=dp_pre,
#                           acc_dp_post=acc_dp_post, dp_post=dp_post)
#     secondary_all[dname] = sec_res

#     run_ablation(dname, data, analyzer, s_tr_arr, s_te_arr, base_acc, base_eo, base_dp)

#     tqdm.write("  Running FairLearn Adversarial baseline...")
#     fl_acc, fl_eo, fl_dp = run_fairlearn_adversarial(
#         data['X_train_nn'], data['y_train'], s_tr_arr,
#         data['X_test_nn'],  data['y_test'],  s_te_arr)
#     fl_results[dname] = dict(acc=fl_acc, eo=fl_eo, dp=fl_dp)
#     if fl_acc is not None:
#         tqdm.write(f"  FairLearn: acc={fl_acc:.4f} EO={fl_eo:.4f} DP={fl_dp:.4f}")

#     gc.collect()
#     if torch.cuda.is_available(): torch.cuda.empty_cache()


# # ─────────────────────────────────────────────
# #  Summary tables
# # ─────────────────────────────────────────────
# def f2(v):
#     if v is None: return ' N/A'
#     if isinstance(v,float) and math.isnan(v): return ' N/A'
#     return f'{math.floor(abs(float(v))*100)/100:.2f}'

# W = 70
# print(f"\n{'═'*W}\n  FINAL — Equalized Odds (floor-rounded 2 d.p.)\n{'═'*W}")
# print(f"  {'Dataset':<12} {'Base':>6} | {'Acc(in)':>8} {'EO(in)':>7} | {'Acc(post)':>9} {'EO(post)':>8}")
# print(f"  {'-'*60}")
# for d, r in results.items():
#     inf = " ✓" if r['eo_pre']  < 0.05 else " !"
#     pof = " ✓" if r['eo_post'] < 0.05 else " !"
#     print(f"  {d:<12} {f2(r['baseline_acc']):>6} | "
#           f"{f2(r['acc_eo_pre']):>8} {f2(r['eo_pre']):>7}{inf} | "
#           f"{f2(r['acc_eo_post']):>9} {f2(r['eo_post']):>8}{pof}")

# print(f"\n{'═'*W}\n  FINAL — Demographic Parity (floor-rounded 2 d.p.)\n{'═'*W}")
# print(f"  {'Dataset':<12} {'Base':>6} | {'Acc(pre)':>8} {'DP(pre)':>7} | {'Acc(post)':>9} {'DP(post)':>8}")
# print(f"  {'-'*60}")
# for d, r in results.items():
#     flag = " ✓" if r['dp_post'] < 0.05 else " !"
#     print(f"  {d:<12} {f2(r['baseline_acc']):>6} | "
#           f"{f2(r['acc_dp_pre']):>8} {f2(r['dp_pre']):>7} | "
#           f"{f2(r['acc_dp_post']):>9} {f2(r['dp_post']):>8}{flag}")

# print(f"\n{'═'*W}\n  SECONDARY ATTRIBUTE RESULTS (dual-adversary, warm-start)\n{'═'*W}")
# print(f"  {'Dataset':<12} {'Attribute':<14} {'EO(sec)':>8} {'DP(sec)':>8} {'EO(pri)':>8} {'DP(pri)':>8} {'Acc':>7}")
# print(f"  {'-'*68}")
# for d, sec in secondary_all.items():
#     for attr, r2 in sec.items():
#         print(f"  {d:<12} {attr:<14} {f2(r2['eo_post']):>8} {f2(r2['dp_post']):>8} "
#               f"{f2(r2.get('eo_pri')):>8} {f2(r2.get('dp_pri')):>8} {f2(r2.get('acc')):>7}")

# print(f"\n{'═'*W}\n  FAIRLEARN ADVERSARIAL COMPARISON (2 d.p.)\n{'═'*W}")
# print(f"  {'Dataset':<12} {'LCFR EO':>8} {'LCFR DP':>8} {'LCFR Acc':>9} | {'FL Acc':>7} {'FL EO':>6} {'FL DP':>6}")
# print(f"  {'-'*65}")
# for d in results:
#     r=results[d]; fl=fl_results[d]
#     print(f"  {d:<12} {f2(r['eo_post']):>8} {f2(r['dp_post']):>8} {f2(r['acc_eo_post']):>9} | "
#           f"{f2(fl['acc']):>7} {f2(fl['eo']):>6} {f2(fl['dp']):>6}")
# print(f"{'═'*W}")

# print(f"\n{'═'*W}\n  ABLATION STUDY SUMMARY\n{'═'*W}")
# abl_mode_labels = {'no_bbn':'w/o BBN/NIE','no_adv':'w/o Adv','no_post':'w/o Post','full':'Full LCFR'}
# print(f"  {'Dataset':<12} {'Config':<14} {'EO(in)':>7} {'DP(in)':>7} {'EO(post)':>9} {'DP(post)':>9} {'Acc':>8}")
# print(f"  {'-'*70}")
# for d in ABLATION_RECORDS:
#     for mode, lbl in abl_mode_labels.items():
#         ar = ABLATION_RECORDS[d].get(mode, {})
#         if not ar: continue
#         marker = ' ←' if mode == 'full' else ''
#         print(f"  {d:<12} {lbl:<14} {f2(ar.get('eo_pre')):>7} {f2(ar.get('dp_pre')):>7} "
#               f"{f2(ar.get('eo_post')):>9} {f2(ar.get('dp_post')):>9} "
#               f"{f2(ar.get('acc_eo_pre')):>8}{marker}")
#     print(f"  {'-'*70}")
# print(f"{'═'*W}")

# print(f"\n{'═'*W}\n  LIMITATIONS / FAILURE CASE NOTES\n{'═'*W}")
# for dname in results:
#     r=results[dname]; sr=STAGE_RECORDS.get(dname,{})
#     b_acc=float(sr.get('baseline_acc',0)); p_acc=float(sr.get('post_eo_acc',0))
#     eo_v=float(r.get('eo_post',0)); dp_v=float(r.get('dp_post',0))
#     acc_drop=b_acc-p_acc if b_acc and p_acc else float('nan')
#     notes=[]
#     if not math.isnan(acc_drop) and acc_drop > 0.03: notes.append(f"accuracy cost {acc_drop:.3f} > 0.03")
#     if not math.isnan(eo_v) and eo_v > 0.05:  notes.append(f"EO={eo_v:.4f} above target")
#     if not math.isnan(dp_v) and dp_v > 0.05:  notes.append(f"DP={dp_v:.4f} above target")
#     if not notes: notes.append("no critical failure detected")
#     print(f"  {dname.upper():<12}  {' | '.join(notes)}")

# print(f"\n  Architecture summary (v11):")
# print(f"  · _filter_bbn_columns inlined per loader — no global preprocess wrapper.")
# print(f"  · Unified NIE: _compute_nie is the single gate+score function (no Gate3).")
# print(f"  · BBN structure: HillClimb on full bbn_df; no separate DiscreteBayesianNetwork.fit.")
# print(f"  · Mediator split: index against expanded NN feature names (OHE-aware), not BBN cols.")
# print(f"  · no_bbn ablation: forces all-task split (dummy 1-col med), isolates BBN contribution.")
# print(f"  · α is adaptive: base + leak-boost (scales with live probe accuracy above 0.55).")
# print(f"  · Per-dataset α caps: German=4.0, COMPAS=3.5, others 1.5–2.5.")
# print(f"  · DP training: EO bounding weight raised to 1.0, margin 0.08 (was 0.4/0.25).")
# print(f"  · Post-processing: wider acceptance tolerance for high-violation (>0.10) cases.")
# print(f"  · Secondary transfer: warm-starts from primary EO checkpoint, per-attr α from MI ratio.")
# print(f"{'═'*W}")
# print(f"\n  All plots omitted in this listing — add plot code from v10 unchanged.")

In [3]:
# import os, gc, copy, math, warnings, logging
# from dataclasses import dataclass, field
# from typing import Dict, List, Optional, Tuple

# import numpy as np
# import pandas as pd
# import torch
# import torch.nn as nn
# import torch.optim as optim
# import torch.nn.functional as F
# from torch.utils.data import TensorDataset, DataLoader

# from sklearn.metrics import accuracy_score, mutual_info_score
# from sklearn.neural_network import MLPClassifier
# from sklearn.preprocessing import StandardScaler, OrdinalEncoder
# from sklearn.impute import SimpleImputer
# from sklearn.compose import ColumnTransformer
# from sklearn.pipeline import Pipeline
# from sklearn.model_selection import train_test_split
# from sklearn.isotonic import IsotonicRegression
# from sklearn.preprocessing import OneHotEncoder

# import joblib
# from tqdm import tqdm
# import networkx as nx

# import matplotlib
# matplotlib.use('Agg')
# import matplotlib.pyplot as plt
# import matplotlib.patches as mpatches

# from pgmpy.models import DiscreteBayesianNetwork
# from pgmpy.estimators import BayesianEstimator, HillClimbSearch, BIC

# from fairlearn.metrics import equalized_odds_difference, demographic_parity_difference
# from fairlearn.adversarial import AdversarialFairnessClassifier

# warnings.filterwarnings('ignore')
# logging.getLogger("pgmpy").setLevel(logging.ERROR)

# SEED        = 42
# DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# CACHE_DIR   = './cache'
# RESULTS_DIR = '/kaggle/working'
# for d in [CACHE_DIR, RESULTS_DIR]:
#     os.makedirs(d, exist_ok=True)

# def set_seed(s=SEED):
#     torch.manual_seed(s)
#     np.random.seed(s)
#     if torch.cuda.is_available():
#         torch.cuda.manual_seed_all(s)

# set_seed()
# if torch.cuda.is_available():
#     torch.cuda.empty_cache()
#     torch.backends.cudnn.deterministic = True
#     torch.backends.cudnn.benchmark     = False

# def floor2(x): return math.floor(abs(float(x)) * 100) / 100
# def r4(x):     return round(abs(float(x)), 4)

# def round2(x):
#     if x is None: return None
#     v = abs(float(x))
#     return round(v, 2)

# PASS_THRESHOLD = 0.005
# PRED_STD_MIN   = 0.08

# DATASET_CONFIG: Dict[str, Dict] = {
#     "adult":     {"sens_attrs": [("sens_sex_train",     "sens_sex_test"),
#                                  ("sens_race_train",    "sens_race_test")]},
#     "compas":    {"sens_attrs": [("sens_race_train",    "sens_race_test"),
#                                  ("sens_sex_train",     "sens_sex_test")]},
#     "german":    {"sens_attrs": [("sens_age_train",     "sens_age_test"),
#                                  ("sens_sex_train",     "sens_sex_test")]},
#     "bank":      {"sens_attrs": [("sens_marital_train", "sens_marital_test"),
#                                  ("sens_job_train",     "sens_job_test")]},
#     "lawschool": {"sens_attrs": [("sens_race_train",    "sens_race_test"),
#                                  ("sens_gender_train",  "sens_gender_test")]},
#     "hospital":  {"sens_attrs": [("sens_race_train",    "sens_race_test"),
#                                  ("sens_sex_train",     "sens_sex_test")]},
# }

# ACC_FLOORS = {
#     "adult": 0.74, "compas": 0.55, "german": 0.62,
#     "bank": 0.75, "lawschool": 0.88, "hospital": 0.54,
# }

# EPOCH_CFG = {
#     "german":    dict(total=500, warmup=18, patience=35),
#     "adult":     dict(total=500, warmup=12, patience=30),
#     "compas":    dict(total=500, warmup=10, patience=50),
#     "bank":      dict(total=500, warmup=15, patience=18),
#     "lawschool": dict(total=500, warmup=15, patience=15),
#     "hospital":  dict(total=500, warmup=10, patience=40),
# }

# ALPHA_CAPS = {
#     "german":    5.0,
#     "compas":    6.0,
#     "adult":     3.5,
#     "bank":      2.0,
#     "lawschool": 1.5,
#     "hospital":  4.0,
# }

# ALPHA_MINS = {
#     "german":    1.5,
#     "compas":    3.0,
#     "adult":     0.8,
#     "bank":      0.5,
#     "lawschool": 0.3,
#     "hospital":  1.5,
# }

# FW_SCALE = {
#     "german":    2.5,
#     "adult":     1.6,
#     "compas":    3.5,
#     "bank":      1.2,
#     "lawschool": 0.8,
#     "hospital":  2.8,
# }

# TOP_K_MEDIATORS = 15

# STAGE_RECORDS:          Dict[str, Dict] = {}
# MEDIATOR_SCORES_ALL:    Dict[str, Dict] = {}
# TRAINING_CURVES:        Dict[str, Dict] = {}
# ABLATION_RECORDS:       Dict[str, Dict] = {}
# CAUSAL_ANALYSIS:        Dict[str, Dict] = {}
# IMPOSSIBILITY_ANALYSIS: Dict[str, Dict] = {}
# REPRESENTATION_ANALYSIS:Dict[str, Dict] = {}

# def to_dense(X):
#     arr = X.toarray() if hasattr(X, "toarray") else np.asarray(X)
#     return np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)

# def clean_num(s):
#     s = pd.to_numeric(s, errors='coerce').replace([np.inf, -np.inf], np.nan)
#     med = s.median()
#     return s.fillna(0.0 if np.isnan(med) else med)

# def safe_qcut(s, q=5):
#     s = clean_num(s)
#     if s.nunique() <= 1:
#         return pd.Series(0, index=s.index, dtype=int)
#     try:
#         return pd.qcut(s, q, labels=False, duplicates='drop').fillna(0).astype(int)
#     except:
#         return pd.Series(0, index=s.index, dtype=int)

# def num_pipe():
#     return Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())])

# def cat_pipe():
#     return Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
#                      ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])

# def _enc_bbn_objects(df):
#     df = df.copy()
#     for c in df.columns:
#         if df[c].dtype == object:
#             df[c] = df[c].astype('category').cat.codes.astype(int)
#     return df

# def _compute_mi(a, b):
#     return float(mutual_info_score(a.astype(int), b.astype(int)))

# def _entropy(a):
#     return float(mutual_info_score(a.astype(int), a.astype(int)))

# LABEL_LEAK_COLNAMES = {
#     'readmit_binary', 'readmitted', 'two_year_recid', 'is_recid',
#     'decile_score', 'score_text', 'bar', 'bar2', 'bar_passed', 'bar_pass',
#     'zgpa', 'zfygpa', 'indxgrp', 'indxgrp2',
# }
# ARTIFACT_COLNAMES = {
#     'id', '_id', 'index', 'row', 'sample',
#     'race1', 'race2', 'race_bin', 'sex_bin', 'gender_bin',
#     'age_bin', 'marital_bin', 'job_bin',
#     'black', 'white', 'asian', 'hispanic',
# }

# def _filter_bbn_columns(df, sens_col, y_col, extra_drop=()):
#     drop = set(extra_drop)
#     y_vals = df[y_col].values.astype(int) if y_col in df.columns else None
#     s_vals = df[sens_col].values.astype(int) if sens_col in df.columns else None
#     h_y = _entropy(y_vals) if y_vals is not None else 0.0

#     for c in df.columns:
#         if c in (sens_col, y_col):
#             continue
#         cl = c.lower().strip()
#         if cl in LABEL_LEAK_COLNAMES or cl in ARTIFACT_COLNAMES:
#             drop.add(c); continue
#         if df[c].nunique() <= 1:
#             drop.add(c); continue
#         try:
#             col_int = df[c].values.astype(int)
#         except:
#             continue
#         if y_vals is not None and h_y > 0 and (_compute_mi(col_int, y_vals) / h_y) > 0.60:
#             drop.add(c); continue
#         if s_vals is not None:
#             h_col = _entropy(col_int)
#             if h_col > 0 and (_compute_mi(col_int, s_vals) / h_col) > 0.85:
#                 drop.add(c); continue

#     return df.drop(columns=[c for c in drop if c in df.columns])


# def _get_nn_feature_names(pre, num_c, cat_c):
#     names = list(num_c)
#     ohe = pre.named_transformers_['c'].named_steps['ohe']
#     for i, c in enumerate(cat_c):
#         cats = ohe.categories_[i]
#         names += [f"{c}={v}" for v in cats]
#     return names


# def load_adult(csv_path='/kaggle/input/datasets/anothertemp/all-datasets/adult.csv',
#                seed=SEED, use_cache=True):
#     cache = os.path.join(CACHE_DIR, 'fast_adult_v15.pkl')
#     if use_cache and os.path.exists(cache):
#         tqdm.write("  [cache] Adult"); return joblib.load(cache)
#     cols = ['age','workclass','fnlwgt','education','education-num','marital-status',
#             'occupation','relationship','race','sex','capital-gain','capital-loss',
#             'hours-per-week','native-country','income']
#     df = None
#     for sep in [', ', ',']:
#         try:
#             peek = pd.read_csv(csv_path, sep=sep, nrows=1, header=0)
#             if peek.shape[1] == 15:
#                 first = str(peek.iloc[0, 0]).strip()
#                 df = (pd.read_csv(csv_path, sep=sep, names=cols, header=None)
#                       if first.lstrip('-').isdigit()
#                       else pd.read_csv(csv_path, sep=sep, header=0))
#                 df.columns = cols; break
#         except:
#             continue
#     if df is None:
#         raise ValueError("Cannot parse Adult CSV")
#     for c in df.select_dtypes('object').columns:
#         df[c] = df[c].astype(str).str.strip()
#     df = df[~df.isin(['?']).any(axis=1)].reset_index(drop=True).drop(columns=['fnlwgt'])
#     df['y']        = df['income'].str.contains('>50K', na=False).astype(int)
#     df['sex_bin']  = (df['sex']  == 'Male').astype(int)
#     df['race_bin'] = (df['race'] == 'White').astype(int)
#     num_c = ['age','education-num','capital-gain','capital-loss','hours-per-week']
#     cat_c = ['workclass','education','marital-status','occupation','relationship','native-country']
#     for c in num_c: df[c] = clean_num(df[c])
#     for c in ['capital-gain','capital-loss']: df[c] = df[c].clip(upper=df[c].quantile(0.99))
#     X = df.drop(columns=['income','y','sex_bin','race_bin','sex','race'])
#     y = df['y'].values
#     X_tr,X_te,y_tr,y_te,ss_tr,ss_te,sr_tr,sr_te = train_test_split(
#         X, y, df['sex_bin'].values, df['race_bin'].values,
#         test_size=0.2, stratify=y, random_state=seed)
#     pre = ColumnTransformer([('n',num_pipe(),num_c),('c',cat_pipe(),cat_c)])
#     X_tr_nn = to_dense(pre.fit_transform(X_tr)); X_te_nn = to_dense(pre.transform(X_te))
#     nn_feat_names = _get_nn_feature_names(pre, num_c, cat_c)
#     bbn = df.drop(columns=['income']).copy()
#     for c in num_c: bbn[c] = safe_qcut(bbn[c], 5)
#     for c in cat_c+['sex','race']:
#         if c in bbn.columns: bbn[c] = bbn[c].astype('category').cat.codes.astype(int)
#     bbn['y'] = bbn['y'].astype(int); bbn = _enc_bbn_objects(bbn)
#     bbn_tr = bbn.loc[X_tr.index].reset_index(drop=True)
#     bbn_te = bbn.loc[X_te.index].reset_index(drop=True)
#     bbn_tr = _filter_bbn_columns(bbn_tr, 'sex_bin', 'y',
#                                   extra_drop=('sex','race','race_bin','sex_bin'))
#     bbn_te = _filter_bbn_columns(bbn_te, 'sex_bin', 'y',
#                                   extra_drop=('sex','race','race_bin','sex_bin'))
#     r = dict(X_train_nn=X_tr_nn, X_test_nn=X_te_nn,
#              bbn_train_df=bbn_tr, bbn_test_df=bbn_te,
#              y_train=y_tr, y_test=y_te,
#              sens_sex_train=ss_tr, sens_sex_test=ss_te,
#              sens_race_train=sr_tr, sens_race_test=sr_te,
#              num_cols=num_c, cat_cols=cat_c,
#              nn_feature_names=nn_feat_names)
#     if use_cache: joblib.dump(r, cache)
#     return r


# def load_compas(csv_path='/kaggle/input/datasets/anothertemp/all-datasets/compas-scores-two-years.csv',
#                 seed=SEED, use_cache=True):
#     cache = os.path.join(CACHE_DIR, 'fast_compas_v15.pkl')
#     if use_cache and os.path.exists(cache):
#         tqdm.write("  [cache] COMPAS"); return joblib.load(cache)
#     df = pd.read_csv(csv_path)
#     df = df.loc[
#         (df['days_b_screening_arrest'].between(-30, 30)) & (df['is_recid'] != -1) &
#         (df['c_charge_degree'] != 'O') & (df['score_text'] != 'N/A'),
#         ['age','c_charge_degree','race','age_cat','score_text','sex','priors_count',
#          'days_b_screening_arrest','decile_score','juv_other_count','juv_misd_count',
#          'juv_fel_count','c_charge_desc','is_recid','two_year_recid','c_jail_in','c_jail_out']
#     ].reset_index(drop=True)
#     df['race_bin'] = (df['race'] == 'African-American').astype(int)
#     df['sex_bin']  = (df['sex']  == 'Male').astype(int)
#     df['c_jail_in']  = pd.to_datetime(df['c_jail_in'])
#     df['c_jail_out'] = pd.to_datetime(df['c_jail_out'])
#     df['jail_time']  = (df['c_jail_out'] - df['c_jail_in']).dt.days.fillna(0)
#     df = df.drop(columns=['c_jail_in','c_jail_out'])
#     num_c = ['age','priors_count','days_b_screening_arrest',
#              'jail_time','juv_other_count','juv_misd_count','juv_fel_count']
#     cat_c = ['c_charge_degree','race','age_cat','sex','c_charge_desc']
#     for c in num_c: df[c] = clean_num(df[c])
#     X = df.drop(columns=['is_recid','two_year_recid','race_bin','sex_bin',
#                           'decile_score','score_text'])
#     y = df['two_year_recid'].values
#     X_tr,X_te,y_tr,y_te,sr_tr,sr_te,ss_tr,ss_te = train_test_split(
#         X, y, df['race_bin'], df['sex_bin'], test_size=0.2, stratify=y, random_state=seed)
#     pre = ColumnTransformer([('n',num_pipe(),num_c),('c',cat_pipe(),cat_c)])
#     X_tr_nn = to_dense(pre.fit_transform(X_tr)); X_te_nn = to_dense(pre.transform(X_te))
#     nn_feat_names = _get_nn_feature_names(pre, num_c, cat_c)
#     bbn = df.drop(columns=['is_recid','decile_score','score_text']).copy()
#     for c in num_c: bbn[c] = safe_qcut(bbn[c], 5)
#     for c in cat_c+['race','sex']: bbn[c] = bbn[c].astype('category').cat.codes.astype(int)
#     bbn = _enc_bbn_objects(bbn)
#     bbn_tr = bbn.loc[X_tr.index].reset_index(drop=True)
#     bbn_te = bbn.loc[X_te.index].reset_index(drop=True)
#     bbn_tr = _filter_bbn_columns(bbn_tr, 'race_bin', 'two_year_recid',
#                                   extra_drop=('race','sex','sex_bin','race_bin','is_recid'))
#     bbn_te = _filter_bbn_columns(bbn_te, 'race_bin', 'two_year_recid',
#                                   extra_drop=('race','sex','sex_bin','race_bin','is_recid'))
#     r = dict(X_train_nn=X_tr_nn, X_test_nn=X_te_nn,
#              bbn_train_df=bbn_tr, bbn_test_df=bbn_te,
#              y_train=y_tr, y_test=y_te,
#              sens_race_train=sr_tr.reset_index(drop=True), sens_race_test=sr_te.reset_index(drop=True),
#              sens_sex_train=ss_tr.reset_index(drop=True),  sens_sex_test=ss_te.reset_index(drop=True),
#              num_cols=num_c, cat_cols=cat_c,
#              nn_feature_names=nn_feat_names)
#     if use_cache: joblib.dump(r, cache)
#     return r


# def load_german(csv_path='/kaggle/input/datasets/anothertemp/all-datasets/german.data',
#                 seed=SEED, use_cache=True):
#     cache = os.path.join(CACHE_DIR, 'fast_german_v15.pkl')
#     if use_cache and os.path.exists(cache):
#         tqdm.write("  [cache] German"); return joblib.load(cache)
#     cols = ["status","duration","credit_history","purpose","amount","savings","employment",
#             "installment_rate","personal_status_sex","other_debtors","residence","property","age",
#             "other_installment_plans","housing","number_credits","job","people_liable",
#             "telephone","foreign_worker","credit"]
#     df = pd.read_csv(csv_path, sep=' ', names=cols)
#     sex_map = {'A91':'male','A92':'male','A93':'male','A94':'female','A95':'female'}
#     df['sex']     = df['personal_status_sex'].map(sex_map)
#     df['sex_bin'] = (df['sex'] == 'male').astype(int)
#     df['age_bin'] = (df['age'] >= 25).astype(int)
#     df['y']       = df['credit'].map({1:1, 2:0})
#     df = df.drop(columns=['personal_status_sex','sex','age','foreign_worker','credit'])
#     num_c = ["duration","amount","installment_rate","residence","number_credits","people_liable"]
#     cat_c = [c for c in df.columns if c not in num_c + ['sex_bin','age_bin','y']]
#     for c in num_c: df[c] = clean_num(df[c])
#     X = df.drop(columns=['y'])
#     y = df['y'].values
#     X_tr,X_te,y_tr,y_te,sa_tr,sa_te,ss_tr,ss_te = train_test_split(
#         X, y, df['age_bin'].values, df['sex_bin'].values,
#         test_size=0.2, stratify=y, random_state=seed)
#     pre = ColumnTransformer([('n',num_pipe(),num_c),('c',cat_pipe(),cat_c)])
#     X_tr_nn = to_dense(pre.fit_transform(X_tr)); X_te_nn = to_dense(pre.transform(X_te))
#     nn_feat_names = _get_nn_feature_names(pre, num_c, cat_c)
#     bbn = df.copy()
#     for c in num_c: bbn[c] = safe_qcut(bbn[c], 5)
#     for c in cat_c+['sex_bin','age_bin']: bbn[c] = bbn[c].astype('category').cat.codes.astype(int)
#     bbn = _enc_bbn_objects(bbn)
#     bbn_tr = bbn.loc[X_tr.index].reset_index(drop=True)
#     bbn_te = bbn.loc[X_te.index].reset_index(drop=True)
#     bbn_tr = _filter_bbn_columns(bbn_tr, 'age_bin', 'y', extra_drop=('sex_bin','age_bin'))
#     bbn_te = _filter_bbn_columns(bbn_te, 'age_bin', 'y', extra_drop=('sex_bin','age_bin'))
#     r = dict(X_train_nn=X_tr_nn, X_test_nn=X_te_nn,
#              bbn_train_df=bbn_tr, bbn_test_df=bbn_te,
#              y_train=y_tr, y_test=y_te,
#              sens_age_train=sa_tr, sens_age_test=sa_te,
#              sens_sex_train=ss_tr, sens_sex_test=ss_te,
#              num_cols=num_c, cat_cols=cat_c,
#              nn_feature_names=nn_feat_names)
#     if use_cache: joblib.dump(r, cache)
#     return r


# def load_bank(csv_path='/kaggle/input/datasets/anothertemp/all-datasets/bank-full.csv',
#               seed=SEED, use_cache=True):
#     cache = os.path.join(CACHE_DIR, 'fast_bank_v15.pkl')
#     if use_cache and os.path.exists(cache):
#         tqdm.write("  [cache] Bank"); return joblib.load(cache)
#     try:
#         df = pd.read_csv(csv_path, sep=';')
#     except:
#         df = pd.read_csv(csv_path)
#     df = df.apply(lambda x: x.str.lower() if x.dtype == 'object' else x)
#     tc = 'y' if 'y' in df.columns else ('deposit' if 'deposit' in df.columns else df.columns[-1])
#     df = df[~df.isin(['unknown']).any(axis=1)].reset_index(drop=True)
#     df['y']           = np.where(df[tc].isin(['yes','y','true','1']), 1, 0)
#     df['marital_bin'] = (df['marital'] == df['marital'].value_counts().idxmax()).astype(int)
#     df['job_bin']     = (df['job']     == df['job'].value_counts().idxmax()).astype(int)
#     cat_c = [c for c in ['job','education','default','housing','loan','contact','month','poutcome'] if c in df.columns]
#     num_c = [c for c in ['age','balance','day','duration','campaign','pdays','previous'] if c in df.columns]
#     for c in num_c: df[c] = clean_num(df[c])
#     for c in ['balance','duration','pdays','previous']:
#         if c in df.columns: df[c] = df[c].clip(upper=df[c].quantile(0.99))
#     X = df.drop(columns=['y','marital_bin','job_bin'])
#     y = df['y'].values
#     X_tr,X_te,y_tr,y_te,sm_tr,sm_te,sj_tr,sj_te = train_test_split(
#         X, y, df['marital_bin'], df['job_bin'], test_size=0.2, stratify=y, random_state=seed)
#     pre = ColumnTransformer([('n',num_pipe(),num_c),('c',cat_pipe(),cat_c)])
#     X_tr_nn = to_dense(pre.fit_transform(X_tr)); X_te_nn = to_dense(pre.transform(X_te))
#     nn_feat_names = _get_nn_feature_names(pre, num_c, cat_c)
#     bbn = df.copy()
#     for c in num_c: bbn[c] = safe_qcut(bbn[c], 5)
#     for c in cat_c+['marital','job']: bbn[c] = bbn[c].astype('category').cat.codes.astype(int)
#     bbn = _enc_bbn_objects(bbn)
#     bbn_tr = bbn.loc[X_tr.index].reset_index(drop=True)
#     bbn_te = bbn.loc[X_te.index].reset_index(drop=True)
#     bbn_tr = _filter_bbn_columns(bbn_tr, 'marital_bin', 'y',
#                                   extra_drop=('marital','job','marital_bin','job_bin'))
#     bbn_te = _filter_bbn_columns(bbn_te, 'marital_bin', 'y',
#                                   extra_drop=('marital','job','marital_bin','job_bin'))
#     r = dict(X_train_nn=X_tr_nn, X_test_nn=X_te_nn,
#              bbn_train_df=bbn_tr, bbn_test_df=bbn_te,
#              y_train=y_tr, y_test=y_te,
#              sens_marital_train=sm_tr.reset_index(drop=True), sens_marital_test=sm_te.reset_index(drop=True),
#              sens_job_train=sj_tr.reset_index(drop=True),     sens_job_test=sj_te.reset_index(drop=True),
#              num_cols=num_c, cat_cols=cat_c,
#              nn_feature_names=nn_feat_names)
#     if use_cache: joblib.dump(r, cache)
#     return r


# def load_lawschool(csv_path='/kaggle/input/datasets/anothertemp/all-datasets/bar_pass_prediction.csv',
#                    use_cache=True):
#     cache = os.path.join(CACHE_DIR, 'fast_lawschool_v15.pkl')
#     if use_cache and os.path.exists(cache):
#         tqdm.write("  [cache] LawSchool"); return joblib.load(cache)
#     df = pd.read_csv(csv_path)
#     df.columns = [c.strip().lower() for c in df.columns]
#     df = df.dropna(subset=['pass_bar','race','sex'])
#     for c in df.select_dtypes(include=[np.number]).columns: df[c] = clean_num(df[c])
#     mc = df['race'].value_counts().idxmax()
#     df['race_bin']   = (df['race'] == mc).astype(int)
#     gm = {v:i for i,v in enumerate(sorted(df['sex'].unique()))}
#     df['gender_bin'] = df['sex'].map(gm)
#     num_c = [c for c in ['lsat','ugpa','fam_inc','age']
#              if c in df.columns and df[c].nunique() > 1]
#     cat_c = [c for c in ['tier','cluster','fulltime','parttime'] if c in df.columns]
#     X = df[num_c + cat_c + ['race','sex']]
#     y = df['pass_bar'].values
#     X_tr,X_te,y_tr,y_te,sr_tr,sr_te,sg_tr,sg_te = train_test_split(
#         X, y, df['race_bin'], df['gender_bin'], test_size=0.2, stratify=y, random_state=SEED)
#     pre = ColumnTransformer([('n',num_pipe(),num_c),('c',cat_pipe(),cat_c+['race','sex'])])
#     X_tr_nn = to_dense(pre.fit_transform(X_tr)); X_te_nn = to_dense(pre.transform(X_te))
#     nn_feat_names = _get_nn_feature_names(pre, num_c, cat_c+['race','sex'])
#     bbn = df.copy()
#     for c in num_c: bbn[c] = safe_qcut(df[c], 5)
#     for c in cat_c+['race','sex']:
#         if c in bbn.columns: bbn[c] = bbn[c].astype('category').cat.codes.astype(int)
#     if 'pass_bar' in bbn.columns and bbn['pass_bar'].dtype == object:
#         bbn['pass_bar'] = bbn['pass_bar'].astype('category').cat.codes.astype(int)
#     bbn = _enc_bbn_objects(bbn)
#     bbn_tr = bbn.loc[X_tr.index].reset_index(drop=True)
#     bbn_te = bbn.loc[X_te.index].reset_index(drop=True)
#     bbn_tr = _filter_bbn_columns(bbn_tr, 'race_bin', 'pass_bar',
#                                   extra_drop=('race','sex','race_bin','gender_bin'))
#     bbn_te = _filter_bbn_columns(bbn_te, 'race_bin', 'pass_bar',
#                                   extra_drop=('race','sex','race_bin','gender_bin'))
#     r = dict(X_train_nn=X_tr_nn, X_test_nn=X_te_nn,
#              bbn_train_df=bbn_tr, bbn_test_df=bbn_te,
#              y_train=y_tr, y_test=y_te,
#              sens_race_train=sr_tr.reset_index(drop=True),   sens_race_test=sr_te.reset_index(drop=True),
#              sens_gender_train=sg_tr.reset_index(drop=True), sens_gender_test=sg_te.reset_index(drop=True),
#              num_cols=num_c, cat_cols=cat_c,
#              nn_feature_names=nn_feat_names)
#     if use_cache: joblib.dump(r, cache)
#     return r


# def load_hospital(csv_path='/kaggle/input/datasets/anothertemp/all-datasets/diabetes_hospital_fairlearn.csv',
#                   seed=SEED, use_cache=True):
#     cache = os.path.join(CACHE_DIR, 'fast_hospital_v15.pkl')
#     if use_cache and os.path.exists(cache):
#         tqdm.write("  [cache] Hospital"); return joblib.load(cache)
#     df = pd.read_csv(csv_path)
#     df = df.drop(columns=[c for c in ['max_glu_serum','A1Cresult','readmitted.1'] if c in df.columns])
#     df = df[~df.isin(['Missing']).any(axis=1)].dropna(subset=['race','gender']).reset_index(drop=True)
#     age_map = {"'0-10'":5,"'10-20'":15,"'20-30'":25,"'30-40'":35,"'40-50'":45,
#                "'50-60'":55,"'60-70'":65,"'70-80'":75,"'80-90'":85,"'90-100'":95,
#                "'30 years or younger'":20,"'30-60 years'":45,"'Over 60 years'":70}
#     df['age']        = df['age'].replace(age_map).astype(float)
#     df['y']          = df['readmit_binary'].astype(int)
#     mc               = df['race'].value_counts().idxmax()
#     df['race_bin']   = (df['race'] == mc).astype(int)
#     df['gender_bin'] = df['gender'].map({'Male':0,'Female':1}).fillna(0).astype(int)
#     cat_c = ['discharge_disposition_id','admission_source_id','medical_specialty',
#              'primary_diagnosis','insulin','change','diabetesMed']
#     num_c = ['age','time_in_hospital','num_lab_procedures','num_procedures','num_medications',
#              'number_diagnoses','had_emergency','had_inpatient_days','had_outpatient_days','medicare','medicaid']
#     for c in num_c: df[c] = clean_num(df[c])
#     X = df.drop(columns=['readmit_binary','y','race_bin','gender_bin'])
#     y = df['y'].values
#     X_tr,X_te,y_tr,y_te,sr_tr,sr_te,sg_tr,sg_te = train_test_split(
#         X, y, df['race_bin'], df['gender_bin'], test_size=0.2, stratify=y, random_state=seed)
#     pre = ColumnTransformer([('n',num_pipe(),num_c),('c',cat_pipe(),cat_c)])
#     X_tr_nn = to_dense(pre.fit_transform(X_tr)); X_te_nn = to_dense(pre.transform(X_te))
#     nn_feat_names = _get_nn_feature_names(pre, num_c, cat_c)
#     bbn = df.copy()
#     for c in num_c: bbn[c] = safe_qcut(bbn[c], 5)
#     for c in cat_c+['race','gender']: bbn[c] = bbn[c].astype('category').cat.codes.astype(int)
#     bbn = _enc_bbn_objects(bbn)
#     bbn_tr = bbn.loc[X_tr.index].reset_index(drop=True)
#     bbn_te = bbn.loc[X_te.index].reset_index(drop=True)
#     bbn_tr = _filter_bbn_columns(bbn_tr, 'race_bin', 'y',
#                                   extra_drop=('race','gender','race_bin','gender_bin','readmit_binary'))
#     bbn_te = _filter_bbn_columns(bbn_te, 'race_bin', 'y',
#                                   extra_drop=('race','gender','race_bin','gender_bin','readmit_binary'))
#     r = dict(X_train_nn=X_tr_nn, X_test_nn=X_te_nn,
#              bbn_train_df=bbn_tr, bbn_test_df=bbn_te,
#              y_train=y_tr, y_test=y_te,
#              sens_race_train=sr_tr.reset_index(drop=True), sens_race_test=sr_te.reset_index(drop=True),
#              sens_sex_train=sg_tr.reset_index(drop=True),  sens_sex_test=sg_te.reset_index(drop=True),
#              num_cols=num_c, cat_cols=cat_c,
#              nn_feature_names=nn_feat_names)
#     if use_cache: joblib.dump(r, cache)
#     return r


# def _compute_nie(bbn_df, sens_col, med_col, label_col):
#     s = bbn_df[sens_col].values.astype(int)
#     m = bbn_df[med_col].values.astype(int)
#     y = bbn_df[label_col].values.astype(int)

#     mi_sm = _compute_mi(s, m)
#     mi_sy = _compute_mi(s, y)

#     cond_mi_parts = []
#     weights       = []
#     for sv in np.unique(s):
#         mask = s == sv
#         if mask.sum() > 30:
#             cond_mi_parts.append(_compute_mi(m[mask], y[mask]))
#             weights.append(mask.sum())

#     if not cond_mi_parts:
#         return 0.0, 0.0, False

#     w_total  = sum(weights)
#     avg_cond = sum(c * w for c, w in zip(cond_mi_parts, weights)) / w_total
#     nie      = float(mi_sm * avg_cond)
#     nie_ratio = nie / max(mi_sy, 1e-9)
#     is_valid  = nie > 1e-8

#     return nie, nie_ratio, is_valid


# def _build_nie_skeleton(bbn_df, sens_col, label_col, nie_scores, top_k=6):
#     """
#     Seed the skeleton with explicit (sens -> top_NIE_mediator, top_NIE_mediator -> outcome)
#     chains to enable 3-node indirect path discovery, instead of MI-only star initialization.
#     """
#     skeleton_edges = []
#     candidates = [c for c in bbn_df.columns if c not in (sens_col, label_col)]

#     ranked_by_nie = sorted(
#         [(c, nie_scores.get(c, 0.0)) for c in candidates],
#         key=lambda x: x[1], reverse=True
#     )

#     for feat, nie_val in ranked_by_nie[:top_k]:
#         if nie_val > 1e-8:
#             skeleton_edges.append((sens_col, feat))
#             if label_col in bbn_df.columns:
#                 skeleton_edges.append((feat, label_col))

#     return skeleton_edges


# def _learn_graph_structure(bbn_df, sens_col, label_col, nie_scores, max_nodes=20):
#     cols = [c for c in bbn_df.columns if c != label_col]
#     if sens_col not in cols:
#         cols = [sens_col] + cols

#     if len(cols) > max_nodes:
#         variances = bbn_df[cols].var().sort_values(ascending=False)
#         cols = list(variances.index[:max_nodes])
#         if sens_col not in cols:
#             cols = [sens_col] + cols[:-1]

#     sub = bbn_df[cols + ([label_col] if label_col in bbn_df.columns else [])].copy()

#     skeleton = _build_nie_skeleton(sub, sens_col, label_col, nie_scores, top_k=6)

#     try:
#         hc = HillClimbSearch(sub)
#         est = hc.estimate(
#             scoring_method=BIC(sub),
#             max_iter=1500,
#             max_indegree=3,
#             epsilon=1e-6,
#             start_dag=DiscreteBayesianNetwork(skeleton) if skeleton else None,
#         )
#         edges = list(est.edges())
#         return edges if edges else skeleton
#     except:
#         return skeleton


# @dataclass
# class MediatorBiasScores:
#     sens_col:    str
#     label_col:   str
#     scores:      Dict[str, float] = field(default_factory=dict)
#     nie_scores:  Dict[str, float] = field(default_factory=dict)
#     nie_ratio:   Dict[str, float] = field(default_factory=dict)
#     sens_mi:     float = 0.0
#     verification: Dict[str, dict] = field(default_factory=dict)

#     def fit(self, bbn_df):
#         if self.sens_col not in bbn_df.columns or self.label_col not in bbn_df.columns:
#             return self
#         s = bbn_df[self.sens_col].values.astype(int)
#         y = bbn_df[self.label_col].values.astype(int)
#         self.sens_mi = _compute_mi(s, y)

#         candidates = [c for c in bbn_df.columns if c not in (self.sens_col, self.label_col)]
#         raw_scores = {}

#         for feat in candidates:
#             f = bbn_df[feat].values.astype(int)
#             mi_sf = _compute_mi(s, f)
#             mi_fy = _compute_mi(f, y)
#             nie, nie_r, valid = _compute_nie(bbn_df, self.sens_col, feat, self.label_col)
#             self.verification[feat] = dict(nie=nie, nie_ratio=nie_r, mi_sf=mi_sf, mi_fy=mi_fy, valid=valid)
#             self.nie_ratio[feat] = nie_r

#             if not valid:
#                 raw_scores[feat]      = 0.0
#                 self.nie_scores[feat] = 0.0
#                 continue

#             raw_scores[feat]      = float(0.5 * mi_fy + 0.3 * mi_sf + 0.2 * nie)
#             self.nie_scores[feat] = nie

#         valid_sc = {k: v for k, v in raw_scores.items() if v > 0}
#         if valid_sc:
#             mx = max(valid_sc.values()) + 1e-9
#             self.scores = {k: (v/mx if v > 0 else 0.0) for k, v in raw_scores.items()}
#         else:
#             self.scores = raw_scores

#         valid_nie = {k: v for k, v in self.nie_scores.items() if v > 0}
#         if valid_nie:
#             mx_nie = max(valid_nie.values()) + 1e-9
#             self.nie_scores = {k: (v/mx_nie if v > 0 else 0.0) for k, v in self.nie_scores.items()}
#         return self

#     def top_mediators(self, k=TOP_K_MEDIATORS, threshold=0.005):
#         verified = {f for f, vr in self.verification.items() if vr.get('valid', False)}
#         ranked   = sorted(verified, key=lambda f: self.scores.get(f, 0.0), reverse=True)
#         return [(f, self.scores[f]) for f in ranked if self.scores.get(f, 0.0) >= threshold][:k]


# @dataclass
# class BBNPathAnalyzer:
#     sens_col:    str
#     label_col:   str
#     bias_scores: MediatorBiasScores = field(default=None)
#     _pathway_weights: Dict[str, float] = field(default_factory=dict, repr=False)
#     _nie_weights:     Dict[str, float] = field(default_factory=dict, repr=False)
#     _learned_edges:   list = field(default_factory=list, repr=False)

#     def fit(self, bbn_df):
#         self.bias_scores = MediatorBiasScores(sens_col=self.sens_col, label_col=self.label_col)
#         self.bias_scores.fit(bbn_df)

#         top  = self.bias_scores.top_mediators(k=TOP_K_MEDIATORS, threshold=0.005)
#         self._pathway_weights = {f: s for f, s in top}
#         self._nie_weights     = {f: self.bias_scores.nie_scores.get(f, 0.0) for f, _ in top}
#         top_feats = [f for f, _ in top]

#         bbn_cols = ([self.sens_col] + top_feats +
#                     ([self.label_col] if self.label_col in bbn_df.columns else []))
#         bbn_cols = [c for c in dict.fromkeys(bbn_cols) if c in bbn_df.columns]

#         if len(bbn_cols) >= 3:
#             self._learned_edges = _learn_graph_structure(
#                 bbn_df[bbn_cols], self.sens_col, self.label_col,
#                 nie_scores=self.bias_scores.nie_scores,
#                 max_nodes=len(bbn_cols))
#         else:
#             self._learned_edges = []

#         expanded = set(top_feats)
#         for u, v in self._learned_edges:
#             if u in bbn_df.columns: expanded.add(u)
#             if v in bbn_df.columns: expanded.add(v)
#         expanded.discard(self.sens_col)
#         expanded.discard(self.label_col)

#         for f in list(expanded):
#             if f not in self._pathway_weights:
#                 self._pathway_weights[f] = self.bias_scores.scores.get(f, 0.01)
#             if f not in self._nie_weights:
#                 self._nie_weights[f] = self.bias_scores.nie_scores.get(f, 0.0)

#         return self

#     def get_pathway_weights(self):   return self._pathway_weights
#     def get_nie_weights(self):       return self._nie_weights
#     def get_sens_mi(self):           return self.bias_scores.sens_mi if self.bias_scores else 0.0
#     def get_feature_bias_scores(self): return self.bias_scores.scores if self.bias_scores else {}
#     def get_verification(self):      return self.bias_scores.verification if self.bias_scores else {}
#     def get_learned_edges(self):     return self._learned_edges


# def plot_learned_dag(dataset_name, edges, sens_col, label_col, pathway_weights,
#                      nie_scores, verification):
#     fig, ax = plt.subplots(figsize=(14, 9))
#     ax.set_facecolor('#0f0f1a')
#     fig.patch.set_facecolor('#0f0f1a')

#     G = nx.DiGraph()
#     G.add_edges_from(edges)
#     if not G.nodes():
#         plt.close(); return

#     causal_path_nodes = set()
#     causal_path_edges = set()
#     for u, v in edges:
#         if u == sens_col:
#             causal_path_nodes.add(v)
#             causal_path_edges.add((u, v))
#         if v == label_col and u != sens_col:
#             causal_path_edges.add((u, v))
#             causal_path_nodes.add(u)

#     try:
#         pos = nx.spring_layout(G, seed=42, k=2.5)
#     except:
#         pos = nx.circular_layout(G)

#     special = {sens_col, label_col}
#     regular_nodes = [n for n in G.nodes() if n not in special]
#     path_nodes    = [n for n in regular_nodes if n in causal_path_nodes]
#     other_nodes   = [n for n in regular_nodes if n not in causal_path_nodes]

#     nx.draw_networkx_nodes(G, pos, nodelist=other_nodes,
#                            node_color='#2a2a4a', node_size=800, ax=ax, alpha=0.8)
#     nx.draw_networkx_nodes(G, pos, nodelist=path_nodes,
#                            node_color='#e05c5c', node_size=950, ax=ax, alpha=0.9)

#     for sp, col, sz in [(sens_col, '#f5a623', 1200), (label_col, '#7ed321', 1200)]:
#         if sp in G.nodes():
#             nx.draw_networkx_nodes(G, pos, nodelist=[sp],
#                                    node_color=col, node_size=sz, ax=ax)

#     regular_edges   = [e for e in G.edges() if e not in causal_path_edges]
#     path_edges_list = [e for e in G.edges() if e in causal_path_edges]

#     nx.draw_networkx_edges(G, pos, edgelist=regular_edges,
#                            edge_color='#444466', arrows=True,
#                            arrowsize=15, width=1.0, ax=ax,
#                            connectionstyle='arc3,rad=0.1')
#     nx.draw_networkx_edges(G, pos, edgelist=path_edges_list,
#                            edge_color='#e05c5c', arrows=True,
#                            arrowsize=20, width=2.5, ax=ax,
#                            connectionstyle='arc3,rad=0.1')

#     labels = {}
#     for n in G.nodes():
#         nie_v = nie_scores.get(n, 0.0)
#         vrf   = verification.get(n, {})
#         mi_sf = vrf.get('mi_sf', 0.0)
#         if n == sens_col:
#             labels[n] = f"{n}\n[sensitive]"
#         elif n == label_col:
#             labels[n] = f"{n}\n[outcome]"
#         elif nie_v > 0.01:
#             labels[n] = f"{n}\nNIE={nie_v:.3f}"
#         else:
#             labels[n] = f"{n}\nMI={mi_sf:.3f}"

#     nx.draw_networkx_labels(G, pos, labels, font_size=7,
#                             font_color='white', ax=ax)

#     sens_patch  = mpatches.Patch(color='#f5a623', label=f'Sensitive: {sens_col}')
#     label_patch = mpatches.Patch(color='#7ed321', label=f'Outcome: {label_col}')
#     path_patch  = mpatches.Patch(color='#e05c5c', label='Bias pathway nodes/edges')
#     other_patch = mpatches.Patch(color='#2a2a4a', label='Other nodes')
#     ax.legend(handles=[sens_patch, label_patch, path_patch, other_patch],
#               loc='lower left', facecolor='#1a1a2e', labelcolor='white', fontsize=8)

#     ax.set_title(f"Learned Causal DAG — {dataset_name.upper()}\n"
#                  f"Red edges = bias pathways (sens→mediator or mediator→outcome)",
#                  color='white', fontsize=11, pad=12)
#     ax.axis('off')
#     plt.tight_layout()
#     out = os.path.join(RESULTS_DIR, f'dag_{dataset_name}.png')
#     plt.savefig(out, dpi=120, bbox_inches='tight', facecolor=fig.get_facecolor())
#     plt.close()
#     tqdm.write(f"  [DAG] saved → {out}")


# def analyze_causal_pathways(dataset_name, edges, sens_col, label_col,
#                              pathway_weights, nie_scores, verification):
#     direct_edges   = [(u, v) for u, v in edges if u == sens_col]
#     indirect_paths = []
#     for u, v in edges:
#         if u == sens_col:
#             for u2, v2 in edges:
#                 if u2 == v and v2 == label_col:
#                     indirect_paths.append((u, v, v2))

#     print(f"\n  {'─'*60}")
#     print(f"  CAUSAL PATHWAY ANALYSIS — {dataset_name.upper()}")
#     print(f"  {'─'*60}")
#     print(f"  Sensitive attribute : {sens_col}")
#     print(f"  Outcome node        : {label_col}")
#     print(f"  Total learned edges : {len(edges)}")
#     print(f"  Direct edges from {sens_col}: {len(direct_edges)}")
#     for u, v in direct_edges:
#         nie_v = nie_scores.get(v, 0.0)
#         vrf   = verification.get(v, {})
#         print(f"    {u} → {v}  (NIE={nie_v:.4f}, MI_SF={vrf.get('mi_sf',0):.4f}, MI_FY={vrf.get('mi_fy',0):.4f})")

#     if indirect_paths:
#         print(f"  Indirect paths sens→med→outcome ({len(indirect_paths)}):")
#         for u, m, v in indirect_paths:
#             nie_v = nie_scores.get(m, 0.0)
#             print(f"    {u} → {m} → {v}  (NIE={nie_v:.4f})")
#     else:
#         print(f"  No direct sens→mediator→outcome 3-node paths found in learned graph.")
#         print(f"  Note: bias transmission may operate through longer or distributed pathways.")

#     print(f"  Top mediators by NIE score (path-specific natural indirect effect):")
#     top_nie = sorted(nie_scores.items(), key=lambda x: x[1], reverse=True)[:5]
#     for feat, nie_v in top_nie:
#         vrf = verification.get(feat, {})
#         print(f"    {feat:<25} NIE={nie_v:.4f}  ratio={vrf.get('nie_ratio',0):.4f}")

#     analysis = dict(
#         direct_edges=direct_edges,
#         indirect_paths=indirect_paths,
#         top_nie=top_nie,
#         n_edges=len(edges),
#     )
#     CAUSAL_ANALYSIS[dataset_name] = analysis
#     return analysis


# def analyze_impossibility(dataset_name, y_test, pred_eo, pred_dp, s_test, eo_val, dp_val):
#     s0_mask = s_test == 0
#     s1_mask = s_test == 1

#     base_rate_0 = float(y_test[s0_mask].mean()) if s0_mask.sum() > 0 else float('nan')
#     base_rate_1 = float(y_test[s1_mask].mean()) if s1_mask.sum() > 0 else float('nan')
#     base_rate_diff = abs(base_rate_0 - base_rate_1)

#     pred_rate_0_eo = float(pred_eo[s0_mask].mean()) if s0_mask.sum() > 0 else float('nan')
#     pred_rate_1_eo = float(pred_eo[s1_mask].mean()) if s1_mask.sum() > 0 else float('nan')
#     pred_const_eo  = float(np.std(pred_eo)) < 0.01

#     pred_rate_0_dp = float(pred_dp[s0_mask].mean()) if s0_mask.sum() > 0 else float('nan')
#     pred_rate_1_dp = float(pred_dp[s1_mask].mean()) if s1_mask.sum() > 0 else float('nan')
#     pred_const_dp  = float(np.std(pred_dp)) < 0.01

#     chouldechova_condition = base_rate_diff < 0.02
#     both_near_zero = eo_val < PASS_THRESHOLD and dp_val < PASS_THRESHOLD

#     if both_near_zero:
#         if pred_const_eo or pred_const_dp:
#             mechanism = "degenerate_constant"
#             explanation = ("Classifier collapsed toward near-constant predictions under "
#                            "fairness pressure. Both EO and DP approach zero trivially.")
#         elif chouldechova_condition:
#             mechanism = "equal_base_rates"
#             explanation = ("Base rates across sensitive groups are approximately equal "
#                            f"({base_rate_0:.3f} vs {base_rate_1:.3f}). Under equal base rates, "
#                            "the Chouldechova impossibility constraint relaxes and simultaneous "
#                            "EO+DP satisfaction is achievable without degeneracy.")
#         else:
#             mechanism = "approximate_satisfaction"
#             explanation = (f"Base rates differ ({base_rate_0:.3f} vs {base_rate_1:.3f}) but "
#                            "both metrics are below rounding threshold. This is consistent with "
#                            "the theoretical impossibility — exact simultaneous satisfaction is "
#                            "not achieved, but practical near-zero is reached within measurement precision.")
#     else:
#         mechanism = "partial"
#         explanation = "One or both metrics above threshold — impossibility constraint active."

#     print(f"\n  {'─'*60}")
#     print(f"  IMPOSSIBILITY ANALYSIS — {dataset_name.upper()}")
#     print(f"  {'─'*60}")
#     print(f"  Group base rates: s=0 → {base_rate_0:.4f}, s=1 → {base_rate_1:.4f}  (diff={base_rate_diff:.4f})")
#     print(f"  EO={eo_val:.4f}  DP={dp_val:.4f}  both_near_zero={both_near_zero}")
#     print(f"  Mechanism : {mechanism}")
#     print(f"  Explanation: {explanation}")
#     if pred_const_eo:
#         print(f"  WARNING: EO model predictions have std < 0.01 — near-constant output detected. Result flagged as degenerate.")
#     if pred_const_dp:
#         print(f"  WARNING: DP model predictions have std < 0.01 — near-constant output detected. Result flagged as degenerate.")

#     result = dict(
#         base_rate_0=base_rate_0, base_rate_1=base_rate_1,
#         base_rate_diff=base_rate_diff,
#         pred_rate_0_eo=pred_rate_0_eo, pred_rate_1_eo=pred_rate_1_eo,
#         pred_const_eo=pred_const_eo,
#         pred_rate_0_dp=pred_rate_0_dp, pred_rate_1_dp=pred_rate_1_dp,
#         pred_const_dp=pred_const_dp,
#         chouldechova_condition=chouldechova_condition,
#         mechanism=mechanism, explanation=explanation,
#         eo_val=eo_val, dp_val=dp_val,
#         degenerate=(pred_const_eo or pred_const_dp),
#     )
#     IMPOSSIBILITY_ANALYSIS[dataset_name] = result
#     return result


# def analyze_representation_suppression(dataset_name, enc, task_hd,
#                                         X_te_med, X_te_task,
#                                         s_test, y_test,
#                                         top_med_names, nie_scores,
#                                         nn_feat_names):
#     enc.eval(); task_hd.eval()
#     dev = next(enc.parameters()).device

#     X_med_t  = torch.tensor(X_te_med,  dtype=torch.float32).to(dev)
#     X_task_t = torch.tensor(X_te_task, dtype=torch.float32).to(dev)

#     with torch.no_grad():
#         z_task, z_med = enc(X_med_t, X_task_t)
#         z_task_np = z_task.cpu().numpy()
#         z_med_np  = z_med.cpu().numpy()

#     z_task_binned = np.digitize(z_task_np, bins=np.linspace(z_task_np.min(), z_task_np.max(), 10)) - 1
#     mi_ztask_s = float(np.mean([_compute_mi(z_task_binned[:, i], s_test.astype(int))
#                                  for i in range(min(10, z_task_np.shape[1]))]))
#     mi_ztask_y = float(np.mean([_compute_mi(z_task_binned[:, i], y_test.astype(int))
#                                  for i in range(min(10, z_task_np.shape[1]))]))

#     z_med_binned = np.digitize(z_med_np, bins=np.linspace(z_med_np.min(), z_med_np.max(), 10)) - 1
#     mi_zmed_s = float(np.mean([_compute_mi(z_med_binned[:, i], s_test.astype(int))
#                                 for i in range(min(10, z_med_np.shape[1]))]))

#     raw_mi_s = _compute_mi(s_test.astype(int), y_test.astype(int))

#     print(f"\n  {'─'*60}")
#     print(f"  REPRESENTATION SUPPRESSION ANALYSIS — {dataset_name.upper()}")
#     print(f"  {'─'*60}")
#     print(f"  MI(S, Y) raw data          : {raw_mi_s:.5f}")
#     print(f"  MI(z_task, S) avg per-dim  : {mi_ztask_s:.5f}  (target: near 0)")
#     print(f"  MI(z_task, Y) avg per-dim  : {mi_ztask_y:.5f}  (want: high)")
#     print(f"  MI(z_med,  S) avg per-dim  : {mi_zmed_s:.5f}  (want: non-trivial — encodes mediated S signal)")
#     suppression_ratio = 1.0 - min(mi_ztask_s / max(raw_mi_s, 1e-9), 1.0)
#     print(f"  Suppression ratio (z_task) : {suppression_ratio:.4f}  (1.0 = perfect)")
#     if mi_ztask_s > raw_mi_s:
#         print(f"  WARNING: z_task is amplifying sensitive info (MI(z_task,S) > MI(S,Y)). "
#               f"Adversary is not suppressing nonlinear S leakage effectively.")

#     print(f"  Per-mediator NIE vs representation MI:")
#     print(f"  {'Feature':<22} {'NIE':>7} {'Expected suppression':>20}")
#     rep_details = {}
#     for feat in top_med_names[:8]:
#         nie_v = nie_scores.get(feat, 0.0)
#         expected = "high" if nie_v > 0.3 else ("medium" if nie_v > 0.05 else "low")
#         print(f"  {feat:<22} {nie_v:>7.4f} {expected:>20}")
#         rep_details[feat] = dict(nie=nie_v, expected_suppression=expected)

#     result = dict(
#         mi_raw_s=raw_mi_s,
#         mi_ztask_s=mi_ztask_s,
#         mi_ztask_y=mi_ztask_y,
#         mi_zmed_s=mi_zmed_s,
#         suppression_ratio=suppression_ratio,
#         per_mediator=rep_details,
#     )
#     REPRESENTATION_ANALYSIS[dataset_name] = result
#     return result


# def _split_x_by_mediators(X_nn, mediator_names, nn_feature_names):
#     if not mediator_names or not nn_feature_names:
#         dummy = np.zeros((X_nn.shape[0], 1), dtype=X_nn.dtype)
#         return dummy, X_nn

#     med_idx = []
#     for i, fname in enumerate(nn_feature_names):
#         base = fname.split('=')[0]
#         if fname in mediator_names or base in mediator_names:
#             med_idx.append(i)

#     if not med_idx:
#         dummy = np.zeros((X_nn.shape[0], 1), dtype=X_nn.dtype)
#         return dummy, X_nn

#     task_idx = [i for i in range(X_nn.shape[1]) if i not in set(med_idx)]
#     if not task_idx:
#         task_idx = list(range(X_nn.shape[1]))
#     return X_nn[:, med_idx], X_nn[:, task_idx]


# def _split_x_ablation_no_bbn(X_nn):
#     dummy = np.zeros((X_nn.shape[0], 1), dtype=X_nn.dtype)
#     return dummy, X_nn


# class GradRevFn(torch.autograd.Function):
#     @staticmethod
#     def forward(ctx, x, alpha):
#         ctx.alpha = alpha
#         return x.view_as(x)
#     @staticmethod
#     def backward(ctx, grad_output):
#         return -ctx.alpha * grad_output, None


# class SplitInputEncoder(nn.Module):
#     def __init__(self, in_dim_med, in_dim_task, hidden=256,
#                  z_med_dim=64, z_task_dim=128, dropout=0.3):
#         super().__init__()
#         self.z_med_dim  = z_med_dim
#         self.z_task_dim = z_task_dim
#         self.med_encoder = nn.Sequential(
#             nn.Linear(in_dim_med, max(hidden // 2, 16)), nn.LayerNorm(max(hidden // 2, 16)), nn.GELU(), nn.Dropout(dropout),
#             nn.Linear(max(hidden // 2, 16), max(hidden // 2, 16)), nn.LayerNorm(max(hidden // 2, 16)), nn.GELU(), nn.Dropout(dropout),
#             nn.Linear(max(hidden // 2, 16), z_med_dim), nn.LayerNorm(z_med_dim),
#         )
#         self.task_encoder = nn.Sequential(
#             nn.Linear(in_dim_task, hidden), nn.LayerNorm(hidden), nn.GELU(), nn.Dropout(dropout),
#             nn.Linear(hidden, hidden), nn.LayerNorm(hidden), nn.GELU(), nn.Dropout(dropout),
#             nn.Linear(hidden, z_task_dim), nn.LayerNorm(z_task_dim),
#         )

#     def forward(self, x_med, x_task):
#         return self.task_encoder(x_task), self.med_encoder(x_med)


# class TaskHead(nn.Module):
#     def __init__(self, z_dim=128):
#         super().__init__()
#         self.net = nn.Sequential(
#             nn.Linear(z_dim, z_dim // 2), nn.GELU(),
#             nn.Linear(z_dim // 2, 1),
#         )
#     def forward(self, z): return self.net(z).squeeze(-1)


# class TaskAdversary(nn.Module):
#     def __init__(self, z_task_dim=128, hidden=128, nie_scalar=1.0):
#         super().__init__()
#         self.nie_scalar = nie_scalar
#         self.net = nn.Sequential(
#             nn.Linear(z_task_dim, hidden), nn.LayerNorm(hidden), nn.GELU(), nn.Dropout(0.2),
#             nn.Linear(hidden, hidden // 2), nn.GELU(),
#             nn.Linear(hidden // 2, 2),
#         )
#     def forward(self, z_task, alpha=1.0):
#         return self.net(GradRevFn.apply(z_task, alpha * self.nie_scalar))


# class NonlinearLeakageProbe(nn.Module):
#     """
#     MLP probe for nonlinear sensitive-attribute leakage detection in z_task.
#     Replaces the single linear layer to capture nonlinear S-leakage that
#     the linear probe would miss. Gradients flow into the encoder during training
#     to actively suppress leakage, not just diagnose it.
#     """
#     def __init__(self, z_task_dim=128, hidden=64):
#         super().__init__()
#         self.net = nn.Sequential(
#             nn.Linear(z_task_dim, hidden),
#             nn.ReLU(),
#             nn.Linear(hidden, hidden // 2),
#             nn.ReLU(),
#             nn.Linear(hidden // 2, 2),
#         )
#     def forward(self, z): return self.net(z)


# class MediatorReconstructor(nn.Module):
#     def __init__(self, z_med_dim=64, n_mediators=5, hidden=128):
#         super().__init__()
#         self.net = nn.Sequential(
#             nn.Linear(z_med_dim, hidden), nn.GELU(),
#             nn.Linear(hidden, hidden), nn.GELU(),
#             nn.Linear(hidden, n_mediators),
#         )
#     def forward(self, z): return self.net(z)


# def fair_loss_eo(logit, y, s, margin=0.02):
#     p = torch.sigmoid(logit)
#     tprs, fprs = [], []
#     for g in (0, 1):
#         pm = (s == g) & (y == 1); nm = (s == g) & (y == 0)
#         tprs.append(p[pm].mean() if pm.sum() > 1 else torch.tensor(0.5, device=logit.device))
#         fprs.append(p[nm].mean() if nm.sum() > 1 else torch.tensor(0.5, device=logit.device))
#     return F.relu(torch.max(torch.abs(tprs[0]-tprs[1]), torch.abs(fprs[0]-fprs[1])) - margin)


# def fair_loss_dp(logit, s, margin=0.02):
#     p = torch.sigmoid(logit)
#     p0 = p[s==0].mean() if (s==0).sum() > 1 else torch.tensor(0.5, device=logit.device)
#     p1 = p[s==1].mean() if (s==1).sum() > 1 else torch.tensor(0.5, device=logit.device)
#     return F.relu(torch.abs(p0-p1) - margin)


# def fair_loss_eo_bounded(logit, y, s, margin=0.08):
#     return fair_loss_eo(logit, y, s, margin=margin)


# def hsic_loss(z_a, z_b):
#     """
#     Hilbert-Schmidt Independence Criterion approximation using RBF kernel.
#     Captures nonlinear dependence between z_med and z_task, replacing the
#     linear cross-covariance orthogonality loss.
#     """
#     n = z_a.shape[0]
#     if n < 4:
#         return torch.tensor(0.0, device=z_a.device)
#     sigma_a = z_a.std().detach().clamp(min=1e-6)
#     sigma_b = z_b.std().detach().clamp(min=1e-6)
#     K = torch.exp(-torch.cdist(z_a, z_a) ** 2 / (2 * sigma_a ** 2))
#     L = torch.exp(-torch.cdist(z_b, z_b) ** 2 / (2 * sigma_b ** 2))
#     H = torch.eye(n, device=z_a.device) - (1.0 / n)
#     KH = K @ H
#     LH = L @ H
#     return (KH * LH.T).sum() / ((n - 1) ** 2)


# def mediator_preservation_loss(z_med, s, med_targets):
#     """
#     Ensures z_med retains the S->mediator correlation signal rather than suppressing it.
#     Loss = negative MI proxy between z_med and s, weighted by whether mediator targets
#     actually differ across s groups. This makes the asymmetry explicit:
#     z_task should NOT encode S; z_med SHOULD encode the mediated S signal.
#     """
#     if med_targets is None or med_targets.shape[1] == 0:
#         return torch.tensor(0.0, device=z_med.device)
#     s0_mask = (s == 0)
#     s1_mask = (s == 1)
#     if s0_mask.sum() < 2 or s1_mask.sum() < 2:
#         return torch.tensor(0.0, device=z_med.device)
#     med_mean_0 = med_targets[s0_mask].mean(0)
#     med_mean_1 = med_targets[s1_mask].mean(0)
#     med_diff = (med_mean_0 - med_mean_1).abs().mean().detach()
#     if med_diff < 1e-6:
#         return torch.tensor(0.0, device=z_med.device)
#     z_med_mean_0 = z_med[s0_mask].mean(0)
#     z_med_mean_1 = z_med[s1_mask].mean(0)
#     z_diff = (z_med_mean_0 - z_med_mean_1).norm()
#     return -z_diff * med_diff


# def weighted_orthogonality_loss(z_med, z_task, med_weights):
#     zm = z_med  - z_med.mean(0, keepdim=True)
#     zt = z_task - z_task.mean(0, keepdim=True)
#     cross = torch.mm(F.normalize(zm, dim=1).T, F.normalize(zt, dim=1)) / zm.shape[0]
#     csq   = cross ** 2
#     if med_weights is not None and med_weights.shape[0] == csq.shape[0]:
#         return (csq * med_weights.unsqueeze(1).expand_as(csq)).mean()
#     return csq.mean()


# @torch.no_grad()
# def get_proba(enc, task_hd, X_med, X_task):
#     enc.eval(); task_hd.eval()
#     dev = next(enc.parameters()).device
#     if not isinstance(X_med, torch.Tensor):
#         X_med  = torch.tensor(X_med,  dtype=torch.float32)
#         X_task = torch.tensor(X_task, dtype=torch.float32)
#     z_task, _ = enc(X_med.to(dev), X_task.to(dev))
#     return torch.sigmoid(task_hd(z_task)).cpu().numpy()


# def _train_single(dataset_name, data, bbn_analyzer, primary_sens_train,
#                   primary_sens_test, acc_floor, target, ablation_mode='full'):
#     cfg      = EPOCH_CFG.get(dataset_name, dict(total=500, warmup=20, patience=20))
#     total_ep  = cfg['total']
#     warmup_ep = cfg['warmup']
#     patience  = cfg['patience']
#     batch     = 2048

#     use_bbn  = ablation_mode not in ('no_bbn',)
#     use_adv  = ablation_mode not in ('no_adv',)

#     y_np = data['y_test']
#     s_np = primary_sens_test
#     nn_feat_names = data.get('nn_feature_names', [])
#     alpha_cap = ALPHA_CAPS.get(dataset_name, 2.5)
#     alpha_min = ALPHA_MINS.get(dataset_name, 0.5)
#     fw_scale  = FW_SCALE.get(dataset_name, 1.0) if ablation_mode == 'full' else 0.8

#     if use_bbn:
#         sens_mi   = bbn_analyzer.get_sens_mi()
#         pw        = bbn_analyzer.get_pathway_weights()
#         nie_w     = bbn_analyzer.get_nie_weights()
#         nie_scalar = float(np.clip(1.0 + np.mean(list(nie_w.values())) * 5.0, 1.0, 4.0)) if nie_w else 1.0

#         global_alpha = float(np.clip(
#             alpha_min + sens_mi * 12.0 + np.sqrt(max(sens_mi, 0)) * 3.0,
#             alpha_min, alpha_cap))

#         top_med_names = sorted(pw.items(), key=lambda x: x[1], reverse=True)
#         top_med_names = [f for f, _ in top_med_names]

#         bbn_cols  = list(data['bbn_train_df'].columns)
#         med_col_idx = [bbn_cols.index(m) for m in top_med_names if m in bbn_cols]
#         n_med_feats = len(med_col_idx)
#         bbn_tr_arr  = data['bbn_train_df'].values.astype(np.float32)
#         med_w_np    = np.array([pw.get(f, 0.0) for f in top_med_names], dtype=np.float32)
#         if med_w_np.sum() > 0: med_w_np /= med_w_np.sum()

#         X_tr_med, X_tr_task = _split_x_by_mediators(
#             data['X_train_nn'], top_med_names, nn_feat_names)
#         X_te_med, X_te_task = _split_x_by_mediators(
#             data['X_test_nn'], top_med_names, nn_feat_names)
#     else:
#         sens_mi=0.0; pw={}; nie_w={}; nie_scalar=1.0
#         global_alpha = alpha_min
#         top_med_names=[]; med_col_idx=[]; n_med_feats=0
#         bbn_tr_arr=None; med_w_np=np.array([])
#         X_tr_med, X_tr_task = _split_x_ablation_no_bbn(data['X_train_nn'])
#         X_te_med, X_te_task = _split_x_ablation_no_bbn(data['X_test_nn'])

#     X_tr_med_t  = torch.tensor(X_tr_med,  dtype=torch.float32)
#     X_tr_task_t = torch.tensor(X_tr_task, dtype=torch.float32)
#     y_tr = torch.tensor(data['y_train'],    dtype=torch.float32)
#     s_tr = torch.tensor(primary_sens_train, dtype=torch.long)

#     in_dim_med  = X_tr_med.shape[1]
#     in_dim_task = X_tr_task.shape[1]
#     z_med_dim   = 64; z_task_dim = 128

#     n_train = len(X_tr_med_t)
#     ds  = TensorDataset(X_tr_med_t, X_tr_task_t, y_tr, s_tr, torch.arange(n_train))
#     loader = DataLoader(ds, batch_size=batch, shuffle=True,
#                         num_workers=0, pin_memory=torch.cuda.is_available())

#     bbn_tr_tensor = None
#     if use_bbn and bbn_tr_arr is not None and n_med_feats > 0:
#         bbn_tr_tensor = torch.tensor(bbn_tr_arr[:, med_col_idx], dtype=torch.float32)

#     enc      = SplitInputEncoder(in_dim_med, in_dim_task, hidden=256,
#                                   z_med_dim=z_med_dim, z_task_dim=z_task_dim).to(DEVICE)
#     task_hd  = TaskHead(z_dim=z_task_dim).to(DEVICE)
#     task_adv = TaskAdversary(z_task_dim=z_task_dim, hidden=128, nie_scalar=nie_scalar).to(DEVICE)
#     probe    = NonlinearLeakageProbe(z_task_dim=z_task_dim, hidden=64).to(DEVICE)

#     actual_n_med_feats = n_med_feats if use_bbn and n_med_feats > 0 else 0
#     med_recon = (MediatorReconstructor(z_med_dim, actual_n_med_feats, 128).to(DEVICE)
#                  if use_bbn and actual_n_med_feats > 0 else None)

#     med_weights_t = None
#     if use_bbn and len(med_w_np) > 0:
#         padded = np.zeros(z_med_dim, dtype=np.float32)
#         n_fill = min(len(med_w_np), z_med_dim)
#         padded[:n_fill] = med_w_np[:n_fill]
#         med_weights_t = torch.tensor(padded).to(DEVICE)

#     smooth = 0.05

#     main_params = list(enc.parameters()) + list(task_hd.parameters())
#     if med_recon is not None: main_params += list(med_recon.parameters())
#     main_params += list(probe.parameters())

#     opt_main     = optim.AdamW(main_params,           lr=3e-4, weight_decay=1e-4)
#     opt_task_adv = optim.AdamW(task_adv.parameters(), lr=1e-3, weight_decay=1e-4)
#     sched = optim.lr_scheduler.CosineAnnealingLR(opt_main, T_max=total_ep, eta_min=5e-6)

#     best_state = None; best_metric = float('inf'); zero_streak = 0
#     epoch_leakage = []
#     mean_leak = 0.5
#     leak_ema  = 0.5

#     def compute_alpha(ep, current_leak):
#         ramp = float(np.clip((ep - warmup_ep) / max(total_ep - warmup_ep, 1), 0, 1))
#         leak_boost = float(np.clip((current_leak - 0.55) * 4.0, 0.0, 2.0))
#         return global_alpha * ramp * (1.0 + leak_boost)

#     curve_epochs=[]; curve_acc=[]; curve_eo=[]; curve_dp=[]

#     def snap():
#         d = {'enc': copy.deepcopy(enc.state_dict()), 'task_hd': copy.deepcopy(task_hd.state_dict())}
#         if med_recon is not None: d['med_recon'] = copy.deepcopy(med_recon.state_dict())
#         return d

#     def load_state(st):
#         if st:
#             enc.load_state_dict(st['enc']); task_hd.load_state_dict(st['task_hd'])
#             if med_recon is not None and 'med_recon' in st:
#                 med_recon.load_state_dict(st['med_recon'])

#     best_pred_final = None

#     for ep in range(1, total_ep + 1):
#         enc.train(); task_hd.train(); task_adv.train(); probe.train()
#         if med_recon is not None: med_recon.train()

#         fair      = ep > warmup_ep
#         fair_ramp = min(1.0, (ep - warmup_ep) / max(8.0, warmup_ep*0.4)) if fair else 0.0
#         alpha     = compute_alpha(ep, leak_ema) if fair else 0.0
#         margin = max(0.001, 0.05 * (1.0 - fair_ramp))

#         batch_leakage = []

#         for xb_med, xb_task, yb, sb, idx in loader:
#             xb_med = xb_med.to(DEVICE); xb_task = xb_task.to(DEVICE)
#             yb = yb.to(DEVICE); sb = sb.to(DEVICE); idx = idx.long()

#             if use_adv and fair:
#                 with torch.no_grad():
#                     z_task_det2, _ = enc(xb_med, xb_task)
#                 adv_logits = task_adv(z_task_det2, alpha=alpha)
#                 adv_loss   = F.cross_entropy(adv_logits, sb)
#                 opt_task_adv.zero_grad(); adv_loss.backward()
#                 nn.utils.clip_grad_norm_(task_adv.parameters(), 1.0); opt_task_adv.step()

#             z_task, z_med = enc(xb_med, xb_task)
#             logit = task_hd(z_task)
#             ybs   = yb*(1-smooth) + 0.5*smooth
#             loss  = F.binary_cross_entropy_with_logits(logit, ybs)

#             probe_logits = probe(z_task)
#             probe_loss   = F.cross_entropy(probe_logits, sb)
#             batch_leakage.append((probe_logits.argmax(1)==sb).float().mean().item())
#             loss += 0.5 * probe_loss

#             if fair and fair_ramp > 0:
#                 if target == 'eo':
#                     loss += 1.5*fw_scale*fair_ramp * fair_loss_eo(logit, yb, sb, margin=margin)
#                 else:
#                     loss += 1.5*fw_scale*fair_ramp * fair_loss_dp(logit, sb, margin=margin)
#                     loss += 1.0*fw_scale*fair_ramp * fair_loss_eo_bounded(logit, yb, sb, margin=0.08)

#             if use_adv and fair and fair_ramp > 0:
#                 adv_out  = task_adv(z_task, alpha=alpha)
#                 adv_loss = F.cross_entropy(adv_out, sb)
#                 leak_w = float(np.clip((leak_ema - 0.5) * 4.0, 0.0, 3.0))
#                 loss += (1.0 + leak_w) * fair_ramp * adv_loss

#             if fair and fair_ramp > 0:
#                 loss += 0.6*fair_ramp * hsic_loss(z_med, z_task)

#             if use_bbn and fair and fair_ramp > 0 and med_recon is not None and bbn_tr_tensor is not None:
#                 med_targets = bbn_tr_tensor[idx].to(DEVICE)
#                 recon_loss = F.mse_loss(med_recon(z_med), med_targets)
#                 loss += 0.8*fw_scale*fair_ramp * recon_loss
#                 pres_loss = mediator_preservation_loss(z_med, sb, med_targets)
#                 loss += 0.4*fw_scale*fair_ramp * pres_loss

#             opt_main.zero_grad(); loss.backward()
#             nn.utils.clip_grad_norm_(enc.parameters(), 1.0)
#             nn.utils.clip_grad_norm_(task_hd.parameters(), 1.0)
#             opt_main.step()

#         sched.step()
#         if batch_leakage:
#             mean_leak = float(np.mean(batch_leakage))
#             leak_ema  = 0.9 * leak_ema + 0.1 * mean_leak
#             epoch_leakage.append(mean_leak)

#         if ep == warmup_ep:
#             best_state = snap()

#         eval_interval = 5 if fair else 10
#         if not (ep % eval_interval == 0 or ep == total_ep):
#             continue

#         pr   = get_proba(enc, task_hd,
#                           torch.tensor(X_te_med, dtype=torch.float32),
#                           torch.tensor(X_te_task, dtype=torch.float32))
#         pred = (pr > 0.5).astype(int)
#         pred_std = float(np.std(pr))
#         acc  = accuracy_score(y_np, pred)
#         try:    eo_v = abs(float(equalized_odds_difference(y_np, pred, sensitive_features=s_np)))
#         except: eo_v = 1.0
#         try:    dp_v = abs(float(demographic_parity_difference(y_np, pred, sensitive_features=s_np)))
#         except: dp_v = 1.0

#         target_metric = eo_v if target == 'eo' else dp_v
#         is_non_degenerate = pred_std >= PRED_STD_MIN

#         if fair:
#             penalised = target_metric + max(0.0, acc_floor - acc)*2.5
#             if not is_non_degenerate:
#                 penalised += 10.0
#             if penalised < best_metric and is_non_degenerate:
#                 best_metric = penalised
#                 best_state = snap()
#                 best_pred_final = pred.copy()

#             if target_metric < 0.003 and is_non_degenerate:
#                 zero_streak += 1
#                 if zero_streak >= patience:
#                     tqdm.write(f"    [{target.upper()}] Early stop ep{ep} (metric<0.003 x{zero_streak}, pred_std={pred_std:.3f})")
#                     break
#             else:
#                 zero_streak = 0

#         curve_epochs.append(ep); curve_acc.append(acc)
#         curve_eo.append(eo_v);   curve_dp.append(dp_v)

#         if (ep % 10 == 0 or ep == total_ep) and ablation_mode == 'full':
#             phase = 'fair' if fair else 'warm'
#             degen_flag = ' [DEGEN]' if not is_non_degenerate else ''
#             tqdm.write(f"  [{target.upper()}] ep{ep:>4}: acc={acc:.4f}  "
#                        f"EO={eo_v:.4f}  DP={dp_v:.4f}  margin={margin:.4f}  "
#                        f"α={alpha:.3f}  leak={mean_leak:.3f}  std={pred_std:.3f}  [{phase}]{degen_flag}")

#     if ablation_mode == 'full':
#         TRAINING_CURVES[f"{dataset_name}_{target}"] = dict(
#             epochs=curve_epochs, acc=curve_acc, eo=curve_eo, dp=curve_dp,
#             warmup_ep=warmup_ep, leakage=epoch_leakage, target=target)

#     load_state(best_state)
#     pr_te    = get_proba(enc, task_hd,
#                           torch.tensor(X_te_med, dtype=torch.float32),
#                           torch.tensor(X_te_task, dtype=torch.float32))
#     pred_final = (pr_te > 0.5).astype(int)
#     pred_final_std = float(np.std(pr_te))
#     acc_final  = accuracy_score(y_np, pred_final)
#     try:
#         metric_final = abs(float(
#             equalized_odds_difference(y_np, pred_final, sensitive_features=s_np) if target=='eo'
#             else demographic_parity_difference(y_np, pred_final, sensitive_features=s_np)))
#     except: metric_final = 1.0

#     if pred_final_std < PRED_STD_MIN:
#         tqdm.write(f"  WARNING [{target.upper()}] final pred std={pred_final_std:.4f} < {PRED_STD_MIN} — result may be degenerate.")

#     return acc_final, metric_final, acc_final, metric_final, enc, task_hd, X_tr_med, X_tr_task, X_te_med, X_te_task, pred_final


# def _train_transfer(dataset_name, data, bbn_analyzer,
#                     s_tr_primary, s_te_primary,
#                     s_tr_secondary, s_te_secondary,
#                     acc_floor, target,
#                     primary_enc_state, primary_hd_state,
#                     X_tr_med, X_tr_task, X_te_med, X_te_task):

#     cfg       = EPOCH_CFG.get(dataset_name, dict(total=500, warmup=20, patience=20))
#     finetune_ep = max(int(cfg['total'] * 0.4), 30)
#     batch     = 2048
#     alpha_cap = ALPHA_CAPS.get(dataset_name, 2.5)
#     alpha_min = ALPHA_MINS.get(dataset_name, 0.5)

#     pw      = bbn_analyzer.get_pathway_weights()
#     nie_w   = bbn_analyzer.get_nie_weights()
#     sens_mi = bbn_analyzer.get_sens_mi()
#     nie_scalar = float(np.clip(1.0 + np.mean(list(nie_w.values()))*5.0, 1.0, 4.0)) if nie_w else 1.0
#     global_alpha_pri = float(np.clip(
#         alpha_min + sens_mi * 12.0 + np.sqrt(max(sens_mi, 0)) * 3.0,
#         alpha_min, alpha_cap))

#     s2_mi  = float(_compute_mi(s_tr_secondary.astype(int), data['y_train'].astype(int)))
#     ratio  = float(np.clip(s2_mi / max(sens_mi, 1e-6), 0.5, 1.5))
#     global_alpha_sec = global_alpha_pri * ratio

#     in_dim_med  = X_tr_med.shape[1]
#     in_dim_task = X_tr_task.shape[1]
#     z_task_dim  = 128; z_med_dim = 64

#     X_tr_med_t  = torch.tensor(X_tr_med,  dtype=torch.float32)
#     X_tr_task_t = torch.tensor(X_tr_task, dtype=torch.float32)
#     y_tr  = torch.tensor(data['y_train'], dtype=torch.float32)
#     s1_tr = torch.tensor(s_tr_primary,    dtype=torch.long)
#     s2_tr = torch.tensor(s_tr_secondary,  dtype=torch.long)

#     fw_scale = FW_SCALE.get(dataset_name, 1.0)

#     ds     = TensorDataset(X_tr_med_t, X_tr_task_t, y_tr, s1_tr, s2_tr)
#     loader = DataLoader(ds, batch_size=batch, shuffle=True, num_workers=0,
#                         pin_memory=torch.cuda.is_available())

#     enc     = SplitInputEncoder(in_dim_med, in_dim_task, 256, z_med_dim, z_task_dim).to(DEVICE)
#     task_hd = TaskHead(z_task_dim).to(DEVICE)

#     enc.load_state_dict(primary_enc_state)
#     task_hd.load_state_dict(primary_hd_state)

#     adv1  = TaskAdversary(z_task_dim, 128, nie_scalar).to(DEVICE)
#     adv2  = TaskAdversary(z_task_dim, 128, nie_scalar).to(DEVICE)
#     probe = NonlinearLeakageProbe(z_task_dim, hidden=64).to(DEVICE)

#     med_weights_t = None
#     if pw:
#         top_med_names = sorted(pw.items(), key=lambda x: x[1], reverse=True)
#         top_med_names = [f for f, _ in top_med_names]
#         med_w_np = np.array([pw.get(f, 0.0) for f in top_med_names], dtype=np.float32)
#         if med_w_np.sum() > 0: med_w_np /= med_w_np.sum()
#         padded = np.zeros(z_med_dim, dtype=np.float32)
#         n_fill = min(len(med_w_np), z_med_dim)
#         padded[:n_fill] = med_w_np[:n_fill]
#         med_weights_t = torch.tensor(padded).to(DEVICE)

#     main_params = list(enc.parameters()) + list(task_hd.parameters()) + list(probe.parameters())
#     opt_main  = optim.AdamW(main_params, lr=5e-5, weight_decay=1e-4)
#     opt_adv1  = optim.AdamW(adv1.parameters(), lr=1e-3, weight_decay=1e-4)
#     opt_adv2  = optim.AdamW(adv2.parameters(), lr=1e-3, weight_decay=1e-4)
#     sched     = optim.lr_scheduler.CosineAnnealingLR(opt_main, T_max=finetune_ep, eta_min=1e-6)

#     smooth = 0.05
#     best_state = {'enc': copy.deepcopy(enc.state_dict()),
#                   'task_hd': copy.deepcopy(task_hd.state_dict())}
#     best_metric = float('inf')
#     leak_ema = 0.5

#     y_np = data['y_test']

#     for ep in range(1, finetune_ep+1):
#         enc.train(); task_hd.train(); adv1.train(); adv2.train(); probe.train()
#         fair_ramp = min(1.0, ep / max(8., finetune_ep * 0.3))
#         margin    = max(0.001, 0.05*(1.-fair_ramp))

#         leak_boost = float(np.clip((leak_ema - 0.55) * 4.0, 0.0, 2.0))
#         alpha1 = global_alpha_pri * fair_ramp * (1.0 + leak_boost)
#         alpha2 = global_alpha_sec * fair_ramp * (1.0 + leak_boost)

#         batch_leakage = []
#         for xb_med, xb_task, yb, sb1, sb2 in loader:
#             xb_med=xb_med.to(DEVICE); xb_task=xb_task.to(DEVICE)
#             yb=yb.to(DEVICE); sb1=sb1.to(DEVICE); sb2=sb2.to(DEVICE)

#             for opt_a, adv, sb, alph in [(opt_adv1, adv1, sb1, alpha1), (opt_adv2, adv2, sb2, alpha2)]:
#                 with torch.no_grad():
#                     zt_adv, _ = enc(xb_med, xb_task)
#                 al = F.cross_entropy(adv(zt_adv, alpha=alph), sb)
#                 opt_a.zero_grad(); al.backward()
#                 nn.utils.clip_grad_norm_(adv.parameters(), 1.0); opt_a.step()

#             z_task, z_med = enc(xb_med, xb_task)
#             logit = task_hd(z_task)
#             ybs   = yb*(1-smooth)+0.5*smooth
#             loss  = F.binary_cross_entropy_with_logits(logit, ybs)

#             probe_logits = probe(z_task)
#             probe_loss = F.cross_entropy(probe_logits, sb1)
#             batch_leakage.append((probe_logits.argmax(1) == sb1).float().mean().item())
#             loss += 0.5 * probe_loss

#             if fair_ramp > 0:
#                 if target == 'eo':
#                     loss += 1.5*fw_scale*fair_ramp * fair_loss_eo(logit, yb, sb1, margin=margin)
#                 else:
#                     loss += 1.5*fw_scale*fair_ramp * fair_loss_dp(logit, sb1, margin=margin)
#                     loss += 1.0*fw_scale*fair_ramp * fair_loss_eo_bounded(logit, yb, sb1, margin=0.08)

#                 leak_w = float(np.clip((leak_ema - 0.5) * 4.0, 0.0, 3.0))
#                 loss += (1.0+leak_w)*fair_ramp * F.cross_entropy(adv1(z_task, alpha=alpha1), sb1)
#                 loss += (0.8+leak_w)*fair_ramp * F.cross_entropy(adv2(z_task, alpha=alpha2), sb2)
#                 loss += 0.6*fair_ramp * hsic_loss(z_med, z_task)

#             opt_main.zero_grad(); loss.backward()
#             nn.utils.clip_grad_norm_(enc.parameters(), 1.0)
#             nn.utils.clip_grad_norm_(task_hd.parameters(), 1.0)
#             opt_main.step()
#         sched.step()

#         if batch_leakage:
#             leak_ema = 0.9 * leak_ema + 0.1 * float(np.mean(batch_leakage))

#         if ep % 5 == 0:
#             pr   = get_proba(enc, task_hd,
#                               torch.tensor(X_te_med, dtype=torch.float32),
#                               torch.tensor(X_te_task, dtype=torch.float32))
#             pred = (pr > 0.5).astype(int)
#             pred_std = float(np.std(pr))
#             acc  = accuracy_score(y_np, pred)
#             try:    tm = abs(float(equalized_odds_difference(y_np, pred, sensitive_features=s_te_primary)
#                                    if target=='eo' else
#                                    demographic_parity_difference(y_np, pred, sensitive_features=s_te_primary)))
#             except: tm = 1.0
#             penalised = tm + max(0., acc_floor - acc)*2.5
#             if pred_std < PRED_STD_MIN:
#                 penalised += 10.0
#             if penalised < best_metric and pred_std >= PRED_STD_MIN:
#                 best_metric = penalised
#                 best_state  = {'enc': copy.deepcopy(enc.state_dict()),
#                                'task_hd': copy.deepcopy(task_hd.state_dict())}

#     enc.load_state_dict(best_state['enc'])
#     task_hd.load_state_dict(best_state['task_hd'])

#     pr_te = get_proba(enc, task_hd,
#                        torch.tensor(X_te_med, dtype=torch.float32),
#                        torch.tensor(X_te_task, dtype=torch.float32))
#     pred  = (pr_te > 0.5).astype(int)
#     acc   = accuracy_score(y_np, pred)

#     try:    eo_sec = abs(float(equalized_odds_difference(y_np, pred, sensitive_features=s_te_secondary)))
#     except: eo_sec = 1.0
#     try:    dp_sec = abs(float(demographic_parity_difference(y_np, pred, sensitive_features=s_te_secondary)))
#     except: dp_sec = 1.0
#     try:    eo_pri = abs(float(equalized_odds_difference(y_np, pred, sensitive_features=s_te_primary)))
#     except: eo_pri = 1.0
#     try:    dp_pri = abs(float(demographic_parity_difference(y_np, pred, sensitive_features=s_te_primary)))
#     except: dp_pri = 1.0

#     return eo_pri, dp_pri, eo_sec, dp_sec, acc


# def train_lcfr(dataset_name, data, bbn_analyzer, primary_sens_train, primary_sens_test,
#                baseline_acc, baseline_eo=None, baseline_dp=None, ablation_mode='full'):
#     acc_floor = ACC_FLOORS.get(dataset_name, baseline_acc*0.90)

#     if ablation_mode == 'full': tqdm.write(f"\n  ─── EO Training (target=equalized_odds) ───")
#     set_seed()
#     (acc_eo_pre, eo_pre, acc_eo_post, eo_post,
#      enc_eo, hd_eo, X_tr_med_eo, X_tr_task_eo,
#      X_te_med_eo, X_te_task_eo, pred_eo) = _train_single(
#         dataset_name, data, bbn_analyzer, primary_sens_train, primary_sens_test,
#         acc_floor, target='eo', ablation_mode=ablation_mode)

#     eo_enc_state = copy.deepcopy(enc_eo.state_dict())
#     eo_hd_state  = copy.deepcopy(hd_eo.state_dict())

#     if ablation_mode == 'full':
#         nn_feat_names = data.get('nn_feature_names', [])
#         pw = bbn_analyzer.get_pathway_weights()
#         top_med_names = sorted(pw.items(), key=lambda x: x[1], reverse=True)
#         top_med_names = [f for f, _ in top_med_names]
#         analyze_representation_suppression(
#             dataset_name, enc_eo, hd_eo,
#             X_te_med_eo, X_te_task_eo,
#             primary_sens_test, data['y_test'],
#             top_med_names, bbn_analyzer.get_nie_weights(),
#             nn_feat_names)

#     del enc_eo, hd_eo

#     if ablation_mode == 'full': tqdm.write(f"\n  ─── DP Training (target=demographic_parity) ───")
#     set_seed()
#     (acc_dp_pre, dp_pre, acc_dp_post, dp_post,
#      enc_dp, hd_dp, X_tr_med_dp, X_tr_task_dp,
#      X_te_med_dp, X_te_task_dp, pred_dp) = _train_single(
#         dataset_name, data, bbn_analyzer, primary_sens_train, primary_sens_test,
#         acc_floor, target='dp', ablation_mode=ablation_mode)
#     del enc_dp, hd_dp

#     if ablation_mode == 'full':
#         analyze_impossibility(
#             dataset_name, data['y_test'],
#             pred_eo, pred_dp, primary_sens_test,
#             eo_pre, dp_pre)

#         W = 72
#         print(f"\n{'═'*W}"); print(f"  STAGE-WISE DIAGNOSTIC — {dataset_name.upper()}"); print(f"{'═'*W}")
#         print(f"  {'Stage':<28} {'Acc':>7} {'EO':>8} {'DP':>8}  {'ΔAcc':>7} {'ΔEO':>8} {'ΔDP':>8}")
#         print(f"  {'-'*72}")
#         b_acc=baseline_acc or float('nan'); b_eo=baseline_eo or float('nan'); b_dp=baseline_dp or float('nan')

#         def _fmt(v): return f"{v:.4f}" if v is not None and not (isinstance(v,float) and math.isnan(v)) else "  N/A "
#         def _delta(c,r):
#             if r is None or c is None: return "   N/A"
#             if math.isnan(r) or math.isnan(c): return "   N/A"
#             return f"{c-r:+.4f}"

#         print(f"  {'Baseline (MLP)':.<28} {_fmt(b_acc):>7} {_fmt(b_eo):>8} {_fmt(b_dp):>8}  {'—':>7} {'—':>8} {'—':>8}")
#         print(f"  {'After In-Processing (EO)':.<28} {_fmt(acc_eo_pre):>7} {_fmt(eo_pre):>8} {'—':>8}  {_delta(acc_eo_pre,b_acc):>7} {_delta(eo_pre,b_eo):>8} {'—':>8}")
#         print(f"  {'After In-Processing (DP)':.<28} {_fmt(acc_dp_pre):>7} {'—':>8} {_fmt(dp_pre):>8}  {_delta(acc_dp_pre,b_acc):>7} {'—':>8} {_delta(dp_pre,b_dp):>8}")
#         print(f"{'═'*W}")
#         eo_tension = abs(eo_post - dp_post)
#         print(f"\n  [Impossibility Observation]")
#         print(f"  |EO − DP| = {eo_tension:.4f}  " +
#               ("(tension present)" if eo_tension > 0.01 else "(tension minimal)"))

#         imp = IMPOSSIBILITY_ANALYSIS.get(dataset_name, {})
#         if imp.get('degenerate', False):
#             print(f"  NOTE: result flagged as degenerate (near-constant predictions). "
#                   f"Fairness metrics trivially satisfied — not a genuine impossibility demonstration.")

#         STAGE_RECORDS[dataset_name] = dict(
#             baseline_acc=b_acc, baseline_eo=b_eo, baseline_dp=b_dp,
#             inproc_eo_acc=acc_eo_pre, inproc_eo=eo_pre,
#             post_eo_acc=acc_eo_post,  post_eo=eo_post,
#             inproc_dp_acc=acc_dp_pre, inproc_dp=dp_pre,
#             post_dp_acc=acc_dp_post,  post_dp=dp_post)

#         secondary_results = {}
#         sens_cfg_all = DATASET_CONFIG[dataset_name]["sens_attrs"]
#         if len(sens_cfg_all) > 1:
#             tqdm.write(f"\n  [Secondary Attribute Transfer — single stage, warm-start from EO checkpoint]")

#             nn_feat_names = data.get('nn_feature_names', [])
#             pw_main = bbn_analyzer.get_pathway_weights()
#             top_med_names_main = sorted(pw_main.items(), key=lambda x: x[1], reverse=True)
#             top_med_names_main = [f for f, _ in top_med_names_main]
#             X_te_med_main, X_te_task_main = _split_x_by_mediators(
#                 data['X_test_nn'], top_med_names_main, nn_feat_names)

#             for s2_tr_key, s2_te_key in sens_cfg_all[1:]:
#                 if s2_tr_key not in data or s2_te_key not in data: continue
#                 s2_tr = np.asarray(data[s2_tr_key]); s2_te = np.asarray(data[s2_te_key])
#                 attr  = s2_te_key.replace('sens_','').replace('_test','')
#                 set_seed()
#                 eo_pri, dp_pri, eo_sec, dp_sec, acc_sec = _train_transfer(
#                     dataset_name, data, bbn_analyzer,
#                     primary_sens_train, primary_sens_test,
#                     s2_tr, s2_te, acc_floor, target='eo',
#                     primary_enc_state=eo_enc_state,
#                     primary_hd_state=eo_hd_state,
#                     X_tr_med=X_tr_med_eo, X_tr_task=X_tr_task_eo,
#                     X_te_med=X_te_med_main, X_te_task=X_te_task_main)
#                 secondary_results[attr] = dict(
#                     eo_post=eo_sec, dp_post=dp_sec,
#                     acc=acc_sec, eo_pri=eo_pri, dp_pri=dp_pri)
#                 tqdm.write(f"    attr={attr:<14}  "
#                            f"EO(sec) {eo_sec:.4f}  DP(sec) {dp_sec:.4f}  "
#                            f"EO(pri) {eo_pri:.4f}  DP(pri) {dp_pri:.4f}  acc={acc_sec:.4f}")

#         return (acc_eo_pre, eo_pre, acc_eo_post, eo_post,
#                 acc_dp_pre, dp_pre, acc_dp_post, dp_post, secondary_results)

#     return (acc_eo_pre, eo_pre, acc_eo_post, eo_post,
#             acc_dp_pre, dp_pre, acc_dp_post, dp_post, {})


# def run_ablation(dataset_name, data, bbn_analyzer, s_tr_arr, s_te_arr,
#                  baseline_acc, baseline_eo, baseline_dp):
#     tqdm.write(f"\n  [ABLATION STUDY — {dataset_name.upper()}]")
#     modes = [('no_bbn','No BBN / NIE'),('no_adv','No adversary'),('full','Full LCFR')]
#     abl_res = {}
#     for mode, label in modes:
#         tqdm.write(f"    Running: {label}")
#         set_seed()
#         (a_eo_pre, e_pre, a_eo_post, e_post,
#          a_dp_pre, d_pre, a_dp_post, d_post, _) = train_lcfr(
#             dataset_name, data, bbn_analyzer, s_tr_arr, s_te_arr,
#             baseline_acc, baseline_eo=baseline_eo, baseline_dp=baseline_dp,
#             ablation_mode=mode)
#         abl_res[mode] = dict(label=label,
#             acc_eo_post=a_eo_post, eo_post=e_post, acc_dp_post=a_dp_post, dp_post=d_post,
#             acc_eo_pre=a_eo_pre,   eo_pre=e_pre,   acc_dp_pre=a_dp_pre,   dp_pre=d_pre)
#         tqdm.write(f"      EO={e_post:.4f}  DP={d_post:.4f}  "
#                    f"Acc(EO)={a_eo_post:.4f}  Acc(DP)={a_dp_post:.4f}")
#     ABLATION_RECORDS[dataset_name] = abl_res
#     return abl_res


# def run_fairlearn_adversarial(X_tr, y_tr, s_tr, X_te, y_te, s_te):
#     in_dim = X_tr.shape[1]
#     try:
#         X_tr_f = X_tr.astype(np.float32)
#         X_te_f = X_te.astype(np.float32)
#         y_tr_f = y_tr.astype(np.float32)
#         s_tr_f = s_tr.astype(np.float32)

#         predictor = nn.Sequential(
#             nn.Linear(in_dim, 64), nn.ReLU(),
#             nn.Linear(64, 32), nn.ReLU(),
#             nn.Linear(32, 1)
#         )
#         adversary = nn.Sequential(
#             nn.Linear(1, 32), nn.ReLU(),
#             nn.Linear(32, 16), nn.ReLU(),
#             nn.Linear(16, 1)
#         )
#         clf = AdversarialFairnessClassifier(
#             predictor_model=predictor,
#             adversary_model=adversary,
#             epochs=50,
#             batch_size=512,
#             learning_rate=1e-3,
#             random_state=SEED)
#         clf.fit(X_tr_f, y_tr_f, sensitive_features=s_tr_f)
#         pred = clf.predict(X_te_f)
#         acc  = accuracy_score(y_te, pred)
#         eo   = r4(equalized_odds_difference(y_te, pred, sensitive_features=s_te.astype(np.float32)))
#         dp   = r4(demographic_parity_difference(y_te, pred, sensitive_features=s_te.astype(np.float32)))
#         return round(acc,4), eo, dp
#     except Exception as e:
#         tqdm.write(f"    [FairLearn] failed: {e}"); return None, None, None


# print("=" * 70)
# print(f"  LCFR v15 — All Architectural Fixes Applied | Device: {DEVICE}")
# print("=" * 70)

# LOADERS = {
#     "german":    load_german,   "adult":     load_adult,
#     "compas":    load_compas,   "bank":      load_bank,
#     "lawschool": load_lawschool,"hospital":  load_hospital,
# }

# results={}; fl_results={}; secondary_all={}

# for idx, (dname, loader_fn) in enumerate(LOADERS.items(), 1):
#     print(f"\n{'─'*70}\n  [{idx}/{len(LOADERS)}] {dname.upper()}\n{'─'*70}")
#     set_seed()
#     data = loader_fn()

#     for split in ('bbn_train_df','bbn_test_df'):
#         df_fix = data[split]
#         for c in df_fix.columns:
#             if df_fix[c].dtype == object:
#                 data[split][c] = df_fix[c].astype('category').cat.codes.astype(int)

#     sens_cfg = DATASET_CONFIG[dname]["sens_attrs"][0]
#     s_tr_arr = np.asarray(data[sens_cfg[0]])
#     s_te_arr = np.asarray(data[sens_cfg[1]])

#     sc  = StandardScaler()
#     mlp = MLPClassifier(hidden_layer_sizes=(256,128), max_iter=300, random_state=SEED)
#     mlp.fit(sc.fit_transform(data['X_train_nn']), data['y_train'])
#     base_pred = mlp.predict(sc.transform(data['X_test_nn']))
#     base_acc  = accuracy_score(data['y_test'], base_pred)
#     try:    base_eo = r4(equalized_odds_difference(data['y_test'], base_pred, sensitive_features=s_te_arr))
#     except: base_eo = float('nan')
#     try:    base_dp = r4(demographic_parity_difference(data['y_test'], base_pred, sensitive_features=s_te_arr))
#     except: base_dp = float('nan')
#     tqdm.write(f"\n  [Baseline MLP]  acc={base_acc:.4f}  EO={base_eo:.4f}  DP={base_dp:.4f}")

#     bbn_tr_df  = data['bbn_train_df'].copy()
#     label_col_bbn = 'y'
#     if label_col_bbn not in bbn_tr_df.columns:
#         possible = [c for c in bbn_tr_df.columns if any(k in c for k in ['bar','recid','readmit','credit','pass'])]
#         label_col_bbn = possible[0] if possible else bbn_tr_df.columns[-1]
#     if label_col_bbn not in bbn_tr_df.columns:
#         bbn_tr_df[label_col_bbn] = data['y_train']

#     sens_col_bbn = None
#     primary_sens_col_name = sens_cfg[0].replace('sens_','').replace('_train','')
#     candidate_names = [
#         primary_sens_col_name,
#         primary_sens_col_name + '_bin',
#         sens_cfg[0].replace('_train',''),
#     ]
#     for cname in candidate_names:
#         if cname in bbn_tr_df.columns:
#             sens_col_bbn = cname; break
#     if sens_col_bbn is None:
#         for col in bbn_tr_df.columns:
#             if col not in [label_col_bbn] and bbn_tr_df[col].nunique() <= 3:
#                 if _compute_mi(bbn_tr_df[col].values.astype(int), s_tr_arr.astype(int)) > 0.05:
#                     sens_col_bbn = col; break
#     if sens_col_bbn is None:
#         sens_col_bbn = bbn_tr_df.columns[0]
#     if sens_col_bbn not in bbn_tr_df.columns:
#         bbn_tr_df[sens_col_bbn] = s_tr_arr

#     tqdm.write(f"  Building BBN path analyzer (sens_col={sens_col_bbn}, NIE-initialized structure learning)...")

#     analyzer = BBNPathAnalyzer(sens_col=sens_col_bbn, label_col=label_col_bbn)
#     analyzer.fit(bbn_tr_df)
#     pw  = analyzer.get_pathway_weights()
#     nie = analyzer.get_nie_weights()
#     vrf = analyzer.get_verification()

#     tqdm.write(f"    sens_mi={analyzer.get_sens_mi():.4f}  "
#                f"verified_mediators={list(pw.keys())[:TOP_K_MEDIATORS]}")
#     tqdm.write(f"    learned_edges (top 5): {analyzer.get_learned_edges()[:5]}")
#     top_med = sorted(pw.items(), key=lambda x: x[1], reverse=True)[:TOP_K_MEDIATORS]
#     MEDIATOR_SCORES_ALL[dname] = dict(top_med)

#     tqdm.write(f"\n  [Mediator Verification — top {len(top_med)}]")
#     tqdm.write(f"  {'Feature':<22} {'Score':>7} {'NIE':>7} {'NIE_ratio':>10} {'MI_SF':>8} {'MI_FY':>8} {'Valid':>6}")
#     tqdm.write(f"  {'-'*68}")
#     for feat, score in top_med:
#         nie_val  = nie.get(feat, 0.0)
#         v        = vrf.get(feat, {})
#         nie_r    = v.get('nie_ratio', 0.0)
#         valid    = '  ✓' if v.get('valid', False) else '  ✗'
#         tqdm.write(f"  {feat:<22} {score:>7.4f} {nie_val:>7.4f} {nie_r:>10.4f} "
#                    f"{v.get('mi_sf',0):>8.4f} {v.get('mi_fy',0):>8.4f} {valid:>6}")

#     plot_learned_dag(dname, analyzer.get_learned_edges(), sens_col_bbn, label_col_bbn,
#                      pw, nie, vrf)
#     analyze_causal_pathways(dname, analyzer.get_learned_edges(), sens_col_bbn, label_col_bbn,
#                             pw, nie, vrf)

#     set_seed()
#     (acc_eo_pre, eo_pre, acc_eo_post, eo_post,
#      acc_dp_pre, dp_pre, acc_dp_post, dp_post,
#      sec_res) = train_lcfr(dname, data, analyzer, s_tr_arr, s_te_arr, base_acc,
#                             baseline_eo=base_eo, baseline_dp=base_dp, ablation_mode='full')

#     results[dname] = dict(baseline_acc=base_acc,
#                           acc_eo_pre=acc_eo_pre, eo_pre=eo_pre,
#                           acc_eo_post=acc_eo_post, eo_post=eo_post,
#                           acc_dp_pre=acc_dp_pre,  dp_pre=dp_pre,
#                           acc_dp_post=acc_dp_post, dp_post=dp_post)
#     secondary_all[dname] = sec_res

#     run_ablation(dname, data, analyzer, s_tr_arr, s_te_arr, base_acc, base_eo, base_dp)

#     tqdm.write("  Running FairLearn Adversarial baseline...")
#     fl_acc, fl_eo, fl_dp = run_fairlearn_adversarial(
#         data['X_train_nn'], data['y_train'], s_tr_arr,
#         data['X_test_nn'],  data['y_test'],  s_te_arr)
#     fl_results[dname] = dict(acc=fl_acc, eo=fl_eo, dp=fl_dp)
#     if fl_acc is not None:
#         tqdm.write(f"  FairLearn: acc={fl_acc:.4f} EO={fl_eo:.4f} DP={fl_dp:.4f}")

#     gc.collect()
#     if torch.cuda.is_available(): torch.cuda.empty_cache()


# def f2(v):
#     if v is None: return ' N/A'
#     if isinstance(v,float) and math.isnan(v): return ' N/A'
#     return f'{round(abs(float(v)), 2):.2f}'

# W = 70
# print(f"\n{'═'*W}\n  FINAL — Equalized Odds (normal-rounded 2 d.p.)\n{'═'*W}")
# print(f"  {'Dataset':<12} {'Base':>6} | {'Acc(in)':>8} {'EO(in)':>8} {'Degen?':>8}")
# print(f"  {'-'*50}")
# for d, r in results.items():
#     flag = " ✓" if round(r['eo_pre'], 2) == 0.00 else " !"
#     imp  = IMPOSSIBILITY_ANALYSIS.get(d, {})
#     degen = " DEG" if imp.get('pred_const_eo', False) else "    "
#     print(f"  {d:<12} {f2(r['baseline_acc']):>6} | "
#           f"{f2(r['acc_eo_pre']):>8} {f2(r['eo_pre']):>8}{flag}{degen}")

# print(f"\n{'═'*W}\n  FINAL — Demographic Parity (normal-rounded 2 d.p.)\n{'═'*W}")
# print(f"  {'Dataset':<12} {'Base':>6} | {'Acc(in)':>8} {'DP(in)':>8} {'Degen?':>8}")
# print(f"  {'-'*50}")
# for d, r in results.items():
#     flag = " ✓" if round(r['dp_pre'], 2) == 0.00 else " !"
#     imp  = IMPOSSIBILITY_ANALYSIS.get(d, {})
#     degen = " DEG" if imp.get('pred_const_dp', False) else "    "
#     print(f"  {d:<12} {f2(r['baseline_acc']):>6} | "
#           f"{f2(r['acc_dp_pre']):>8} {f2(r['dp_pre']):>8}{flag}{degen}")

# print(f"\n{'═'*W}\n  SECONDARY ATTRIBUTE RESULTS (single-stage transfer, warm-start)\n{'═'*W}")
# print(f"  {'Dataset':<12} {'Attribute':<14} {'EO(sec)':>8} {'DP(sec)':>8} {'EO(pri)':>8} {'DP(pri)':>8} {'Acc':>7}")
# print(f"  {'-'*68}")
# for d, sec in secondary_all.items():
#     for attr, r2 in sec.items():
#         print(f"  {d:<12} {attr:<14} {f2(r2['eo_post']):>8} {f2(r2['dp_post']):>8} "
#               f"{f2(r2.get('eo_pri')):>8} {f2(r2.get('dp_pri')):>8} {f2(r2.get('acc')):>7}")

# print(f"\n{'═'*W}\n  FAIRLEARN ADVERSARIAL COMPARISON (2 d.p.)\n{'═'*W}")
# print(f"  {'Dataset':<12} {'LCFR EO':>8} {'LCFR DP':>8} {'LCFR Acc':>9} | {'FL Acc':>7} {'FL EO':>6} {'FL DP':>6}")
# print(f"  {'-'*65}")
# for d in results:
#     r=results[d]; fl=fl_results[d]
#     print(f"  {d:<12} {f2(r['eo_pre']):>8} {f2(r['dp_pre']):>8} {f2(r['acc_eo_pre']):>9} | "
#           f"{f2(fl['acc']):>7} {f2(fl['eo']):>6} {f2(fl['dp']):>6}")
# print(f"{'═'*W}")

# print(f"\n{'═'*W}\n  ABLATION STUDY SUMMARY\n{'═'*W}")
# abl_mode_labels = {'no_bbn':'w/o BBN/NIE','no_adv':'w/o Adv','full':'Full LCFR'}
# print(f"  {'Dataset':<12} {'Config':<14} {'EO(in)':>7} {'DP(in)':>7} {'Acc':>8}")
# print(f"  {'-'*55}")
# for d in ABLATION_RECORDS:
#     for mode, lbl in abl_mode_labels.items():
#         ar = ABLATION_RECORDS[d].get(mode, {})
#         if not ar: continue
#         marker = ' ←' if mode == 'full' else ''
#         print(f"  {d:<12} {lbl:<14} {f2(ar.get('eo_pre')):>7} {f2(ar.get('dp_pre')):>7} "
#               f"{f2(ar.get('acc_eo_pre')):>8}{marker}")
#     print(f"  {'-'*55}")
# print(f"{'═'*W}")

# print(f"\n{'═'*W}\n  IMPOSSIBILITY ANALYSIS SUMMARY\n{'═'*W}")
# print(f"  {'Dataset':<12} {'BR(s=0)':>8} {'BR(s=1)':>8} {'BR_diff':>8} {'Mechanism':<25} {'Degen':>6}")
# print(f"  {'-'*72}")
# for d, imp in IMPOSSIBILITY_ANALYSIS.items():
#     degen_flag = "  YES" if imp.get('degenerate', False) else "   no"
#     print(f"  {d:<12} {imp['base_rate_0']:>8.4f} {imp['base_rate_1']:>8.4f} "
#           f"{imp['base_rate_diff']:>8.4f} {imp['mechanism']:<25}{degen_flag}")
# print(f"{'═'*W}")

# print(f"\n{'═'*W}\n  REPRESENTATION SUPPRESSION SUMMARY\n{'═'*W}")
# print(f"  {'Dataset':<12} {'MI(S,Y)':>9} {'MI(z_t,S)':>10} {'MI(z_m,S)':>10} {'MI(z_t,Y)':>10} {'Suppression':>12}")
# print(f"  {'-'*68}")
# for d, rep in REPRESENTATION_ANALYSIS.items():
#     amplified = " AMPLIFIED" if rep['mi_ztask_s'] > rep['mi_raw_s'] else ""
#     print(f"  {d:<12} {rep['mi_raw_s']:>9.5f} {rep['mi_ztask_s']:>10.5f} "
#           f"{rep.get('mi_zmed_s', 0.0):>10.5f} {rep['mi_ztask_y']:>10.5f} {rep['suppression_ratio']:>12.4f}{amplified}")
# print(f"{'═'*W}")

# print(f"\n{'═'*W}\n  CAUSAL PATHWAY SUMMARY\n{'═'*W}")
# for d, ca in CAUSAL_ANALYSIS.items():
#     print(f"  {d:<12}  edges={ca['n_edges']:>3}  "
#           f"direct={len(ca['direct_edges']):>2}  "
#           f"indirect_3node={len(ca['indirect_paths']):>2}  "
#           f"top_nie={ca['top_nie'][0][0] if ca['top_nie'] else 'N/A'}")
# print(f"{'═'*W}")

# print(f"\n{'═'*W}\n  DAG figures saved to {RESULTS_DIR}\n{'═'*W}")
# for d in results:
#     fpath = os.path.join(RESULTS_DIR, f'dag_{d}.png')
#     exists = '✓' if os.path.exists(fpath) else '✗'
#     print(f"  {exists} dag_{d}.png")
# print(f"{'═'*W}")

In [4]:
!pip install torch joblib seaborn pgmpy fairlearn

  Using cached pgmpy-1.1.2-py3-none-any.whl.metadata (13 kB)
  Using cached fairlearn-0.13.0-py3-none-any.whl.metadata (7.3 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 34.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.8/163.8 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 51.8 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: scipy
    Found existing installation: scipy 1.16.3
    Uninstalling scipy-1.16.3:
      Successfully uninstalled scipy-1.16.3


In [5]:
!pip install git+https://github.com/cmu-phil/causal-learn.git

  Cloning https://github.com/cmu-phil/causal-learn.git to /tmp/pip-req-build-ixasmsy6
  Running command git clone --filter=blob:none --quiet https://github.com/cmu-phil/causal-learn.git /tmp/pip-req-build-ixasmsy6
  Resolved https://github.com/cmu-phil/causal-learn.git to commit 520728fa595641625946460a11f85e711d41a9e1
  Preparing metadata (setup.py) ... done
  Created wheel for causal-learn: filename=causal_learn-0.1.4.5-py3-none-any.whl size=255181 sha256=1cb2f1235f0b3ba160758cdac1e79c61f9512519e71c62d205943c2ee598fcef
  Stored in directory: /tmp/pip-ephem-wheel-cache-qvdov9hx/wheels/a7/f5/19/0f01a36bb53da37cf83e9851609b81c542248a0373796c2361
Successfully built causal-learn


In [6]:
"""
LCFR v22 — Approach 3: DAG-Structured Latent Factorization

Core idea:
  The PC DAG encodes d-separation conditions: S ⊥ Y | {mediators}.
  Instead of splitting the INPUT (v21), we split the LATENT SPACE using
  the DAG's Markov factorization as a structural prior.

Architecture:
  SharedTrunk(X) → h  [shared representation, all features]
       ├── CausalMask(h, M_dag) → z_task  [task subspace: 1-M gating]
       └── CausalMask(h, M_dag) → z_med   [med subspace: M gating]

  where M_dag ∈ R^z_dim is a learned mask initialized from DAG structure:
    - Each latent dim's mediator-weight = fraction of its inputs from mediator nodes
    - Initialized via graph propagation over DAG adjacency
    - Then learned (sigmoid-gated), so the network can refine the structural prior
    - z_task projection: trunk → Linear → (1 - sigmoid(M)) element-wise
    - z_med  projection: trunk → Linear → sigmoid(M) element-wise

  NO raw feature split needed. DAG structure flows into initialization only.

Loss terms (same as v21 but applied to this architecture):
  L_task   = BCE(TaskHead(z_task), y)
  L_var    = relu(0.05 - std(logit))
  L_eo/dp  = fairness penalty
  L_sadv   = CE(SensAdv_GRL(z_task), s)      push S out of z_task
  L_sprobe = CE(SensProbe(z_med), s)          pull S into z_med (no GRL)
  L_recon  = MSE(MedDecoder(z_med), X_med)   ground z_med in mediator features
  L_ortho  = |mean(z_task · z_med)|          enforce partition

Comparison: runs v21 (input split) and v22 (latent split) side-by-side on each dataset.
"""

import os, gc, math, warnings, copy
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

from sklearn.metrics import accuracy_score, mutual_info_score
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

import joblib
from tqdm import tqdm

from causallearn.search.ConstraintBased.PC import pc
from causallearn.utils.cit import chisq
from fairlearn.metrics import equalized_odds_difference, demographic_parity_difference

warnings.filterwarnings('ignore')

SMOKE_TEST = True
SEED = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CACHE_DIR = './cache'
for d in [CACHE_DIR, './results']:
    os.makedirs(d, exist_ok=True)

def set_seed(s=SEED):
    torch.manual_seed(s)
    np.random.seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)
set_seed()

# ── Config ────────────────────────────────────────────────────────────────────

PC_ALPHA       = {'adult':0.05,'compas':0.05,'german':0.10,'bank':0.05,'hospital':0.05,'lawschool':0.05}
PRED_STD_MIN   = {'adult':0.06,'compas':0.06,'german':0.04,'bank':0.04,'hospital':0.05,'lawschool':0.06}
FAIR_WEIGHTS   = {          # (fair, sadv_grl, sprobe, recon, ortho)
    'adult':     (2.5, 1.5, 0.8, 0.5, 0.3),
    'compas':    (3.0, 2.0, 1.0, 0.5, 0.3),
    'german':    (1.5, 0.8, 0.0, 0.0, 0.0),
    'bank':      (1.2, 0.6, 0.4, 0.3, 0.2),
    'hospital':  (2.5, 1.5, 0.8, 0.5, 0.3),
    'lawschool': (2.0, 1.2, 0.6, 0.4, 0.2),
}
RAMP_CENTRE    = {'compas':0.10,'adult':0.20,'german':0.25,'bank':0.25,'hospital':0.20,'lawschool':0.20}
RAMP_STEEP     = {'compas':12.0,'adult':10.0,'german':8.0,'bank':8.0,'hospital':10.0,'lawschool':10.0}
SMOKE_EPOCH_CFG = {'total':80, 'warmup':8, 'patience':15}
EPOCH_CFG = {
    'adult':     {'total':600,'warmup':20,'patience':60},
    'compas':    {'total':600,'warmup':15,'patience':60},
    'german':    {'total':600,'warmup':40,'patience':80},
    'bank':      {'total':600,'warmup':40,'patience':60},
    'hospital':  {'total':600,'warmup':15,'patience':50},
    'lawschool': {'total':600,'warmup':20,'patience':60},
}
DATASET_CONFIG = {
    'adult':     {'sens_attrs':[('sens_sex_train','sens_sex_test'),('sens_race_train','sens_race_test')]},
    'compas':    {'sens_attrs':[('sens_race_train','sens_race_test'),('sens_sex_train','sens_sex_test')]},
    'german':    {'sens_attrs':[('sens_age_train','sens_age_test'),('sens_sex_train','sens_sex_test')]},
    'bank':      {'sens_attrs':[('sens_marital_train','sens_marital_test'),('sens_job_train','sens_job_test')]},
    'hospital':  {'sens_attrs':[('sens_race_train','sens_race_test'),('sens_sex_train','sens_sex_test')]},
    'lawschool': {'sens_attrs':[('sens_race_train','sens_race_test'),('sens_sex_train','sens_sex_test')]},
}
SMOKE_DATASETS = ['adult','compas','german','bank','hospital','lawschool']
LABEL_LEAK = {
    'readmit_binary','readmitted','two_year_recid','is_recid',
    'decile_score','score_text','bar','bar2','bar_passed','bar_pass',
    'zgpa','zfygpa','indxgrp','indxgrp2','decile1b','decile3',
    'dnn_bar_pass_prediction',
}
ARTIFACT_COLS = {
    'id','_id','index','row','sample','name','last','first','dob',
    'compas_screening_date','c_case_number','r_case_number','vr_case_number',
    'r_offense_date','vr_offense_date','c_arrest_date','c_offense_date',
    'c_jail_in','c_jail_out','v_screening_date','in_custody','out_custody',
    'event','type_of_assessment','v_type_of_assessment','score_factor',
    'r_charge_desc','vr_charge_desc','r_charge_degree','vr_charge_degree',
    'screening_date','race1','race2','race_bin','sex_bin','gender_bin',
    'age_bin','marital_bin','job_bin','black','white','asian','hispanic',
}

# ── Data utilities ─────────────────────────────────────────────────────────────

def to_dense(X):
    arr = X.toarray() if hasattr(X,'toarray') else np.asarray(X)
    return np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)

def clean_num(s):
    s = pd.to_numeric(s, errors='coerce')
    med = s.median()
    return s.fillna(0.0 if np.isnan(med) else med)

def safe_qcut(s, q=5):
    s = clean_num(s)
    if s.nunique() <= 1: return pd.Series(0, index=s.index, dtype=int)
    try: return pd.qcut(s, q, labels=False, duplicates='drop').fillna(0).astype(int)
    except: return pd.Series(0, index=s.index, dtype=int)

def num_pipe(): return Pipeline([('imp',SimpleImputer(strategy='median')),('sc',StandardScaler())])
def cat_pipe(): return Pipeline([('imp',SimpleImputer(strategy='most_frequent')),
                                  ('ohe',OneHotEncoder(handle_unknown='ignore',sparse_output=False))])
def _safe_cat_codes(s):
    if pd.api.types.is_categorical_dtype(s): return s.cat.codes.astype(int)
    return s.astype('category').cat.codes.astype(int)
def _enc_objects(df):
    for col in df.select_dtypes(include=['object','category']).columns:
        df[col] = df[col].astype('category').cat.codes.astype(int)
    return df
def _filter_bbn(df, sens_col, y_col, extra_drop=()):
    drop = set(extra_drop); n = len(df)
    for c in df.columns:
        if c in (sens_col, y_col): continue
        cl = c.lower().strip()
        if cl in LABEL_LEAK or cl in ARTIFACT_COLS: drop.add(c); continue
        if df[c].nunique() <= 1: drop.add(c); continue
        if df[c].dtype == object: drop.add(c); continue
        if df[c].nunique() > min(50, max(10, int(n*0.05))): drop.add(c)
    return df.drop(columns=[c for c in drop if c in df.columns])
def _nn_feature_names(pre, num_c, cat_c):
    names = list(num_c)
    ohe = pre.named_transformers_['c'].named_steps['ohe']
    for i,c in enumerate(cat_c):
        for v in ohe.categories_[i]: names.append(f'{c}={v}')
    return names

# ── Loaders (same as v21, abbreviated) ───────────────────────────────────────

def load_adult(csv_path='/kaggle/input/datasets/anothertemp/all-datasets/adult.csv', seed=SEED, use_cache=True):
    cache = os.path.join(CACHE_DIR,'adult_v19.pkl')
    if use_cache and os.path.exists(cache): return joblib.load(cache)
    cols=['age','workclass','fnlwgt','education','education-num','marital-status','occupation','relationship','race','sex','capital-gain','capital-loss','hours-per-week','native-country','income']
    df=pd.read_csv(csv_path,skipinitialspace=True,header=None,names=cols)
    df=df[~df.isin(['?']).any(axis=1)].reset_index(drop=True).drop(columns=['fnlwgt'])
    df['y']=df['income'].str.contains('>50K',na=False).astype(int)
    df['sex_bin']=(df['sex']=='Male').astype(int); df['race_bin']=(df['race']=='White').astype(int)
    num_c=['age','education-num','capital-gain','capital-loss','hours-per-week']
    cat_c=['workclass','education','marital-status','occupation','relationship','native-country']
    for c in num_c: df[c]=clean_num(df[c])
    for c in ['capital-gain','capital-loss']: df[c]=df[c].clip(upper=df[c].quantile(0.99))
    X=df.drop(columns=['income','y','sex_bin','race_bin','sex','race']); y=df['y'].values
    X_tr,X_te,y_tr,y_te,ss_tr,ss_te,sr_tr,sr_te=train_test_split(X,y,df['sex_bin'].values,df['race_bin'].values,test_size=0.2,stratify=y,random_state=seed)
    pre=ColumnTransformer([('n',num_pipe(),num_c),('c',cat_pipe(),cat_c)])
    X_tr_nn=to_dense(pre.fit_transform(X_tr)); X_te_nn=to_dense(pre.transform(X_te))
    nn_names=_nn_feature_names(pre,num_c,cat_c)
    bbn=df[[c for c in num_c+cat_c+['y','sex_bin','race_bin'] if c in df.columns]].copy()
    for c in num_c:
        if c in bbn: bbn[c]=safe_qcut(bbn[c],5)
    for c in cat_c:
        if c in bbn: bbn[c]=_safe_cat_codes(bbn[c])
    bbn['y']=bbn['y'].astype(int)
    bbn_tr=bbn.loc[X_tr.index].reset_index(drop=True); bbn_te=bbn.loc[X_te.index].reset_index(drop=True)
    bbn_tr=_filter_bbn(bbn_tr,'sex_bin','y',('sex','race','race_bin','sex_bin'))
    bbn_te=_filter_bbn(bbn_te,'sex_bin','y',('sex','race','race_bin','sex_bin'))
    r=dict(X_train_nn=X_tr_nn,X_test_nn=X_te_nn,bbn_train_df=bbn_tr,bbn_test_df=bbn_te,y_train=y_tr,y_test=y_te,sens_sex_train=ss_tr,sens_sex_test=ss_te,sens_race_train=sr_tr,sens_race_test=sr_te,num_cols=num_c,cat_cols=cat_c,nn_feature_names=nn_names)
    if use_cache: joblib.dump(r,cache)
    return r

def load_compas(csv_path='/kaggle/input/datasets/anothertemp/all-datasets/compas-scores-two-years.csv', seed=SEED, use_cache=True):
    cache=os.path.join(CACHE_DIR,'compas_v19.pkl')
    if use_cache and os.path.exists(cache): return joblib.load(cache)
    df=pd.read_csv(csv_path)
    df=df.loc[(df['days_b_screening_arrest'].between(-30,30))&(df['is_recid']!=-1)&(df['c_charge_degree']!='O')&(df['score_text']!='N/A')].reset_index(drop=True)
    df['race_bin']=(df['race']=='African-American').astype(int); df['sex_bin']=(df['sex']=='Male').astype(int)
    df['c_jail_in']=pd.to_datetime(df['c_jail_in'],errors='coerce'); df['c_jail_out']=pd.to_datetime(df['c_jail_out'],errors='coerce')
    df['jail_time']=(df['c_jail_out']-df['c_jail_in']).dt.days.fillna(0); df=df.drop(columns=['c_jail_in','c_jail_out'])
    num_c=['age','priors_count','days_b_screening_arrest','jail_time','juv_other_count','juv_misd_count','juv_fel_count']
    cat_c=['c_charge_degree','race','age_cat','sex','c_charge_desc']
    keep=[c for c in num_c+cat_c+['two_year_recid','race_bin','sex_bin'] if c in df.columns]
    df=df[keep].copy()
    for c in num_c:
        if c in df: df[c]=clean_num(df[c])
    X=df.drop(columns=[c for c in ['two_year_recid','race_bin','sex_bin'] if c in df.columns]); y=df['two_year_recid'].values
    X_tr,X_te,y_tr,y_te,sr_tr,sr_te,ss_tr,ss_te=train_test_split(X,y,df['race_bin'],df['sex_bin'],test_size=0.2,stratify=y,random_state=seed)
    act_num=[c for c in num_c if c in X.columns]; act_cat=[c for c in cat_c if c in X.columns]
    pre=ColumnTransformer([('n',num_pipe(),act_num),('c',cat_pipe(),act_cat)])
    X_tr_nn=to_dense(pre.fit_transform(X_tr)); X_te_nn=to_dense(pre.transform(X_te))
    nn_names=_nn_feature_names(pre,act_num,act_cat)
    bbn=df[keep].copy()
    for c in num_c:
        if c in bbn: bbn[c]=safe_qcut(bbn[c],5)
    for c in cat_c+['race','sex']:
        if c in bbn: bbn[c]=_safe_cat_codes(bbn[c])
    bbn=_enc_objects(bbn)
    bbn_tr=bbn.loc[X_tr.index].reset_index(drop=True); bbn_te=bbn.loc[X_te.index].reset_index(drop=True)
    bbn_tr=_filter_bbn(bbn_tr,'race_bin','two_year_recid',('race','sex','sex_bin','race_bin','is_recid'))
    bbn_te=_filter_bbn(bbn_te,'race_bin','two_year_recid',('race','sex','sex_bin','race_bin','is_recid'))
    r=dict(X_train_nn=X_tr_nn,X_test_nn=X_te_nn,bbn_train_df=bbn_tr,bbn_test_df=bbn_te,y_train=y_tr,y_test=y_te,sens_race_train=sr_tr,sens_race_test=sr_te,sens_sex_train=ss_tr,sens_sex_test=ss_te,num_cols=act_num,cat_cols=act_cat,nn_feature_names=nn_names)
    if use_cache: joblib.dump(r,cache)
    return r

def load_german(csv_path='/kaggle/input/datasets/anothertemp/all-datasets/german.data', seed=SEED, use_cache=True):
    cache=os.path.join(CACHE_DIR,'german_v19.pkl')
    if use_cache and os.path.exists(cache): return joblib.load(cache)
    cols=['status','duration','credit_history','purpose','amount','savings','employment','installment_rate','personal_status_sex','other_debtors','residence','property','age','other_installment_plans','housing','number_credits','job','people_liable','telephone','foreign_worker','credit']
    df=pd.read_csv(csv_path,sep=' ',names=cols)
    sex_map={'A91':'male','A92':'male','A93':'male','A94':'female','A95':'female'}
    df['sex']=df['personal_status_sex'].map(sex_map); df['sex_bin']=(df['sex']=='male').astype(int)
    df['age_bin']=(df['age']>=25).astype(int); df['y']=df['credit'].map({1:1,2:0})
    df=df.drop(columns=['personal_status_sex','sex','age','foreign_worker','credit'])
    num_c=['duration','amount','installment_rate','residence','number_credits','people_liable']
    cat_c=[c for c in df.columns if c not in num_c+['sex_bin','age_bin','y']]
    for c in num_c: df[c]=clean_num(df[c])
    X=df.drop(columns=['y']); y=df['y'].values
    X_tr,X_te,y_tr,y_te,sa_tr,sa_te,ss_tr,ss_te=train_test_split(X,y,df['age_bin'].values,df['sex_bin'].values,test_size=0.2,stratify=y,random_state=seed)
    pre=ColumnTransformer([('n',num_pipe(),num_c),('c',cat_pipe(),cat_c)])
    X_tr_nn=to_dense(pre.fit_transform(X_tr)); X_te_nn=to_dense(pre.transform(X_te))
    nn_names=_nn_feature_names(pre,num_c,cat_c)
    keep=[c for c in num_c+cat_c+['y','sex_bin','age_bin'] if c in df.columns]
    bbn=df[keep].copy()
    for c in num_c:
        if c in bbn: bbn[c]=safe_qcut(bbn[c],5)
    for c in cat_c:
        if c in bbn: bbn[c]=_safe_cat_codes(bbn[c])
    bbn_tr=bbn.loc[X_tr.index].reset_index(drop=True); bbn_te=bbn.loc[X_te.index].reset_index(drop=True)
    bbn_tr=_filter_bbn(bbn_tr,'age_bin','y',('sex_bin','age_bin'))
    bbn_te=_filter_bbn(bbn_te,'age_bin','y',('sex_bin','age_bin'))
    r=dict(X_train_nn=X_tr_nn,X_test_nn=X_te_nn,bbn_train_df=bbn_tr,bbn_test_df=bbn_te,y_train=y_tr,y_test=y_te,sens_age_train=sa_tr,sens_age_test=sa_te,sens_sex_train=ss_tr,sens_sex_test=ss_te,num_cols=num_c,cat_cols=cat_c,nn_feature_names=nn_names)
    if use_cache: joblib.dump(r,cache)
    return r

def load_bank(csv_path='/kaggle/input/datasets/anothertemp/all-datasets/bank-full.csv', seed=SEED, use_cache=True):
    cache=os.path.join(CACHE_DIR,'bank_v19.pkl')
    if use_cache and os.path.exists(cache): return joblib.load(cache)
    df=pd.read_csv(csv_path,sep=';')
    df=df.apply(lambda x: x.str.lower() if x.dtype=='object' else x)
    df=df[~df.isin(['unknown']).any(axis=1)].reset_index(drop=True)
    df['y']=(df['y']=='yes').astype(int)
    df['marital_bin']=(df['marital']==df['marital'].value_counts().idxmax()).astype(int)
    df['job_bin']=(df['job']==df['job'].value_counts().idxmax()).astype(int)
    cat_c=[c for c in ['job','education','default','housing','loan','contact','month','poutcome'] if c in df.columns]
    num_c=[c for c in ['age','balance','day','duration','campaign','pdays','previous'] if c in df.columns]
    for c in num_c: df[c]=clean_num(df[c])
    X=df.drop(columns=['y','marital_bin','job_bin','marital']); y=df['y'].values
    X_tr,X_te,y_tr,y_te,sm_tr,sm_te,sj_tr,sj_te=train_test_split(X,y,df['marital_bin'],df['job_bin'],test_size=0.2,stratify=y,random_state=seed)
    cat_enc=[c for c in cat_c if c in X.columns]; num_enc=[c for c in num_c if c in X.columns]
    pre=ColumnTransformer([('n',num_pipe(),num_enc),('c',cat_pipe(),cat_enc)])
    X_tr_nn=to_dense(pre.fit_transform(X_tr)); X_te_nn=to_dense(pre.transform(X_te))
    nn_names=_nn_feature_names(pre,num_enc,cat_enc)
    keep=[c for c in num_c+cat_c+['marital','y','marital_bin','job_bin'] if c in df.columns]
    bbn=df[keep].copy()
    for c in num_c:
        if c in bbn: bbn[c]=safe_qcut(bbn[c],5)
    for c in cat_c+['marital']:
        if c in bbn: bbn[c]=_safe_cat_codes(bbn[c])
    bbn_tr=bbn.loc[X_tr.index].reset_index(drop=True); bbn_te=bbn.loc[X_te.index].reset_index(drop=True)
    bbn_tr=_filter_bbn(bbn_tr,'marital_bin','y',('marital','job_bin','marital_bin'))
    bbn_te=_filter_bbn(bbn_te,'marital_bin','y',('marital','job_bin','marital_bin'))
    r=dict(X_train_nn=X_tr_nn,X_test_nn=X_te_nn,bbn_train_df=bbn_tr,bbn_test_df=bbn_te,y_train=y_tr,y_test=y_te,sens_marital_train=sm_tr,sens_marital_test=sm_te,sens_job_train=sj_tr,sens_job_test=sj_te,num_cols=num_enc,cat_cols=cat_enc,nn_feature_names=nn_names)
    if use_cache: joblib.dump(r,cache)
    return r

def load_hospital(csv_path='/kaggle/input/datasets/anothertemp/all-datasets/diabetes_hospital_fairlearn.csv', seed=SEED, use_cache=True):
    cache=os.path.join(CACHE_DIR,'hospital_v19.pkl')
    if use_cache and os.path.exists(cache): return joblib.load(cache)
    df=pd.read_csv(csv_path)
    df=df.drop(columns=[c for c in ['max_glu_serum','A1Cresult','readmitted.1'] if c in df.columns])
    df=df[~df.isin(['Missing']).any(axis=1)].dropna(subset=['race','gender']).reset_index(drop=True)
    age_map={"'0-10'":5,"'10-20'":15,"'20-30'":25,"'30-40'":35,"'40-50'":45,"'50-60'":55,"'60-70'":65,"'70-80'":75,"'80-90'":85,"'90-100'":95,"'30 years or younger'":20,"'30-60 years'":45,"'Over 60 years'":70}
    df['age']=df['age'].replace(age_map).astype(float); df['y']=df['readmit_binary'].astype(int)
    mc=df['race'].value_counts().idxmax(); df['race_bin']=(df['race']==mc).astype(int)
    df['gender_bin']=df['gender'].map({'Male':0,'Female':1}).fillna(0).astype(int)
    cat_c=[c for c in ['discharge_disposition_id','admission_source_id','medical_specialty','primary_diagnosis','insulin','change','diabetesMed'] if c in df.columns]
    num_c=[c for c in ['age','time_in_hospital','num_lab_procedures','num_procedures','num_medications','number_diagnoses','had_emergency','had_inpatient_days','had_outpatient_days','medicare','medicaid'] if c in df.columns]
    for c in num_c: df[c]=clean_num(df[c])
    X=df.drop(columns=['readmit_binary','y','race_bin','gender_bin']); y=df['y'].values
    X_tr,X_te,y_tr,y_te,sr_tr,sr_te,sg_tr,sg_te=train_test_split(X,y,df['race_bin'],df['gender_bin'],test_size=0.2,stratify=y,random_state=seed)
    pre=ColumnTransformer([('n',num_pipe(),num_c),('c',cat_pipe(),cat_c)])
    X_tr_nn=to_dense(pre.fit_transform(X_tr)); X_te_nn=to_dense(pre.transform(X_te))
    nn_names=_nn_feature_names(pre,num_c,cat_c)
    keep=[c for c in num_c+cat_c+['y','race_bin','gender_bin','race','gender'] if c in df.columns]
    bbn=df[keep].copy()
    for c in num_c:
        if c in bbn: bbn[c]=safe_qcut(bbn[c],5)
    for c in cat_c+['race','gender']:
        if c in bbn: bbn[c]=_safe_cat_codes(bbn[c])
    bbn_tr=bbn.loc[X_tr.index].reset_index(drop=True); bbn_te=bbn.loc[X_te.index].reset_index(drop=True)
    bbn_tr=_filter_bbn(bbn_tr,'race_bin','y',('race','gender','race_bin','gender_bin','readmit_binary'))
    bbn_te=_filter_bbn(bbn_te,'race_bin','y',('race','gender','race_bin','gender_bin','readmit_binary'))
    r=dict(X_train_nn=X_tr_nn,X_test_nn=X_te_nn,bbn_train_df=bbn_tr,bbn_test_df=bbn_te,y_train=y_tr,y_test=y_te,sens_race_train=sr_tr,sens_race_test=sr_te,sens_sex_train=sg_tr,sens_sex_test=sg_te,num_cols=num_c,cat_cols=cat_c,nn_feature_names=nn_names)
    if use_cache: joblib.dump(r,cache)
    return r

def load_lawschool(csv_path='/kaggle/input/datasets/anothertemp/all-datasets/bar_pass_prediction.csv', seed=SEED, use_cache=True):
    cache=os.path.join(CACHE_DIR,'lawschool_v21.pkl')
    if use_cache and os.path.exists(cache): return joblib.load(cache)
    df=pd.read_csv(csv_path); df.columns=[c.lower().strip() for c in df.columns]
    drop_cols=['zgpa','zfygpa','indxgrp','indxgrp2','bar','bar2','decile1b','decile3','gpa_stem','fulltime','bar1','id','index','dnn_bar_pass_prediction']
    df=df.drop(columns=[c for c in drop_cols if c in df.columns])
    label_col=next((c for c in ['pass_bar','bar_pass','passed_bar'] if c in df.columns),None)
    if label_col is None: raise ValueError('No pass_bar label')
    df['y']=df[label_col].astype(int); df=df.drop(columns=[label_col])
    race_col=next((c for c in ['race1','race','white'] if c in df.columns),None)
    if race_col is None: df['race_bin']=0
    else:
        df['race_bin']=(df[race_col].astype(str).str.lower().str.strip()=='white').astype(int) if df[race_col].dtype==object else (df[race_col]==sorted(df[race_col].unique())[-1]).astype(int)
        df=df.drop(columns=[race_col])
    sex_col=next((c for c in ['sex','gender','female','male'] if c in df.columns),None)
    if sex_col is None: df['sex_bin']=0
    else:
        df['sex_bin']=(df[sex_col].astype(str).str.lower().str.strip().isin(['female','f','woman'])).astype(int) if df[sex_col].dtype==object else (df[sex_col]==df[sex_col].min()).astype(int)
        df=df.drop(columns=[sex_col])
    num_c,cat_c=[],[]
    for c in df.columns:
        if c in ('y','race_bin','sex_bin'): continue
        if pd.api.types.is_numeric_dtype(df[c]) and df[c].nunique()>6: num_c.append(c); df[c]=clean_num(df[c])
        else: cat_c.append(c)
    df=df.dropna(subset=['y']).reset_index(drop=True)
    X=df.drop(columns=['y','race_bin','sex_bin']); y=df['y'].values
    X_tr,X_te,y_tr,y_te,sr_tr,sr_te,ss_tr,ss_te=train_test_split(X,y,df['race_bin'].values,df['sex_bin'].values,test_size=0.2,stratify=y,random_state=seed)
    ct_num=[c for c in num_c if c in X.columns]; ct_cat=[c for c in cat_c if c in X.columns]
    transformers=[]
    if ct_num: transformers.append(('n',num_pipe(),ct_num))
    if ct_cat: transformers.append(('c',cat_pipe(),ct_cat))
    pre=ColumnTransformer(transformers)
    X_tr_nn=to_dense(pre.fit_transform(X_tr)); X_te_nn=to_dense(pre.transform(X_te))
    nn_names=_nn_feature_names(pre,ct_num,ct_cat) if ct_cat else list(ct_num)
    bbn=df[[c for c in ct_num+ct_cat+['y','race_bin','sex_bin'] if c in df.columns]].copy()
    for c in ct_num:
        if c in bbn: bbn[c]=safe_qcut(bbn[c],5)
    for c in ct_cat:
        if c in bbn: bbn[c]=_safe_cat_codes(bbn[c])
    bbn_tr=bbn.loc[X_tr.index].reset_index(drop=True); bbn_te=bbn.loc[X_te.index].reset_index(drop=True)
    bbn_tr=_filter_bbn(bbn_tr,'race_bin','y',('race_bin','sex_bin'))
    bbn_te=_filter_bbn(bbn_te,'race_bin','y',('race_bin','sex_bin'))
    r=dict(X_train_nn=X_tr_nn,X_test_nn=X_te_nn,bbn_train_df=bbn_tr,bbn_test_df=bbn_te,y_train=y_tr,y_test=y_te,sens_race_train=sr_tr,sens_race_test=sr_te,sens_sex_train=ss_tr,sens_sex_test=ss_te,num_cols=ct_num,cat_cols=ct_cat,nn_feature_names=nn_names)
    if use_cache: joblib.dump(r,cache)
    return r

# ── PC causal discovery ────────────────────────────────────────────────────────

def learn_causal_mediators(bbn_df, sens_col, label_col, nn_feature_names, alpha=0.05, dataset_name=''):
    df=bbn_df.copy(); df=_enc_objects(df)
    for c in df.columns: df[c]=pd.to_numeric(df[c],errors='coerce').fillna(0).astype(int)
    keep_cols=[c for c in df.columns if df[c].nunique()>1]; df=df[keep_cols]
    if sens_col not in df.columns or label_col not in df.columns: return [],[],{}
    col_order=[sens_col]+[c for c in df.columns if c not in (sens_col,label_col)]+[label_col]
    df=df[col_order]; data_mat=df.values.astype(float)
    s_idx=0; y_idx=len(col_order)-1; n_nodes=data_mat.shape[1]
    tqdm.write(f'  [PC] {dataset_name}: {n_nodes} nodes, n={len(df)}, alpha={alpha}')
    try:
        cg=pc(data_mat,alpha=alpha,indep_test=chisq,show_progress=False)
        G=cg.G.graph
    except Exception as e:
        tqdm.write(f'  [PC] Failed: {e}'); return [],[],{}
    adj={i:set() for i in range(n_nodes)}
    for i in range(n_nodes):
        for j in range(n_nodes):
            if G[i,j]!=0: adj[i].add(j)
    directed={i:set() for i in range(n_nodes)}; processed=set()
    for i in range(n_nodes):
        for j in adj[i]:
            pair=tuple(sorted([i,j]))
            if pair in processed: continue
            processed.add(pair); a,b=pair
            if s_idx in (a,b):
                directed[s_idx].add(b if a==s_idx else a)
            elif y_idx in (a,b):
                directed[a if b==y_idx else b].add(y_idx)
            else:
                if G[b,a]==1 and G[a,b]==-1: directed[b].add(a)
                elif G[a,b]==1 and G[b,a]==-1: directed[a].add(b)
                else: directed[min(a,b)].add(max(a,b))
    def reach(start,g):
        visited,stack=set(),[start]
        while stack:
            node=stack.pop()
            if node not in visited: visited.add(node); stack.extend(g[node]-visited)
        return visited
    from_s=reach(s_idx,directed)
    rev={i:set() for i in range(n_nodes)}
    for i,js in directed.items():
        for j in js: rev[j].add(i)
    to_y=reach(y_idx,rev)
    med_indices_bbn=(from_s&to_y)-{s_idx,y_idx}
    mediator_bbn_names=[col_order[m] for m in sorted(med_indices_bbn)]
    med_nn_indices=[]
    for m_name in mediator_bbn_names:
        for i,fn in enumerate(nn_feature_names):
            if fn==m_name or fn.startswith(m_name+'='):
                if i not in med_nn_indices: med_nn_indices.append(i)
    dag_info={
        'col_order':col_order,
        'directed_edges':{col_order[i]:[col_order[j] for j in js] for i,js in directed.items()},
        'mediator_bbn_names':mediator_bbn_names,
        'from_s':[col_order[i] for i in from_s if i not in (s_idx,y_idx)],
        'to_y':[col_order[i] for i in to_y if i not in (s_idx,y_idx)],
        'n_nodes':n_nodes,
        's_idx':s_idx, 'y_idx':y_idx,
        'med_indices_bbn':sorted(med_indices_bbn),
    }
    tqdm.write(f'  [PC] Mediators: {mediator_bbn_names} | NN idx: {med_nn_indices}/{len(nn_feature_names)}')
    return mediator_bbn_names, med_nn_indices, dag_info


# ═══════════════════════════════════════════════════════════════════════════════
# APPROACH 3: DAG-Structured Latent Factorization
# ═══════════════════════════════════════════════════════════════════════════════

def build_dag_mask_init(dag_info: dict, nn_feature_names: List[str], z_dim: int = 64) -> torch.Tensor:
    """
    Build initial causal mask M ∈ R^z_dim from DAG structure ONLY.
    No raw data needed — purely graph operations.

    Method:
      1. Compute mediator_score[i] for each NN feature i:
           = 1.0 if feature i corresponds to a mediator BBN node
           = 0.0 otherwise
         (OHE-expanded features inherit their parent variable's score)

      2. Propagate scores through a 2-hop DAG neighborhood:
           Each node also gets partial credit if it's a neighbor of a mediator
           on the S→?→Y path (captures near-mediator features)

      3. Map feature scores → latent dimension scores via a random projection
         that respects the score distribution:
           - Sort latent dims by score (high-score dims → z_med side)
           - M[d] = mediator_score for the d-th latent dimension

    M is then used as:
      z_task[:, d] = trunk[:, d] * (1 - sigmoid(M[d]))   [low-score dims]
      z_med[:, d]  = trunk[:, d] * sigmoid(M[d])          [high-score dims]

    M is LEARNED (not fixed), so the network can refine the structural prior.
    """
    if not dag_info or 'col_order' not in dag_info:
        # No DAG: uniform mask (0.5 everywhere = no structural prior)
        return torch.full((z_dim,), 0.0)

    col_order = dag_info['col_order']
    directed_edges = dag_info['directed_edges']
    med_bbn_names = set(dag_info['mediator_bbn_names'])
    from_s_names = set(dag_info['from_s'])
    to_y_names = set(dag_info['to_y'])

    # Score each BBN node by causal role
    # Mediator (on S→M→Y): 1.0
    # S-reachable but not Y-ancestor (S-influenced only): 0.6
    # Y-ancestor but not S-reachable (confounders): 0.2
    # Neither: 0.0
    node_score = {}
    for name in col_order:
        if name in med_bbn_names:
            node_score[name] = 1.0
        elif name in from_s_names and name in to_y_names:
            node_score[name] = 1.0  # same as mediator (belt-and-suspenders)
        elif name in from_s_names:
            node_score[name] = 0.6
        elif name in to_y_names:
            node_score[name] = 0.2
        else:
            node_score[name] = 0.0

    # 1-hop propagation: neighbors of high-score nodes get partial credit
    propagated = dict(node_score)
    for src, dsts in directed_edges.items():
        src_score = node_score.get(src, 0.0)
        for dst in dsts:
            propagated[dst] = max(propagated.get(dst, 0.0), src_score * 0.5)
    for name in col_order:
        propagated[name] = max(node_score.get(name, 0.0), propagated.get(name, 0.0))

    # Map NN feature indices to scores
    # OHE-expanded features inherit their parent variable's score
    feat_scores = np.zeros(len(nn_feature_names))
    for i, fn in enumerate(nn_feature_names):
        # fn is either "variable_name" or "variable_name=value"
        base = fn.split('=')[0]
        feat_scores[i] = propagated.get(base, propagated.get(fn, 0.0))

    # Build latent mask: distribute feature scores across z_dim dimensions
    # Strategy: sort feature scores, assign each latent dim the mean score
    # of its "receptive field" fraction of features (uniform partition)
    # High-score → high M[d] → goes to z_med
    sorted_scores = np.sort(feat_scores)[::-1]  # descending
    # Each latent dim gets responsibility for (n_features / z_dim) features
    chunk = max(1, len(sorted_scores) // z_dim)
    mask_vals = np.zeros(z_dim)
    for d in range(z_dim):
        start = d * chunk
        end = min(start + chunk, len(sorted_scores))
        mask_vals[d] = float(np.mean(sorted_scores[start:end])) if start < end else 0.0

    # Scale to logit space so sigmoid(M) ≈ mask_vals
    # sigmoid(x) = v  ↔  x = log(v/(1-v))
    eps = 1e-6
    mask_vals = np.clip(mask_vals, eps, 1 - eps)
    logit_mask = np.log(mask_vals / (1 - mask_vals))

    tqdm.write(f'  [DAG-mask] med_score=1.0 dims: {int((mask_vals > 0.8).sum())} / {z_dim}')
    tqdm.write(f'  [DAG-mask] score dist: max={mask_vals.max():.3f} mean={mask_vals.mean():.3f} min={mask_vals.min():.3f}')

    return torch.tensor(logit_mask, dtype=torch.float32)


class SharedTrunk(nn.Module):
    """Shared encoder reading ALL features → h ∈ R^trunk_dim."""
    def __init__(self, input_dim, trunk_dim=128, dropout=0.3):
        super().__init__()
        h = 128 if SMOKE_TEST else 256
        self.net = nn.Sequential(
            nn.Linear(input_dim, h), nn.LayerNorm(h), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(h, h), nn.LayerNorm(h), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(h, trunk_dim), nn.LayerNorm(trunk_dim),
        )

    def forward(self, x):
        return self.net(x)


class CausalSplitHead(nn.Module):
    """
    Projects trunk → (z_task, z_med) using learned DAG mask M.

    z_task[d] = proj_task(h)[d] * (1 - sigmoid(M[d]))
    z_med[d]  = proj_med(h)[d]  * sigmoid(M[d])

    M is initialized from DAG structure, then learned.
    When M[d] → +∞: dim d is fully med (z_task[d]≈0, z_med[d]≈proj_med[d])
    When M[d] → -∞: dim d is fully task (z_task[d]≈proj_task[d], z_med[d]≈0)

    Both projections share trunk_dim → z_dim, but are gated by complementary masks.
    LayerNorm applied AFTER masking to renormalize the gated outputs.
    """
    def __init__(self, trunk_dim, z_dim, mask_init: torch.Tensor):
        super().__init__()
        self.z_dim = z_dim
        self.proj_task = nn.Linear(trunk_dim, z_dim)
        self.proj_med  = nn.Linear(trunk_dim, z_dim)
        # M is a learned parameter, initialized from DAG
        self.M = nn.Parameter(mask_init.clone())
        self.norm_task = nn.LayerNorm(z_dim)
        self.norm_med  = nn.LayerNorm(z_dim)

    def forward(self, h):
        gate = torch.sigmoid(self.M)          # [z_dim]
        task_raw = self.proj_task(h)           # [B, z_dim]
        med_raw  = self.proj_med(h)            # [B, z_dim]
        z_task = self.norm_task(task_raw * (1.0 - gate))
        z_med  = self.norm_med(med_raw  * gate)
        return z_task, z_med

    def get_gate(self):
        return torch.sigmoid(self.M).detach().cpu()


class TaskHead(nn.Module):
    """Binary classifier on z_task ONLY."""
    def __init__(self, z_dim=64):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(z_dim, 64), nn.GELU(), nn.Linear(64, 1))

    def forward(self, z_task):
        return self.net(z_task).squeeze(-1)


class GradReverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha): ctx.alpha = alpha; return x.view_as(x)
    @staticmethod
    def backward(ctx, grad): return -ctx.alpha * grad, None


class SensAdv(nn.Module):
    """Adversary on z_task via GRL — pushes S out of z_task."""
    def __init__(self, z_dim=64, hidden=64):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(z_dim,hidden),nn.GELU(),nn.Linear(hidden,hidden),nn.GELU(),nn.Linear(hidden,2))

    def forward(self, z, alpha=1.0):
        return self.net(GradReverse.apply(z, alpha))


class SensProbe(nn.Module):
    """Probe on z_med, NO GRL — pulls S into z_med (identifies split)."""
    def __init__(self, z_dim=64, hidden=32):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(z_dim,hidden),nn.GELU(),nn.Linear(hidden,2))

    def forward(self, z):
        return self.net(z)


class MedDecoder(nn.Module):
    """Decodes z_med → X_med features (grounds z_med in actual mediator values)."""
    def __init__(self, z_dim=64, med_dim=1):
        super().__init__()
        h = max(32, med_dim)
        self.net = nn.Sequential(nn.Linear(z_dim,h),nn.GELU(),nn.Linear(h,med_dim))

    def forward(self, z):
        return self.net(z)


def fair_loss_eo(logit, y, s, margin=-0.002):
    p = torch.sigmoid(logit)
    tprs, fprs = [], []
    for g in (0, 1):
        mp=(s==g)&(y==1); mn=(s==g)&(y==0)
        tprs.append(p[mp].mean() if mp.sum()>2 else torch.tensor(0.5,device=logit.device))
        fprs.append(p[mn].mean() if mn.sum()>2 else torch.tensor(0.5,device=logit.device))
    return F.relu(torch.max(torch.abs(tprs[0]-tprs[1]),torch.abs(fprs[0]-fprs[1]))-margin)

def fair_loss_dp(logit, s, margin=-0.002):
    p = torch.sigmoid(logit)
    p0=p[s==0].mean() if (s==0).sum()>2 else torch.tensor(0.5,device=logit.device)
    p1=p[s==1].mean() if (s==1).sum()>2 else torch.tensor(0.5,device=logit.device)
    return F.relu(torch.abs(p0-p1)-margin)


@torch.no_grad()
def get_proba_v22(trunk, split_head, task_head, X_all, device):
    trunk.eval(); split_head.eval(); task_head.eval()
    X_t = torch.tensor(X_all, dtype=torch.float32).to(device)
    h = trunk(X_t)
    z_task, _ = split_head(h)
    return torch.sigmoid(task_head(z_task)).cpu().numpy()


class EarlyStopping:
    def __init__(self, window=5, patience=60):
        self.window=window; self.patience=patience
        self.history=[]; self.best_metric=float('inf')
        self.best_state=None; self.streak=0

    def update(self, metric, acc, **modules):
        self.history.append(metric)
        if metric < self.best_metric:
            self.best_metric=metric
            self.best_state={k:copy.deepcopy(v.state_dict()) if v is not None else None
                             for k,v in modules.items()}
            self.best_state['acc']=acc
        if len(self.history)<self.window: return False
        if np.mean(self.history[-self.window:])<0.005:
            self.streak+=1
            if self.streak>=self.patience: return True
        else: self.streak=0
        return False


def train_v22(dataset_name, data, med_nn_idx, dag_info, s_train, s_test,
              target='eo', use_adv=True):
    """
    v22 training: Approach 3 — DAG-structured latent factorization.
    Single SharedTrunk reads all X. CausalSplitHead uses learned DAG mask
    to route information into z_task (task-relevant) and z_med (mediator-relevant).
    No raw feature split. DAG used only for mask initialization.
    """
    cfg = SMOKE_EPOCH_CFG if SMOKE_TEST else EPOCH_CFG.get(dataset_name,{'total':600,'warmup':20,'patience':50})
    total_ep=cfg['total']; warmup_ep=cfg['warmup']; patience=cfg['patience']
    batch_size=2048

    X_nn=data['X_train_nn']; y_np=data['y_train']
    n_features=X_nn.shape[1]; nn_names=data['nn_feature_names']
    has_med = len(med_nn_idx) > 0
    z_dim = 64
    trunk_dim = 128

    # Build DAG mask init from graph structure (no raw data)
    mask_init = build_dag_mask_init(dag_info, nn_names, z_dim=z_dim).to(DEVICE)

    # Models
    trunk      = SharedTrunk(input_dim=n_features, trunk_dim=trunk_dim).to(DEVICE)
    split_head = CausalSplitHead(trunk_dim=trunk_dim, z_dim=z_dim, mask_init=mask_init).to(DEVICE)
    task_head  = TaskHead(z_dim=z_dim).to(DEVICE)
    sens_adv   = SensAdv(z_dim=z_dim).to(DEVICE)
    sens_probe = SensProbe(z_dim=z_dim).to(DEVICE) if has_med else None
    med_dec    = MedDecoder(z_dim=z_dim, med_dim=len(med_nn_idx)).to(DEVICE) if has_med else None

    # Tensors
    X_t = torch.tensor(X_nn, dtype=torch.float32).to(DEVICE)
    y_t = torch.tensor(y_np, dtype=torch.float32).to(DEVICE)
    s_t = torch.tensor(s_train, dtype=torch.long).to(DEVICE)
    X_med_t = X_t[:, med_nn_idx] if has_med else None  # for reconstruction target

    dataset = TensorDataset(X_t, y_t, s_t, torch.arange(len(y_np)).to(DEVICE))
    loader  = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    # Optimizers
    params_main = list(trunk.parameters()) + list(split_head.parameters()) + list(task_head.parameters())
    opt_main = torch.optim.AdamW(params_main, lr=3e-4, weight_decay=1e-4)
    opt_sadv = torch.optim.AdamW(sens_adv.parameters(), lr=1e-3, weight_decay=1e-4)
    opt_med  = torch.optim.AdamW(
        list(sens_probe.parameters()) + list(med_dec.parameters()), lr=3e-4, weight_decay=1e-4
    ) if has_med else None

    sch_main = torch.optim.lr_scheduler.CosineAnnealingLR(opt_main, T_max=total_ep, eta_min=1e-5)

    fw = FAIR_WEIGHTS.get(dataset_name, (2.5,1.5,0.8,0.5,0.3))
    fw_fair,fw_sadv,fw_probe,fw_recon,fw_ortho = fw
    pred_std_min = PRED_STD_MIN.get(dataset_name, 0.05)
    ramp_centre  = RAMP_CENTRE.get(dataset_name, 0.25)
    ramp_steep   = RAMP_STEEP.get(dataset_name, 10.0)

    stopper = EarlyStopping(window=5, patience=patience)
    X_test=data['X_test_nn']; y_test=data['y_test']

    for ep in range(1, total_ep+1):
        trunk.train(); split_head.train(); task_head.train(); sens_adv.train()
        if sens_probe: sens_probe.train()
        if med_dec:    med_dec.train()

        fair = ep > warmup_ep
        if fair:
            ramp_progress = (ep-warmup_ep)/max(1.0,total_ep-warmup_ep)
            fair_ramp = float(1.0/(1.0+math.exp(-ramp_steep*(ramp_progress-ramp_centre))))
        else:
            fair_ramp = 0.0
        alpha = min(2.0, fair_ramp*2.0)

        for Xb, yb, sb, idxb in loader:
            h = trunk(Xb)
            z_task, z_med = split_head(h)
            logit = task_head(z_task)

            # Task loss
            loss = F.binary_cross_entropy_with_logits(logit, yb)
            loss = loss + 0.5 * F.relu(0.05 - logit.std())

            # Fairness penalty
            if fair and fair_ramp > 0:
                fair_pen = fair_loss_eo(logit,yb,sb) if target=='eo' else fair_loss_dp(logit,sb)
                loss = loss + fw_fair * fair_ramp * fair_pen

            # SensAdv GRL on z_task
            if use_adv and fair and fair_ramp > 0:
                sadv_out = sens_adv(z_task, alpha=alpha)
                loss = loss + fw_sadv * fair_ramp * F.cross_entropy(sadv_out, sb)

            opt_main.zero_grad(); opt_sadv.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(trunk.parameters(), 1.0)
            torch.nn.utils.clip_grad_norm_(split_head.parameters(), 1.0)
            torch.nn.utils.clip_grad_norm_(task_head.parameters(), 1.0)
            opt_main.step()
            if use_adv: opt_sadv.step()

            # Med path: separate backward
            if has_med and opt_med is not None:
                # Recompute z_med with fresh graph (no grad from main backward)
                h2 = trunk(Xb).detach()  # detach trunk — med path shouldn't move trunk
                _, z_med2 = split_head(h2)

                # Probe: z_med should predict S
                probe_out = sens_probe(z_med2)
                loss_probe = F.cross_entropy(probe_out, sb)

                # Recon: z_med should reconstruct X_med features
                recon_out  = med_dec(z_med2)
                X_med_b    = Xb[:, med_nn_idx]
                loss_recon = F.mse_loss(recon_out, X_med_b)

                # Ortho: z_task ⊥ z_med (detach z_task so ortho doesn't move trunk twice)
                z_task2 = split_head(trunk(Xb).detach())[0].detach()
                loss_ortho = (z_task2 * z_med2).mean().abs()

                loss_med = (fw_probe * loss_probe +
                            fw_recon * loss_recon +
                            fw_ortho * fair_ramp * loss_ortho)

                opt_med.zero_grad()
                loss_med.backward()
                opt_med.step()

        sch_main.step()

        eval_every = 5 if SMOKE_TEST else 10
        if ep % eval_every == 0 or ep == total_ep:
            proba = get_proba_v22(trunk, split_head, task_head, X_test, DEVICE)
            pred  = (proba > 0.5).astype(int)
            acc   = accuracy_score(y_test, pred)
            pred_std = float(np.std(proba))
            try:
                metric = abs(float(
                    equalized_odds_difference(y_test,pred,sensitive_features=s_test)
                    if target=='eo' else
                    demographic_parity_difference(y_test,pred,sensitive_features=s_test)
                ))
            except: metric=1.0

            if fair and pred_std >= pred_std_min:
                should_stop = stopper.update(metric, acc,
                    trunk=trunk, split_head=split_head, task_head=task_head,
                    sens_adv=sens_adv, sens_probe=sens_probe, med_dec=med_dec)
                if should_stop:
                    tqdm.write(f'  Early stop ep{ep}')
                    break
            elif fair:
                tqdm.write(f'  [DEGEN ep{ep}] std={pred_std:.3f}')

            log_every = 5 if SMOKE_TEST else 50
            if ep % log_every == 0:
                gate = split_head.get_gate()
                med_dims = int((gate > 0.5).sum()); task_dims = z_dim - med_dims
                tqdm.write(f'  [v22|{target.upper()}] ep{ep}: acc={acc:.4f} {target}={metric:.4f} '
                           f'std={pred_std:.3f} gate_med={med_dims}/{z_dim}')

    if stopper.best_state:
        trunk.load_state_dict(stopper.best_state['trunk'])
        split_head.load_state_dict(stopper.best_state['split_head'])
        task_head.load_state_dict(stopper.best_state['task_head'])
        if sens_probe and stopper.best_state.get('sens_probe'):
            sens_probe.load_state_dict(stopper.best_state['sens_probe'])
        tqdm.write(f'  Restored best: {target}={stopper.best_metric:.4f} acc={stopper.best_state["acc"]:.4f}')

    proba = get_proba_v22(trunk, split_head, task_head, X_test, DEVICE)
    pred  = (proba > 0.5).astype(int)
    acc   = accuracy_score(y_test, pred)
    try:
        metric = abs(float(
            equalized_odds_difference(y_test,pred,sensitive_features=s_test)
            if target=='eo' else
            demographic_parity_difference(y_test,pred,sensitive_features=s_test)
        ))
    except: metric=1.0

    return acc, metric, trunk, split_head, task_head, sens_probe, med_dec


@torch.no_grad()
def compute_mi_v22(trunk, split_head, task_head, X_nn, y_np, s_np, med_nn_idx, device, n_bins=10):
    trunk.eval(); split_head.eval(); task_head.eval()
    X_t = torch.tensor(X_nn, dtype=torch.float32).to(device)
    h = trunk(X_t)
    z_task, z_med = split_head(h)
    z_task = z_task.cpu().numpy()
    z_med  = z_med.cpu().numpy()

    def disc(z): return np.digitize(z, np.percentile(z, np.linspace(0,100,n_bins+1)[1:-1]))
    zt1 = disc(z_task[:,0]); zm1 = disc(z_med[:,0])
    result = {
        'MI(z_task,S)': float(mutual_info_score(s_np,zt1)),
        'MI(z_task,Y)': float(mutual_info_score(y_np,zt1)),
        'MI(z_med,S)':  float(mutual_info_score(s_np,zm1)),
        'MI(z_med,Y)':  float(mutual_info_score(y_np,zm1)),
    }
    if len(med_nn_idx)>0:
        X_t_cpu = X_t.cpu().numpy()
        m1 = disc(X_t_cpu[:,med_nn_idx[0]])
        result['MI(z_task,M)'] = float(mutual_info_score(m1,zt1))
        result['MI(z_med,M)']  = float(mutual_info_score(m1,zm1))

    # Gate stats
    gate = split_head.get_gate().numpy()
    result['gate_med_dims']  = int((gate>0.5).sum())
    result['gate_mean']      = float(gate.mean())
    return result


# ── v21 reference (input-split) — inline minimal version for comparison ────────

class V21TaskEncoder(nn.Module):
    def __init__(self, input_dim, z_dim=64, dropout=0.3):
        super().__init__()
        h=128 if SMOKE_TEST else 256
        self.net=nn.Sequential(nn.Linear(input_dim,h),nn.LayerNorm(h),nn.GELU(),nn.Dropout(dropout),nn.Linear(h,h),nn.LayerNorm(h),nn.GELU(),nn.Dropout(dropout),nn.Linear(h,z_dim),nn.LayerNorm(z_dim))
    def forward(self,x): return self.net(x)

class V21MedEncoder(nn.Module):
    def __init__(self, med_dim, z_dim=32, dropout=0.2):
        super().__init__()
        h=max(32,med_dim*2)
        self.net=nn.Sequential(nn.Linear(med_dim,h),nn.LayerNorm(h),nn.GELU(),nn.Dropout(dropout),nn.Linear(h,z_dim),nn.LayerNorm(z_dim))
    def forward(self,x): return self.net(x)

class V21MedDecoder(nn.Module):
    def __init__(self, z_dim=32, med_dim=1):
        super().__init__()
        h=max(32,med_dim*2)
        self.net=nn.Sequential(nn.Linear(z_dim,h),nn.GELU(),nn.Linear(h,med_dim))
    def forward(self,z): return self.net(z)

class V21TaskHead(nn.Module):
    def __init__(self, z_dim=64):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(z_dim,64),nn.GELU(),nn.Linear(64,1))
    def forward(self,z): return self.net(z).squeeze(-1)

class V21SensAdv(nn.Module):
    def __init__(self, z_dim=64, hidden=64):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(z_dim,hidden),nn.GELU(),nn.Linear(hidden,hidden),nn.GELU(),nn.Linear(hidden,2))
    def forward(self,z,alpha=1.0): return self.net(GradReverse.apply(z,alpha))

class V21SensProbe(nn.Module):
    def __init__(self, z_dim=32, hidden=32):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(z_dim,hidden),nn.GELU(),nn.Linear(hidden,2))
    def forward(self,z): return self.net(z)


@torch.no_grad()
def get_proba_v21(task_enc, task_head, X_all, rest_mask, device):
    task_enc.eval(); task_head.eval()
    X_t=torch.tensor(X_all,dtype=torch.float32).to(device)
    return torch.sigmoid(task_head(task_enc(X_t[:,rest_mask]))).cpu().numpy()


def train_v21(dataset_name, data, med_nn_idx, s_train, s_test, target='eo', use_adv=True):
    """v21: input-split reference (from lcfr_v21.py)."""
    cfg=SMOKE_EPOCH_CFG if SMOKE_TEST else EPOCH_CFG.get(dataset_name,{'total':600,'warmup':20,'patience':50})
    total_ep=cfg['total']; warmup_ep=cfg['warmup']; patience=cfg['patience']
    X_nn=data['X_train_nn']; y_np=data['y_train']; n_features=X_nn.shape[1]

    med_mask=torch.zeros(n_features,dtype=torch.bool)
    has_med=len(med_nn_idx)>0
    if has_med: med_mask[torch.tensor(med_nn_idx,dtype=torch.long)]=True
    rest_mask=~med_mask
    n_rest=int(rest_mask.sum()); n_med=int(med_mask.sum())

    task_enc=V21TaskEncoder(n_rest).to(DEVICE)
    task_head=V21TaskHead().to(DEVICE)
    sens_adv=V21SensAdv().to(DEVICE)
    med_enc=V21MedEncoder(n_med).to(DEVICE) if has_med else None
    med_dec=V21MedDecoder(z_dim=32,med_dim=n_med).to(DEVICE) if has_med else None
    sens_probe=V21SensProbe(z_dim=32).to(DEVICE) if has_med else None

    X_t=torch.tensor(X_nn,dtype=torch.float32).to(DEVICE)
    y_t=torch.tensor(y_np,dtype=torch.float32).to(DEVICE)
    s_t=torch.tensor(s_train,dtype=torch.long).to(DEVICE)
    loader=DataLoader(TensorDataset(X_t,y_t,s_t),batch_size=2048,shuffle=True)

    opt_task=torch.optim.AdamW(list(task_enc.parameters())+list(task_head.parameters()),lr=3e-4,weight_decay=1e-4)
    opt_sadv=torch.optim.AdamW(sens_adv.parameters(),lr=1e-3,weight_decay=1e-4)
    opt_med=torch.optim.AdamW(list(med_enc.parameters())+list(med_dec.parameters()),lr=3e-4,weight_decay=1e-4) if has_med else None
    opt_probe=torch.optim.AdamW(sens_probe.parameters(),lr=1e-3,weight_decay=1e-4) if has_med else None
    sch=torch.optim.lr_scheduler.CosineAnnealingLR(opt_task,T_max=total_ep,eta_min=1e-5)

    fw=FAIR_WEIGHTS.get(dataset_name,(2.5,1.5,0.8,0.5,0.3))
    fw_fair,fw_sadv,fw_probe,fw_recon,fw_ortho=fw
    pred_std_min=PRED_STD_MIN.get(dataset_name,0.05)
    rc=RAMP_CENTRE.get(dataset_name,0.25); rs=RAMP_STEEP.get(dataset_name,10.0)

    stopper=EarlyStopping(window=5,patience=patience)
    X_test=data['X_test_nn']; y_test=data['y_test']

    for ep in range(1,total_ep+1):
        task_enc.train(); task_head.train(); sens_adv.train()
        if med_enc: med_enc.train(); med_dec.train(); sens_probe.train()
        fair=ep>warmup_ep
        fair_ramp=float(1.0/(1.0+math.exp(-rs*((ep-warmup_ep)/max(1.0,total_ep-warmup_ep)-rc)))) if fair else 0.0
        alpha=min(2.0,fair_ramp*2.0)

        for Xb,yb,sb in loader:
            z_task=task_enc(Xb[:,rest_mask]); logit=task_head(z_task)
            loss=F.binary_cross_entropy_with_logits(logit,yb)+0.5*F.relu(0.05-logit.std())
            if fair and fair_ramp>0:
                loss=loss+fw_fair*fair_ramp*(fair_loss_eo(logit,yb,sb) if target=='eo' else fair_loss_dp(logit,sb))
                if use_adv: loss=loss+fw_sadv*fair_ramp*F.cross_entropy(sens_adv(z_task,alpha),sb)
            opt_task.zero_grad(); opt_sadv.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(task_enc.parameters(),1.0)
            opt_task.step()
            if use_adv: opt_sadv.step()
            if has_med:
                z_med=med_enc(Xb[:,med_mask])
                lm=(fw_probe*F.cross_entropy(sens_probe(z_med),sb)+
                    fw_recon*F.mse_loss(med_dec(z_med),Xb[:,med_mask])+
                    fw_ortho*fair_ramp*(task_enc(Xb[:,rest_mask]).detach()[:, :32] * z_med).mean().abs())
                opt_med.zero_grad(); opt_probe.zero_grad(); lm.backward()
                opt_med.step(); opt_probe.step()
        sch.step()

        eval_every=5 if SMOKE_TEST else 10
        if ep%eval_every==0 or ep==total_ep:
            proba=get_proba_v21(task_enc,task_head,X_test,rest_mask,DEVICE)
            pred=(proba>0.5).astype(int); acc=accuracy_score(y_test,pred)
            pred_std=float(np.std(proba))
            try:
                metric=abs(float(equalized_odds_difference(y_test,pred,sensitive_features=s_test) if target=='eo' else demographic_parity_difference(y_test,pred,sensitive_features=s_test)))
            except: metric=1.0
            if fair and pred_std>=pred_std_min:
                stopper.update(metric,acc,task_enc=task_enc,task_head=task_head,sens_adv=sens_adv,
                               med_enc=med_enc,med_dec=med_dec,sens_probe=sens_probe)

        log_every=5 if SMOKE_TEST else 50
        if ep%log_every==0:
            proba=get_proba_v21(task_enc,task_head,X_test,rest_mask,DEVICE)
            pred=(proba>0.5).astype(int); acc=accuracy_score(y_test,pred); pred_std=float(np.std(proba))
            try: metric=abs(float(equalized_odds_difference(y_test,pred,sensitive_features=s_test) if target=='eo' else demographic_parity_difference(y_test,pred,sensitive_features=s_test)))
            except: metric=1.0
            tqdm.write(f'  [v21|{target.upper()}] ep{ep}: acc={acc:.4f} {target}={metric:.4f} std={pred_std:.3f}')

    if stopper.best_state:
        task_enc.load_state_dict(stopper.best_state['task_enc'])
        task_head.load_state_dict(stopper.best_state['task_head'])
        if med_enc and stopper.best_state.get('med_enc'): med_enc.load_state_dict(stopper.best_state['med_enc'])
        tqdm.write(f'  v21 restored: {target}={stopper.best_metric:.4f} acc={stopper.best_state["acc"]:.4f}')

    proba=get_proba_v21(task_enc,task_head,X_test,rest_mask,DEVICE)
    pred=(proba>0.5).astype(int); acc=accuracy_score(y_test,pred)
    try:
        metric=abs(float(equalized_odds_difference(y_test,pred,sensitive_features=s_test) if target=='eo' else demographic_parity_difference(y_test,pred,sensitive_features=s_test)))
    except: metric=1.0

    # MI for v21
    med_mask_f=torch.zeros(n_features,dtype=torch.bool)
    if has_med: med_mask_f[torch.tensor(med_nn_idx,dtype=torch.long)]=True
    X_t2=torch.tensor(X_test,dtype=torch.float32).to(DEVICE)
    with torch.no_grad():
        task_enc.eval(); task_head.eval()
        zt=task_enc(X_t2[:,rest_mask]).cpu().numpy()
        zm=med_enc(X_t2[:,med_mask_f]).cpu().numpy() if has_med and med_enc else None
    def disc(z,n=10): return np.digitize(z,np.percentile(z,np.linspace(0,100,n+1)[1:-1]))
    zt1=disc(zt[:,0])
    mi={'MI(z_task,S)':float(mutual_info_score(s_test,zt1)),'MI(z_task,Y)':float(mutual_info_score(y_test,zt1))}
    if zm is not None:
        zm1=disc(zm[:,0])
        mi['MI(z_med,S)']=float(mutual_info_score(s_test,zm1))
        mi['MI(z_med,Y)']=float(mutual_info_score(y_test,zm1))
        if len(med_nn_idx)>0:
            m1=disc(X_t2[:,med_nn_idx[0]].cpu().numpy())
            mi['MI(z_task,M)']=float(mutual_info_score(m1,zt1))
            mi['MI(z_med,M)']=float(mutual_info_score(m1,zm1))

    return acc, metric, mi


# ── Main comparison loop ───────────────────────────────────────────────────────

LOADERS = {
    'compas':load_compas,'adult':load_adult,'german':load_german,
    'bank':load_bank,'hospital':load_hospital,'lawschool':load_lawschool,
}

print('='*70)
print(f'  LCFR v22 — Approach 3: DAG Latent Factorization | Device: {DEVICE}')
print('  Comparing: v21 (input split) vs v22 (latent split via DAG mask)')
print('='*70)

active = {k:v for k,v in LOADERS.items() if k in SMOKE_DATASETS} if SMOKE_TEST else LOADERS
comparison = {}

for idx,(dname,loader_fn) in enumerate(active.items(),1):
    print(f'\n{"─"*70}\n  [{idx}/{len(active)}] {dname.upper()}\n{"─"*70}')
    set_seed()
    data = loader_fn(use_cache=True)
    sens_cfg = DATASET_CONFIG[dname]['sens_attrs'][0]
    s_train  = np.asarray(data[sens_cfg[0]])
    s_test   = np.asarray(data[sens_cfg[1]])

    # PC
    bbn_df = data['bbn_train_df'].copy(); bbn_df = _enc_objects(bbn_df)
    label_col='y'
    if label_col not in bbn_df.columns:
        for c in bbn_df.columns:
            if c in ['two_year_recid','pass_bar','readmit_binary']: label_col=c; break
    if label_col not in bbn_df.columns: bbn_df[label_col]=data['y_train'].astype(int)
    bbn_df['_sens']=s_train[:len(bbn_df)].astype(int)
    med_bbn,med_nn_idx,dag_info = learn_causal_mediators(
        bbn_df,'_sens',label_col,data['nn_feature_names'],
        alpha=PC_ALPHA.get(dname,0.05),dataset_name=dname)

    # ── v21 (input split)
    print(f'\n  ── v21 (input split) ──')
    set_seed()
    acc_eo_21, eo_21, mi_21 = train_v21(dname,data,med_nn_idx,s_train,s_test,'eo')
    set_seed()
    acc_dp_21, dp_21, _     = train_v21(dname,data,med_nn_idx,s_train,s_test,'dp')

    # ── v22 (latent split)
    print(f'\n  ── v22 (latent split, DAG mask) ──')
    set_seed()
    acc_eo_22,eo_22,tr22,sh22,th22,sp22,md22 = train_v22(dname,data,med_nn_idx,dag_info,s_train,s_test,'eo')
    set_seed()
    acc_dp_22,dp_22,_,_,_,_,_ = train_v22(dname,data,med_nn_idx,dag_info,s_train,s_test,'dp')
    mi_22 = compute_mi_v22(tr22,sh22,th22,data['X_test_nn'],data['y_test'],s_test,med_nn_idx,DEVICE)

    comparison[dname] = {
        'n_med':len(med_bbn),
        'v21':{'acc_eo':acc_eo_21,'eo':eo_21,'acc_dp':acc_dp_21,'dp':dp_21,'mi':mi_21},
        'v22':{'acc_eo':acc_eo_22,'eo':eo_22,'acc_dp':acc_dp_22,'dp':dp_22,'mi':mi_22},
    }

    print(f'\n  FINAL {dname.upper()}:')
    print(f'    v21 (input split):  EO={eo_21:.4f} DP={dp_21:.4f} Acc={acc_eo_21:.4f}')
    print(f'    v22 (latent split): EO={eo_22:.4f} DP={dp_22:.4f} Acc={acc_eo_22:.4f}  '
          f'gate_med={mi_22.get("gate_med_dims","?")}dims  gate_mean={mi_22.get("gate_mean",0):.3f}')

    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()


# ── Summary table ──────────────────────────────────────────────────────────────

print(f'\n{"="*70}')
print('  HEAD-TO-HEAD: v21 (input split) vs v22 (DAG latent factorization)')
print(f'{"="*70}')
print(f'  {"Dataset":<12} {"#Med":>4} | {"v21 EO":>8} {"v21 DP":>8} {"v21 Acc":>8} | '
      f'{"v22 EO":>8} {"v22 DP":>8} {"v22 Acc":>8} | {"Winner":>8}')
print(f'  {"─"*78}')
for dname,r in comparison.items():
    v21=r['v21']; v22=r['v22']
    # winner by sum(EO+DP) — lower is better
    score21=v21['eo']+v21['dp']; score22=v22['eo']+v22['dp']
    winner='v22 ✓' if score22<score21 else ('v21 ✓' if score21<score22 else 'tie')
    print(f'  {dname:<12} {r["n_med"]:>4} | '
          f'{v21["eo"]:>8.4f} {v21["dp"]:>8.4f} {v21["acc_eo"]:>8.4f} | '
          f'{v22["eo"]:>8.4f} {v22["dp"]:>8.4f} {v22["acc_eo"]:>8.4f} | '
          f'{winner:>8}')

print(f'\n{"="*70}')
print('  DISENTANGLEMENT: MI(z_task,S)↓  MI(z_med,S)↑  MI(z_task,M)↓  MI(z_med,M)↑')
print(f'{"="*70}')
print(f'  {"Dataset":<12} {"Arch":>5} | {"zt_S":>8} {"zm_S":>8} {"zt_M":>8} {"zm_M":>8} | '
      f'{"S→zm":>6} {"M→zm":>6}')
print(f'  {"─"*68}')
for dname,r in comparison.items():
    for arch,mi in [('v21',r['v21']['mi']),('v22',r['v22']['mi'])]:
        zt_s=mi.get('MI(z_task,S)',float('nan'))
        zm_s=mi.get('MI(z_med,S)',float('nan'))
        zt_m=mi.get('MI(z_task,M)',float('nan'))
        zm_m=mi.get('MI(z_med,M)',float('nan'))
        s_ok='✓' if (not math.isnan(zm_s)) and zt_s<zm_s else '~'
        m_ok='✓' if (not math.isnan(zm_m)) and zt_m<zm_m else '~'
        print(f'  {dname:<12} {arch:>5} | {zt_s:>8.4f} {zm_s:>8.4f} {zt_m:>8.4f} {zm_m:>8.4f} | '
              f'{s_ok:>6} {m_ok:>6}')

print(f'\n{"="*70}')
print('  GATE ANALYSIS (v22 only) — learned mask after training')
print('  gate_med_dims: how many of 64 latent dims ended up "med-side" (gate>0.5)')
print(f'{"="*70}')
print(f'  {"Dataset":<12} {"#Med_nodes":>10} {"gate_med_dims":>14} {"gate_mean":>10}')
print(f'  {"─"*50}')
for dname,r in comparison.items():
    mi22=r['v22']['mi']
    print(f'  {dname:<12} {r["n_med"]:>10} '
          f'{mi22.get("gate_med_dims","?"):>14} '
          f'{mi22.get("gate_mean",float("nan")):>10.4f}')
print(f'{"="*70}')

  LCFR v22 — Approach 3: DAG Latent Factorization | Device: cuda
  Comparing: v21 (input split) vs v22 (latent split via DAG mask)

──────────────────────────────────────────────────────────────────────
  [1/6] COMPAS
──────────────────────────────────────────────────────────────────────
  [PC] compas: 8 nodes, n=4937, alpha=0.05
  [PC] Mediators: ['age', 'priors_count', 'jail_time', 'c_charge_degree'] | NN idx: [0, 1, 3, 7, 8]/377

  ── v21 (input split) ──
  [v21|EO] ep5: acc=0.6194 eo=0.3088 std=0.089
  [v21|EO] ep10: acc=0.6186 eo=0.2258 std=0.125
  [v21|EO] ep15: acc=0.6138 eo=0.0183 std=0.095
  [v21|EO] ep20: acc=0.6146 eo=0.0661 std=0.096
  [v21|EO] ep25: acc=0.6121 eo=0.0468 std=0.092
  [v21|EO] ep30: acc=0.6073 eo=0.0740 std=0.073
  [v21|EO] ep35: acc=0.6097 eo=0.0645 std=0.056
  [v21|EO] ep40: acc=0.6049 eo=0.0868 std=0.054
  [v21|EO] ep45: acc=0.6146 eo=0.0810 std=0.057
  [v21|EO] ep50: acc=0.6089 eo=0.0055 std=0.059
  [v21|EO] ep55: acc=0.6097 eo=0.0114 std=0.059
  [v21|EO]